# Flood Susceptibility Modelling of the Federal Capital Territory (FCT), Abuja
### Random Forest, XGBoost and CART — Google Colab notebook

This notebook supports the study *Geospatial Assessment of Flood Susceptibility and Coping Capacity of the FCT, Abuja*. It trains three tree-based classifiers (Random Forest, XGBoost and CART) on the flood inventory (`Flood_points_1.geojson`, field `flooded`: 1 = flooded, 0 = non-flooded) using the flood-conditioning rasters supplied in `FCT_FLOOD.rar`. It then validates the models, ranks the conditioning factors, and produces wall-to-wall susceptibility maps.

The notebook has two parts. **Part A** (sections 1–16) builds and explains the models and writes the rasters; **Part B** (sections 17–26) aggregates the results to the 62 electoral wards, estimates population exposure and reproduces the remaining manuscript figures (Figures 4, 7, 11 and 15–18).

**Workflow (Part A)**
1. Install packages and extract the dataset (from Google Drive or a direct upload).
2. Align all conditioning factors to a single reference grid (≈30 m, EPSG:4326, nearest neighbour) and mask them to the FCT boundary.
3. Quality-control the factors (see the note on distance to water below).
4. Extract factor values at the inventory points.
5. Run multicollinearity diagnostics (Pearson correlation, VIF, tolerance).
6. Make a stratified 70/30 train–test split and tune hyper-parameters with 5-fold cross-validation, scored by AUC.
7. Validate with AUC-ROC, accuracy, precision, recall, specificity, F1 and Cohen's κ, plus repeated random CV and spatial-block CV.
8. Rank the factors with model-specific importance and permutation importance (ΔAUC).
9. Predict a flood probability for every pixel, classify it into five susceptibility classes, compute class areas and export GeoTIFFs.
10. Explain all three models with SHAP values and partial dependence, including data-driven factor thresholds.
11. Map where the models agree on high susceptibility, and how far their predictions spread.
12. Build a coping-capacity index from emergency access, health access, night-time light and road density, cross it with susceptibility in a 3 × 3 bivariate map, and count the population in each combination, flagging the high-susceptibility, low-capacity hotspot.

**Notes on the supplied data (checked before writing this notebook)**
- **Inventory.** There are 353 records. One has no geometry and is skipped, leaving 352 points: 215 flooded and 137 non-flooded.
- **Grids.** Most hazard rasters are 3630 × 3531 or 3631 × 3531 pixels at 0.000269° (≈30 m). The grids are offset from one another by a fraction of a pixel, so every layer is re-sampled onto the `fct_elevation.tif` grid.
- **Distance to water.** In `fct_dist_to_water.tif`, about 53% of the pixels inside the FCT hold values of ≈1,376–1,378 km. That is physically impossible within the territory and looks like an export artefact. The valid part of the raster never exceeds ≈20.4 km. By default (`DIST_TO_WATER_MODE = "recompute"`), the notebook rebuilds Euclidean distance to the land-cover water class (code 80). *Caveat: this class probably omits narrow streams, so the recomputed layer is a proxy; please confirm it suits your study, or set the mode to `"drop"`.*
- **Rainfall.** `fct_rainfall.tif` is a coarse grid of 0.05° cells (≈5.5 km; 282 distinct values) copied onto the 30 m pixels. If used as-is, its cell edges appear as straight-edged rectangles in the susceptibility maps. By default (`SMOOTH_RAINFALL = True`), the 0.05° cells are rebuilt and bilinearly interpolated to 30 m. *The 0.05° resolution matches CHIRPS, but the source product is not recorded in the file; please confirm it.*
- **Coping-capacity layers.** These are emergency access, health access, night-time light, road density and population. They describe capacity to respond rather than the physical likelihood of flooding, so they are **excluded** from the susceptibility models by default. Set `INCLUDE_COPING_LAYERS = True` if you wish to test them. `fct_Emergency.tif` is in UTM 32N and is re-projected automatically.
- **SPI.** Its minimum value is −6.908 = ln(0.001), which appears to be a floor applied before the logarithm (an inference, not documented in the data).

## 1. Install packages (Colab)
`rasterio` is not pre-installed on Colab. `scikit-learn`, `xgboost`, `pandas` and `matplotlib` normally are; the cell installs any that are missing.

In [ ]:
import importlib, subprocess, sys
for pkg in ["rasterio", "xgboost", "sklearn", "joblib", "shap"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        name = "scikit-learn" if pkg == "sklearn" else pkg
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", name], check=True)
import rasterio, xgboost, sklearn
print("rasterio", rasterio.__version__, "| xgboost", xgboost.__version__, "| scikit-learn", sklearn.__version__)

## 2. Load the dataset
**Option A (recommended): Google Drive.** Upload `FCT_FLOOD.rar` (≈340 MB) to your Drive once, then set `RAR_PATH` below.
**Option B:** set `USE_DRIVE = False`. The cell will then open an upload dialog, which is slower for a file of this size.
If you have already extracted the archive on Drive, point `RAR_PATH` at the folder `FCT FLOOD` instead; it will be used directly.

In [ ]:
import os, glob, shutil, subprocess

USE_DRIVE = True
RAR_PATH = "/content/drive/MyDrive/FCT_FLOOD.rar"   # <-- edit to your Drive location
EXTRACT_TO = "/content/FCT_FLOOD_data"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    from google.colab import files
    up = files.upload()                       # choose FCT_FLOOD.rar
    RAR_PATH = os.path.join("/content", next(iter(up)))

def extract_rar(rar, dest):
    os.makedirs(dest, exist_ok=True)
    if shutil.which("unrar") is None:
        subprocess.run("apt-get -qq install -y unrar", shell=True)
    if shutil.which("unrar"):
        subprocess.run(["unrar", "x", "-o+", "-idq", rar, dest + "/"], check=True)
    else:                                      # fallback
        subprocess.run("apt-get -qq install -y p7zip-full", shell=True)
        subprocess.run(["7z", "x", "-y", f"-o{dest}", rar], check=True)

if os.path.isdir(RAR_PATH):
    search_root = RAR_PATH
else:
    extract_rar(RAR_PATH, EXTRACT_TO)
    search_root = EXTRACT_TO

# locate the folder that contains both 'geojson' and the conditioning factors
DATA_DIR = None
for root, dirs, _ in os.walk(search_root):
    if "geojson" in dirs and "flood_vulnerability_mapping-conditioning_factors" in dirs:
        DATA_DIR = root
        break
assert DATA_DIR, "Could not find the 'FCT FLOOD' folder after extraction."
OUT_DIR = "/content/fct_flood_ml_results"
print("DATA_DIR =", DATA_DIR)

## 3. Imports and configuration
Edit this cell to change the predictors, the treatment of distance to water, the class breaks or the tuning grids (section 6).

In [ ]:
import os, json, math, time, shutil, warnings
import joblib
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import geometry_mask
from rasterio.warp import transform as warp_transform
from scipy import ndimage
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV,
                                     RepeatedStratifiedKFold, StratifiedGroupKFold,
                                     cross_val_score)
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, roc_curve, accuracy_score, precision_score,
                             recall_score, f1_score, cohen_kappa_score, confusion_matrix)
from xgboost import XGBClassifier
warnings.filterwarnings("ignore", category=UserWarning)

SEED, TEST_SIZE, CV_FOLDS, N_JOBS = 42, 0.30, 5, -1

FACTOR_DIR    = "flood_vulnerability_mapping-conditioning_factors"
POINTS_FILE   = os.path.join("geojson", "Flood_points_1.geojson")
BOUNDARY_FILE = os.path.join("geojson", "FCT_Boundary.geojson")
LABEL_FIELD   = "flooded"

# Flood-conditioning (hazard) factors used as predictors
FACTORS = {
    "elevation":        "fct_elevation.tif",
    "slope":            "fct_slope.tif",
    "TWI":              "fct_TWI.tif",
    "SPI":              "fct_SPI.tif",
    "drainage_density": "fct_drainage_density.tif",
    "dist_to_water":    "fct_dist_to_water.tif",
    "rainfall":         "fct_rainfall.tif",
    "NDVI":             "fct_NDVI.tif",
    "landcover":        "fct_landcover.tif",
    "built_up":         "fct_built_up.tif",
}
# Coping-capacity layers (excluded from susceptibility models by default)
COPING_LAYERS = {
    "emergency_access": "fct_Emergency.tif",   # EPSG:32632 -> re-projected on the fly
    "health_access":    "health.tif",
    "night_light":      "fctlight.tif",
    "road_density":     "fct_road_density.tif",
    "population":       "fct_pop.tif",
}
INCLUDE_COPING_LAYERS = False

REFERENCE_FACTOR = "elevation"    # all layers are aligned to this grid
CATEGORICAL      = ["landcover"]  # trees use the class codes directly
LANDCOVER_NODATA = 0              # 0 = outside the clipped area
WATER_CLASS      = 80             # ESA WorldCover permanent water bodies

# fct_rainfall.tif is a coarse 0.05-degree grid (~5.5 km cells) copied onto 30 m pixels;
# used as-is, its cell edges print straight lines on the susceptibility maps.
# True = rebuild the 0.05-degree cells and bilinearly interpolate them to the 30 m grid.
SMOOTH_RAINFALL    = True
RAINFALL_CELL_DEG  = 0.05

# "recompute" (default) | "original" (not recommended) | "drop"
DIST_TO_WATER_MODE   = "recompute"
MAX_PLAUSIBLE_DIST_M = 200_000

# Susceptibility classes: "equal_interval" (0-0.2-0.4-0.6-0.8-1) or "quantile"
CLASS_METHOD  = "equal_interval"
CLASS_NAMES   = ["Very low", "Low", "Moderate", "High", "Very high"]
CLASS_COLOURS = ["#1a9641", "#a6d96a", "#ffffbf", "#fdae61", "#d7191c"]

SPATIAL_BLOCK_KM = 10.0
PRED_CHUNK_ROWS  = 256
OUT_NODATA       = -9999.0

FIG_DIR = os.path.join(OUT_DIR, "figures"); MAP_DIR = os.path.join(OUT_DIR, "maps")
for d in (OUT_DIR, FIG_DIR, MAP_DIR):
    os.makedirs(d, exist_ok=True)
FDIR = os.path.join(DATA_DIR, FACTOR_DIR)
LOG = []
def log(msg=""):
    print(msg); LOG.append(str(msg))

## 4. Helper functions

In [ ]:
def read_geojson(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_points(path):
    rows, skipped = [], 0
    for ft in read_geojson(path)["features"]:
        geom, props = ft.get("geometry"), ft.get("properties") or {}
        if geom is None or geom.get("type") != "Point" or props.get(LABEL_FIELD) is None:
            skipped += 1
            continue
        x, y = geom["coordinates"][:2]
        rows.append({"fid": props.get("fid"), "lon": x, "lat": y, "label": int(props[LABEL_FIELD])})
    return pd.DataFrame(rows), skipped

def boundary_geoms(path):
    return [ft["geometry"] for ft in read_geojson(path)["features"] if ft.get("geometry")]

def is_same_grid(src, ref):
    return (src.crs == ref["crs"] and src.width == ref["width"] and src.height == ref["height"]
            and np.allclose(tuple(src.transform)[:6], tuple(ref["transform"])[:6], atol=1e-9))

def align_to_reference(path, ref):
    """Nearest-neighbour resampling of a raster onto the reference grid:
    each reference pixel centre is located in the source raster."""
    with rasterio.open(path) as src:
        data, nodata, st = src.read(1), src.nodata, src.transform
        out = np.full((ref["height"], ref["width"]), np.nan, dtype=np.float32)
        rt = ref["transform"]
        if is_same_grid(src, ref):
            out[:] = data.astype(np.float32)
        elif src.crs == ref["crs"]:
            xs = rt.c + (np.arange(ref["width"]) + 0.5) * rt.a
            ys = rt.f + (np.arange(ref["height"]) + 0.5) * rt.e
            cols = np.floor((xs - st.c) / st.a).astype(np.int64)
            rows = np.floor((ys - st.f) / st.e).astype(np.int64)
            vc, vr = (cols >= 0) & (cols < src.width), (rows >= 0) & (rows < src.height)
            out[np.ix_(np.where(vr)[0], np.where(vc)[0])] = data[np.ix_(rows[vr], cols[vc])]
        else:   # different CRS (e.g. UTM 32N): transform pixel centres in row chunks
            xs = rt.c + (np.arange(ref["width"]) + 0.5) * rt.a
            for r0 in range(0, ref["height"], PRED_CHUNK_ROWS):
                r1 = min(r0 + PRED_CHUNK_ROWS, ref["height"])
                ys = rt.f + (np.arange(r0, r1) + 0.5) * rt.e
                X, Y = np.meshgrid(xs, ys)
                tx, ty = warp_transform(ref["crs"], src.crs, X.ravel().tolist(), Y.ravel().tolist())
                cols = np.floor((np.asarray(tx) - st.c) / st.a).astype(np.int64)
                rows = np.floor((np.asarray(ty) - st.f) / st.e).astype(np.int64)
                ok = (cols >= 0) & (cols < src.width) & (rows >= 0) & (rows < src.height)
                blk = np.full(X.size, np.nan, dtype=np.float32)
                blk[ok] = data[rows[ok], cols[ok]]
                out[r0:r1] = blk.reshape(X.shape)
    if nodata is not None:
        out[np.isclose(out, nodata)] = np.nan
    out[~np.isfinite(out)] = np.nan
    out[np.abs(out) > 1e30] = np.nan          # float-max no-data values
    return out

def smooth_coarse_layer(arr, ref, cell_deg):
    """Rebuild a coarse grid (cells aligned to multiples of cell_deg) from a layer that was
    copied onto the fine reference grid, then bilinearly interpolate it back to the fine grid."""
    from scipy.ndimage import map_coordinates, distance_transform_edt
    t, H, W = ref["transform"], ref["height"], ref["width"]
    xs = t.c + (np.arange(W) + 0.5) * t.a
    ys = t.f + (np.arange(H) + 0.5) * t.e
    x0 = math.floor(xs.min() / cell_deg) * cell_deg          # western edge of coarse grid
    y0 = math.ceil(ys.max() / cell_deg) * cell_deg           # northern edge of coarse grid
    ci = np.floor((xs - x0) / cell_deg).astype(int)
    ri = np.floor((y0 - ys) / cell_deg).astype(int)
    nr, nc = ri.max() + 1, ci.max() + 1
    # coarse cell value = median of the fine pixels falling in it
    coarse = np.full((nr, nc), np.nan)
    for r in range(nr):
        rows_r = np.where(ri == r)[0]
        if rows_r.size == 0:
            continue
        band = arr[rows_r]
        for c in range(nc):
            v = band[:, ci == c]
            v = v[np.isfinite(v)]
            if v.size:
                coarse[r, c] = np.median(v)
    # fill empty cells with nearest valid value so interpolation works up to the boundary
    miss = ~np.isfinite(coarse)
    if miss.any():
        idx = distance_transform_edt(miss, return_distances=False, return_indices=True)
        coarse = coarse[tuple(idx)]
    # fractional coarse-grid coordinates of fine pixel centres (cell centres at k + 0.5)
    fc = (xs - x0) / cell_deg - 0.5
    out = np.empty((H, W), dtype=np.float32)
    for r0 in range(0, H, PRED_CHUNK_ROWS):
        r1 = min(r0 + PRED_CHUNK_ROWS, H)
        fr = (y0 - ys[r0:r1]) / cell_deg - 0.5
        FR, FC = np.meshgrid(fr, fc, indexing="ij")
        out[r0:r1] = map_coordinates(coarse, [FR.ravel(), FC.ravel()], order=1,
                                     mode="nearest").reshape(FR.shape)
    out[~np.isfinite(arr)] = np.nan
    return out

def pixel_size_m(ref):
    t = ref["transform"]
    if ref["crs"].is_geographic:
        lat_mid = t.f + t.e * ref["height"] / 2.0
        return abs(t.a) * 111_320.0 * math.cos(math.radians(lat_mid)), abs(t.e) * 110_574.0
    return abs(t.a), abs(t.e)

def pixel_area_km2(ref):
    """Approximate per-row pixel area (km^2); varies with latitude on a geographic grid."""
    t = ref["transform"]
    if ref["crs"].is_geographic:
        lats = t.f + (np.arange(ref["height"]) + 0.5) * t.e
        return abs(t.a) * 111_320.0 * np.cos(np.radians(lats)) * abs(t.e) * 110_574.0 / 1e6
    return np.full(ref["height"], abs(t.a * t.e) / 1e6)

def points_to_rowcol(lon, lat, ref):
    t = ref["transform"]
    x, y = np.asarray(lon), np.asarray(lat)
    if not ref["crs"].is_geographic:
        x, y = map(np.asarray, warp_transform("EPSG:4326", ref["crs"], x.tolist(), y.tolist()))
    return (np.floor((y - t.f) / t.e).astype(np.int64),
            np.floor((x - t.c) / t.a).astype(np.int64))

def vif_table(X):
    """Variance inflation factor and tolerance (VIF > 10 / tolerance < 0.1 = problem)."""
    Z = (X - X.mean()) / X.std(ddof=0)
    out = []
    for c in Z.columns:
        y = Z[c].values
        A = np.column_stack([np.ones(len(Z)), Z.drop(columns=c).values])
        beta, *_ = np.linalg.lstsq(A, y, rcond=None)
        r2 = 1 - (y - A @ beta).var() / y.var() if y.var() > 0 else 0.0
        out.append({"factor": c, "R2": r2, "tolerance": 1 - r2,
                    "VIF": np.inf if r2 >= 1 else 1 / (1 - r2)})
    return pd.DataFrame(out).sort_values("VIF", ascending=False)

def metrics(y_true, prob, thr=0.5):
    pred = (prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {"AUC": roc_auc_score(y_true, prob), "Accuracy": accuracy_score(y_true, pred),
            "Precision": precision_score(y_true, pred, zero_division=0),
            "Recall (sensitivity)": recall_score(y_true, pred, zero_division=0),
            "Specificity": tn / (tn + fp) if (tn + fp) else np.nan,
            "F1": f1_score(y_true, pred, zero_division=0),
            "Kappa": cohen_kappa_score(y_true, pred), "TP": tp, "FP": fp, "TN": tn, "FN": fn}

def spatial_blocks(df, ref, block_km):
    dx, dy = pixel_size_m(ref)
    rows, cols = points_to_rowcol(df["lon"].values, df["lat"].values, ref)
    return (np.floor(rows * dy / (block_km * 1000)).astype(int) * 10_000 +
            np.floor(cols * dx / (block_km * 1000)).astype(int))

## 5. Reference grid, FCT mask and factor alignment
This step reads ≈450 MB of rasters and typically takes a few minutes.

In [ ]:
t0 = time.time()
layers = dict(FACTORS)
if INCLUDE_COPING_LAYERS:
    layers.update(COPING_LAYERS)
if DIST_TO_WATER_MODE == "drop":
    layers.pop("dist_to_water", None)

with rasterio.open(os.path.join(FDIR, FACTORS[REFERENCE_FACTOR])) as r:
    ref = {"crs": r.crs, "transform": r.transform, "width": r.width,
           "height": r.height, "profile": r.profile.copy()}
assert ref["crs"].is_geographic, "Boundary/points are WGS84; reference grid expected in EPSG:4326."
log(f"Reference grid: {REFERENCE_FACTOR} {ref['height']} x {ref['width']} px, CRS {ref['crs']}, "
    f"~{pixel_size_m(ref)[0]:.1f} m pixels")

geoms = boundary_geoms(os.path.join(DATA_DIR, BOUNDARY_FILE))
inside = geometry_mask(geoms, out_shape=(ref["height"], ref["width"]),
                       transform=ref["transform"], invert=True)
log(f"Pixels inside FCT boundary: {inside.sum():,}")

stack = {}
for name, fn in layers.items():
    arr = align_to_reference(os.path.join(FDIR, fn), ref)
    if name == "landcover":
        arr[arr == LANDCOVER_NODATA] = np.nan
    if name == "rainfall" and SMOOTH_RAINFALL:
        arr = smooth_coarse_layer(arr, ref, RAINFALL_CELL_DEG)
        log(f"  rainfall: {RAINFALL_CELL_DEG} deg cells rebuilt and bilinearly interpolated")
    arr[~inside] = np.nan
    stack[name] = arr
    log(f"  aligned {name:<17s} valid px inside FCT: {np.isfinite(arr[inside]).mean()*100:5.1f}%")
log(f"Alignment finished in {time.time()-t0:.0f}s")

## 6. Quality control: distance to water

In [ ]:
qc = {}
if "dist_to_water" in stack:
    d = stack["dist_to_water"]
    bad = np.nanmean(d[inside] > MAX_PLAUSIBLE_DIST_M) * 100
    qc["dist_to_water_pct_implausible_original"] = bad
    log(f"QC: {bad:.1f}% of FCT pixels in {FACTORS['dist_to_water']} exceed "
        f"{MAX_PLAUSIBLE_DIST_M/1000:.0f} km (implausible inside the FCT).")
    if DIST_TO_WATER_MODE == "recompute":
        lc_full = align_to_reference(os.path.join(FDIR, FACTORS["landcover"]), ref)
        water = lc_full == WATER_CLASS
        assert water.sum() > 0, "No water pixels in land cover; set DIST_TO_WATER_MODE = 'drop'."
        dx, dy = pixel_size_m(ref)
        new = ndimage.distance_transform_edt(~water, sampling=(dy, dx)).astype(np.float32)
        ok = inside & np.isfinite(d) & (d <= MAX_PLAUSIBLE_DIST_M)
        if ok.sum() > 100:
            idx = np.flatnonzero(ok)[:: max(1, ok.sum() // 200_000)]
            rr = np.corrcoef(new.ravel()[idx], d.ravel()[idx])[0, 1]
            qc["dist_to_water_corr_recomputed_vs_valid_original"] = rr
            log(f"  Recomputed Euclidean distance (m) to land-cover class {WATER_CLASS}; "
                f"r = {rr:.3f} against the valid part of the original raster.")
        new[~inside] = np.nan
        stack["dist_to_water"] = new
        del lc_full, water
    elif bad > 0:
        log("  WARNING: using the original dist_to_water including implausible values.")

feature_names = list(stack.keys())
print("Predictors:", feature_names)

## 7. Extract factor values at the inventory points
Points outside the FCT, points with missing factor values and duplicate points that fall in the same pixel are removed.

In [ ]:
pts, skipped = load_points(os.path.join(DATA_DIR, POINTS_FILE))
log(f"Inventory: {len(pts)} points read ({skipped} without geometry/label skipped)")
rows, cols = points_to_rowcol(pts["lon"].values, pts["lat"].values, ref)
in_grid = (rows >= 0) & (rows < ref["height"]) & (cols >= 0) & (cols < ref["width"])
rows_c, cols_c = np.clip(rows, 0, ref["height"] - 1), np.clip(cols, 0, ref["width"] - 1)
pts["in_fct"] = in_grid & inside[rows_c, cols_c]
for f in feature_names:
    pts[f] = np.where(in_grid, stack[f][rows_c, cols_c], np.nan)
pts["dup_pixel"] = pd.Series(rows_c * ref["width"] + cols_c).duplicated(keep="first").values

log(f"  outside FCT: {(~pts['in_fct']).sum()}, missing factor value: "
    f"{pts[feature_names].isna().any(axis=1).sum()}, duplicate pixel: {pts['dup_pixel'].sum()}")
data = pts[pts["in_fct"] & ~pts[feature_names].isna().any(axis=1) & ~pts["dup_pixel"]].reset_index(drop=True)
for c in CATEGORICAL:
    if c in data:
        data[c] = data[c].round().astype(int)
log(f"  modelling sample: {len(data)} points (flood = {(data.label == 1).sum()}, "
    f"non-flood = {(data.label == 0).sum()})")
data.to_csv(os.path.join(OUT_DIR, "training_samples_extracted.csv"), index=False)
X, y = data[feature_names].astype(float), data["label"].values
data[feature_names + ["label"]].describe().T.round(3)

## 8. Multicollinearity diagnostics

In [ ]:
num_feats = [f for f in feature_names if f not in CATEGORICAL]
vif = vif_table(X[num_feats])
vif.to_csv(os.path.join(OUT_DIR, "multicollinearity_VIF.csv"), index=False)
log("Multicollinearity (VIF > 10 or tolerance < 0.1 indicates a problem):")
log(vif.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

corr = X[num_feats].corr(method="pearson")
corr.to_csv(os.path.join(OUT_DIR, "correlation_matrix.csv"))
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_feats)), num_feats, rotation=45, ha="right")
ax.set_yticks(range(len(num_feats)), num_feats)
for i in range(len(num_feats)):
    for j in range(len(num_feats)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.8, label="Pearson r")
ax.set_title("Correlation among flood conditioning factors")
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "correlation_matrix.png"), dpi=300); plt.show()

## 9. Model definition, train–test split and hyper-parameter tuning
The grids are deliberately modest so that the notebook runs within a free Colab session; widen them if time permits.

In [ ]:
num = [f for f in feature_names if f not in CATEGORICAL]
cat = [f for f in feature_names if f in CATEGORICAL]
MODELS = {
    "RF": (RandomForestClassifier(random_state=SEED, n_jobs=N_JOBS),
           {"n_estimators": [300, 500], "max_depth": [None, 10, 20],
            "min_samples_leaf": [1, 3, 5], "max_features": ["sqrt", 0.5]}),
    "XGBoost": (XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                              random_state=SEED, n_jobs=N_JOBS, tree_method="hist"),
                {"n_estimators": [200, 400], "max_depth": [3, 5], "learning_rate": [0.03, 0.1],
                 "subsample": [0.8, 1.0], "colsample_bytree": [0.7, 1.0]}),
    "CART": (DecisionTreeClassifier(random_state=SEED),
             {"criterion": ["gini", "entropy"], "max_depth": [3, 5, 7, 10, None],
              "min_samples_leaf": [1, 5, 10], "ccp_alpha": [0.0, 0.005, 0.01]}),
}

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED)
log(f"Train: {len(y_tr)}  Test: {len(y_te)}  (stratified {int((1-TEST_SIZE)*100)}/{int(TEST_SIZE*100)})")

fitted, rows_metrics, best_params, roc_data = {}, [], {}, {}
inner_cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
for name, (est, grid) in MODELS.items():
    ts = time.time()
    gs = GridSearchCV(est, grid, scoring="roc_auc", cv=inner_cv, n_jobs=N_JOBS, refit=True)
    gs.fit(X_tr, y_tr)
    best = gs.best_estimator_
    fitted[name], best_params[name] = best, gs.best_params_
    p_tr, p_te = best.predict_proba(X_tr)[:, 1], best.predict_proba(X_te)[:, 1]
    m_tr, m_te = metrics(y_tr, p_tr), metrics(y_te, p_te)
    rows_metrics += [{"Model": name, "Dataset": "Training", **m_tr},
                     {"Model": name, "Dataset": "Testing", **m_te}]
    roc_data[name] = (roc_curve(y_te, p_te), m_te["AUC"], roc_curve(y_tr, p_tr), m_tr["AUC"])
    log(f"  {name:<8s} CV-AUC {gs.best_score_:.3f} | test AUC {m_te['AUC']:.3f} | "
        f"{time.time()-ts:.0f}s | best {gs.best_params_}")
    joblib.dump(best, os.path.join(OUT_DIR, f"model_{name}.joblib"))

met = pd.DataFrame(rows_metrics)
met.to_csv(os.path.join(OUT_DIR, "model_performance.csv"), index=False)
with open(os.path.join(OUT_DIR, "best_hyperparameters.json"), "w") as f:
    json.dump({k: {kk: (vv.item() if isinstance(vv, np.generic) else vv) for kk, vv in v.items()}
               for k, v in best_params.items()}, f, indent=2)
log("\nValidation metrics (threshold 0.5):")
log(met.drop(columns=["TP", "FP", "TN", "FN"]).to_string(index=False, float_format=lambda v: f"{v:.3f}"))

## 10. Robustness: repeated random CV, spatial-block CV and ROC curves
Spatial-block CV holds out whole 10 km blocks (`SPATIAL_BLOCK_KM`). It gives a more conservative estimate than random CV because nearby, spatially autocorrelated points cannot sit in both the training and test folds. *Note: the hyper-parameters were tuned on the training split, so these whole-sample CV scores may be slightly optimistic.*

In [ ]:
rcv = RepeatedStratifiedKFold(n_splits=CV_FOLDS, n_repeats=5, random_state=SEED)
groups = spatial_blocks(data, ref, SPATIAL_BLOCK_KM)
scv = StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
rob = []
for name, est in fitted.items():
    a_r = cross_val_score(est, X, y, scoring="roc_auc", cv=rcv, n_jobs=N_JOBS)
    a_s = cross_val_score(est, X, y, scoring="roc_auc", cv=scv, groups=groups, n_jobs=N_JOBS)
    rob.append({"Model": name, "Random CV AUC mean": a_r.mean(), "Random CV AUC sd": a_r.std(),
                f"Spatial-block ({SPATIAL_BLOCK_KM:g} km) CV AUC mean": a_s.mean(),
                f"Spatial-block ({SPATIAL_BLOCK_KM:g} km) CV AUC sd": a_s.std()})
rob = pd.DataFrame(rob)
rob.to_csv(os.path.join(OUT_DIR, "cross_validation_auc.csv"), index=False)
log(f"Spatial blocks used: {len(np.unique(groups))}")
log(rob.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

MODEL_COLOURS = {"RF": "#2a78d6", "XGBoost": "#eb6834", "CART": "#1baf7a"}

## 10b. ROC curve comparison of the three models
ROC curves for the training data (success rate) and the testing data (prediction rate). Each model's AUC is given with its **95% confidence interval** from DeLong's method (DeLong et al., 1988; fast algorithm of Sun & Xu, 2014). The models are then compared pairwise with the **DeLong test** for correlated ROC curves, since all three were evaluated on the same points; a p-value below 0.05 means the two AUCs differ significantly. Lines differ in both colour and dash style, so the figure stays readable in greyscale print.

In [ ]:
from scipy.stats import norm
ROC_MODELS = ["RF", "XGBoost", "CART"]
ROC_STYLES = {"RF": "-", "XGBoost": "--", "CART": "-."}

def _midrank(x):
    order = np.argsort(x, kind="mergesort"); z = x[order]; n = len(x)
    ranks = np.zeros(n); i = 0
    while i < n:
        j = i
        while j < n and z[j] == z[i]:
            j += 1
        ranks[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(n); out[order] = ranks
    return out

def delong(y_true, preds):
    """Fast DeLong (Sun & Xu, 2014). preds: (k, n) scores. Returns AUCs (k,) and covariance (k, k)."""
    y_true = np.asarray(y_true); preds = np.atleast_2d(preds)
    pos, neg = preds[:, y_true == 1], preds[:, y_true == 0]
    m, n = pos.shape[1], neg.shape[1]
    tx = np.array([_midrank(r) for r in pos]); ty = np.array([_midrank(r) for r in neg])
    tz = np.array([_midrank(r) for r in np.hstack([pos, neg])])
    aucs = tz[:, :m].sum(axis=1) / (m * n) - (m + 1.0) / (2.0 * n)
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.atleast_2d(np.cov(v01)) / m + np.atleast_2d(np.cov(v10)) / n
    return aucs, cov

roc_rows, pair_rows, curves = [], [], {}
for ds, Xd, yd in [("Training", X_tr, y_tr), ("Testing", X_te, y_te)]:
    P = np.vstack([fitted[m].predict_proba(Xd)[:, 1] for m in ROC_MODELS])
    aucs, cov = delong(yd, P)
    for i, m in enumerate(ROC_MODELS):
        se = np.sqrt(cov[i, i]); lo, hi = max(0, aucs[i] - 1.96 * se), min(1, aucs[i] + 1.96 * se)
        roc_rows.append({"Dataset": ds, "Model": m, "AUC": aucs[i], "SE": se, "95% CI lower": lo, "95% CI upper": hi})
        curves[(ds, m)] = (roc_curve(yd, P[i]), aucs[i], lo, hi)
    for i in range(len(ROC_MODELS)):
        for j in range(i + 1, len(ROC_MODELS)):
            var = cov[i, i] + cov[j, j] - 2 * cov[i, j]
            z = (aucs[i] - aucs[j]) / np.sqrt(var) if var > 0 else np.nan
            p = 2 * norm.sf(abs(z)) if np.isfinite(z) else np.nan
            pair_rows.append({"Dataset": ds, "Comparison": f"{ROC_MODELS[i]} vs {ROC_MODELS[j]}",
                              "AUC difference": aucs[i] - aucs[j], "z": z, "p-value": p,
                              "Significant (p < 0.05)": "yes" if p < 0.05 else "no"})
roc_tab = pd.DataFrame(roc_rows); pair_tab = pd.DataFrame(pair_rows)
roc_tab.to_csv(os.path.join(OUT_DIR, "roc_auc_delong_ci.csv"), index=False)
pair_tab.to_csv(os.path.join(OUT_DIR, "roc_delong_pairwise_tests.csv"), index=False)
log("AUC with 95% DeLong confidence intervals:")
log(roc_tab.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
log("Pairwise DeLong tests:")
log(pair_tab.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

fig = plt.figure(figsize=(18, 6.2))
gsp = fig.add_gridspec(1, 3, width_ratios=[1, 1, 0.8], wspace=0.28)
for c, (ds, ttl) in enumerate([("Training", "(a) Training data (success rate)"), ("Testing", "(b) Testing data (prediction rate)")]):
    ax = fig.add_subplot(gsp[0, c])
    ax.plot([0, 1], [0, 1], color="#9a9893", lw=1, ls=(0, (2, 2)), label="Random classifier (AUC = 0.500)")
    for m in ROC_MODELS:
        (fpr, tpr, _), auc_, lo, hi = curves[(ds, m)]
        ax.plot(fpr, tpr, color=MODEL_COLOURS[m], ls=ROC_STYLES[m], lw=2.2,
                label=f"{m}  AUC = {auc_:.3f} ({lo:.3f}–{hi:.3f})")
    ax.set_xlim(-0.01, 1.01); ax.set_ylim(-0.01, 1.01); ax.set_aspect("equal")
    ax.set_xlabel("False positive rate (1 − specificity)"); ax.set_ylabel("True positive rate (sensitivity)")
    ax.set_title(ttl, loc="left"); ax.grid(color="#e6e5e1", lw=0.6); ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="lower right", frameon=False, fontsize=9, title="AUC (95% CI)", title_fontsize=9)
ax = fig.add_subplot(gsp[0, 2])
for k, m in enumerate(ROC_MODELS):
    for ds, mk, off in [("Training", "o", -0.16), ("Testing", "s", 0.16)]:
        r = roc_tab[(roc_tab.Model == m) & (roc_tab.Dataset == ds)].iloc[0]
        ax.errorbar(r["AUC"], k + off, xerr=[[r["AUC"] - r["95% CI lower"]], [r["95% CI upper"] - r["AUC"]]],
                    fmt=mk, color=MODEL_COLOURS[m], mfc="white" if ds == "Training" else MODEL_COLOURS[m],
                    ms=7, capsize=4, lw=1.6)
        ax.annotate(f"{r['AUC']:.3f}", (r["95% CI upper"], k + off), xytext=(5, -3), textcoords="offset points", fontsize=8)
ax.set_yticks(range(len(ROC_MODELS)), ROC_MODELS); ax.set_ylim(len(ROC_MODELS) - 0.4, -1.0)
ax.axvline(0.5, color="#9a9893", lw=1, ls=(0, (2, 2)))
ax.set_xlim(0.5, 1.05); ax.set_xlabel("AUC with 95% confidence interval")
ax.set_title("(c) AUC comparison", loc="left"); ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#e6e5e1", lw=0.6)
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([], [], marker="o", ls="", mfc="white", mec="#52514e", color="#52514e", label="Training"),
                   Line2D([], [], marker="s", ls="", color="#52514e", label="Testing")], frameon=False,
          loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2)
te = pair_tab[pair_tab.Dataset == "Testing"]
txt = "DeLong tests (testing data):\n" + "\n".join(f"{r.Comparison}: p = {r['p-value']:.3f}" for _, r in te.iterrows())
ax.text(0.03, 0.985, txt, transform=ax.transAxes, fontsize=8.5, va="top", color="#0b0b0b",
        bbox=dict(boxstyle="round,pad=0.4", fc="#f7f7f5", ec="#d9d8d3", lw=0.8))
fig.savefig(os.path.join(FIG_DIR, "roc_curves.png"), dpi=300, bbox_inches="tight"); plt.show()

## 11. Factor importance
Built-in importances are not comparable across algorithms. Permutation importance (the mean drop in test AUC when a factor is shuffled, 30 repeats) is computed for all three models on the same basis.

In [ ]:
imp = pd.DataFrame(index=feature_names)
imp["RF (MDI)"] = fitted["RF"].feature_importances_
xg = pd.Series(fitted["XGBoost"].get_booster().get_score(importance_type="gain")).reindex(feature_names).fillna(0)
imp["XGBoost (gain)"] = (xg / (xg.sum() or 1)).values
imp["CART (Gini/entropy)"] = fitted["CART"].feature_importances_
for name, est in fitted.items():
    pi = permutation_importance(est, X_te, y_te, scoring="roc_auc", n_repeats=30,
                                random_state=SEED, n_jobs=N_JOBS)
    imp[f"{name} permutation (ΔAUC)"] = pi.importances_mean
imp.index.name = "factor"
imp.to_csv(os.path.join(OUT_DIR, "factor_importance.csv"))
log("Factor importance:")
log(imp.to_string(float_format=lambda v: f"{v:.3f}"))

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.ravel(), imp.columns):
    s = imp[col].sort_values()
    ax.barh(s.index, s.values, color="#3b6ea5"); ax.set_title(col, fontsize=10); ax.axvline(0, color="k", lw=0.6)
for ax in axes.ravel()[len(imp.columns):]:
    ax.axis("off")
fig.suptitle("Relative importance of flood conditioning factors", fontsize=13)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "factor_importance.png"), dpi=300); plt.show()

## 12. Wall-to-wall susceptibility prediction and GeoTIFF export
This step predicts ≈8.4 million pixels with each of the three models and is the slowest part of the notebook; expect several minutes per model on a free Colab runtime. The outputs are a probability GeoTIFF (0–1) and a five-class GeoTIFF (1 = very low … 5 = very high) per model.

In [ ]:
valid = inside.copy()
for f in feature_names:
    valid &= np.isfinite(stack[f])
log(f"Pixels with complete data: {valid.sum():,} ({valid.sum()/inside.sum()*100:.1f}% of FCT)")
area_row = pixel_area_km2(ref)
prof = ref["profile"].copy()
prof.update(driver="GTiff", dtype="float32", count=1, nodata=OUT_NODATA, compress="lzw",
            tiled=True, blockxsize=256, blockysize=256)
prof_cls = dict(prof, dtype="uint8", nodata=0)
pr, pc = points_to_rowcol(data["lon"].values, data["lat"].values, ref)
lab = data["label"].values
area_rows, prob_maps, class_breaks = [], {}, {}
for name, est in fitted.items():
    ts = time.time()
    prob = np.full((ref["height"], ref["width"]), np.nan, dtype=np.float32)
    for r0 in range(0, ref["height"], PRED_CHUNK_ROWS):
        r1 = min(r0 + PRED_CHUNK_ROWS, ref["height"])
        v = valid[r0:r1]
        if not v.any():
            continue
        Xc = pd.DataFrame({f: stack[f][r0:r1][v] for f in feature_names})
        for c in CATEGORICAL:
            if c in Xc:
                Xc[c] = Xc[c].round().astype(int)
        blk = prob[r0:r1]
        blk[v] = est.predict_proba(Xc)[:, 1]
    fin = np.isfinite(prob)
    if CLASS_METHOD == "quantile":
        inner = np.quantile(prob[fin], [0.2, 0.4, 0.6, 0.8])
    else:
        inner = np.array([0.2, 0.4, 0.6, 0.8])
    class_breaks[name] = inner.tolist()
    cls = np.zeros(prob.shape, dtype=np.uint8)
    cls[fin] = np.digitize(prob[fin], inner) + 1
    with rasterio.open(os.path.join(MAP_DIR, f"FCT_flood_susceptibility_{name}.tif"), "w", **prof) as dst:
        dst.write(np.where(fin, prob, OUT_NODATA).astype(np.float32), 1)
    with rasterio.open(os.path.join(MAP_DIR, f"FCT_flood_susceptibility_{name}_classes.tif"), "w", **prof_cls) as dst:
        dst.write(cls, 1)
    pix_area = np.broadcast_to(area_row[:, None], cls.shape)
    tot, pcls = pix_area[fin].sum(), cls[pr, pc]
    for k, cname in enumerate(CLASS_NAMES, start=1):
        a = pix_area[cls == k].sum()
        nfl = int(((pcls == k) & (lab == 1)).sum())
        area_rows.append({"Model": name, "Class": cname, "Area (km2)": a, "Area (%)": 100 * a / tot,
                          "Flood points in class": nfl,
                          "Flood points (%)": 100 * nfl / max(1, (lab == 1).sum())})
    prob_maps[name] = (prob, cls)
    log(f"  {name:<8s} predicted in {time.time()-ts:.0f}s; class breaks {np.round(inner, 3).tolist()}")

areas = pd.DataFrame(area_rows)
# share of flood points / share of area (> 1 means floods concentrate in that class)
areas["Flood density ratio"] = areas["Flood points (%)"] / areas["Area (%)"].replace(0, np.nan)
areas.to_csv(os.path.join(OUT_DIR, "susceptibility_class_areas.csv"), index=False)
log("\nSusceptibility class areas (approximate km^2; flood points include training and test points):")
log(areas.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

## 13. Susceptibility maps and class-area chart

In [ ]:
t = ref["transform"]
extent = [t.c, t.c + t.a * ref["width"], t.f + t.e * ref["height"], t.f]
cmap = ListedColormap(CLASS_COLOURS); cmap.set_bad("white")
norm = BoundaryNorm(np.arange(0.5, 6.5, 1), cmap.N)
fl, nf = data[data.label == 1], data[data.label == 0]
fig, axes = plt.subplots(1, 3, figsize=(21, 8))
axes = np.atleast_1d(axes)
for ax, (name, (prob, cls)) in zip(axes.ravel(), prob_maps.items()):
    ax.imshow(np.ma.masked_equal(cls, 0)[::2, ::2], cmap=cmap, norm=norm, extent=extent, interpolation="nearest")
    for g in geoms:
        for poly in (g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]):
            ring = np.asarray(poly[0]); ax.plot(ring[:, 0], ring[:, 1], color="k", lw=0.8)
    ax.scatter(fl.lon, fl.lat, s=8, c="#08306b", marker="o", label="Flood point", lw=0)
    ax.scatter(nf.lon, nf.lat, s=8, c="#737373", marker="^", label="Non-flood point", lw=0)
    auc = met[(met.Model == name) & (met.Dataset == "Testing")]["AUC"].iloc[0]
    ax.set_title(f"{name} (test AUC = {auc:.3f})", fontsize=12)
    ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
handles = [Patch(color=c, label=n) for c, n in zip(CLASS_COLOURS, CLASS_NAMES)]
handles += axes[0].get_legend_handles_labels()[0]
fig.suptitle("Flood susceptibility of the Federal Capital Territory, Abuja", fontsize=15)
fig.tight_layout(rect=[0, 0.10, 1, 0.95])
fig.legend(handles=handles, loc="lower center", ncol=7, frameon=False, fontsize=11, bbox_to_anchor=(0.5, 0.01))
fig.savefig(os.path.join(FIG_DIR, "susceptibility_maps.png"), dpi=300); plt.show()

piv = areas.pivot(index="Model", columns="Class", values="Area (%)")[CLASS_NAMES]
ax = piv.plot(kind="bar", stacked=True, color=CLASS_COLOURS, figsize=(8, 5), edgecolor="k", lw=0.3)
ax.set_ylabel("Share of FCT area (%)"); ax.set_xlabel(""); ax.set_title("Area by susceptibility class")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left"); plt.xticks(rotation=0); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "class_area_share.png"), dpi=300); plt.show()

with open(os.path.join(OUT_DIR, "qc_summary.json"), "w") as f:
    json.dump({**{k: float(v) for k, v in qc.items()}, "class_method": CLASS_METHOD,
               "class_breaks": class_breaks}, f, indent=2)
with open(os.path.join(OUT_DIR, "run_log.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(LOG))
log(f"Total run time: {(time.time()-t0)/60:.1f} min")

## 14. Explainable machine learning: SHAP values for RF, XGBoost and CART
The importance charts in section 11 show *how much* each factor matters. This section uses SHAP (SHapley Additive exPlanations) to show *how* each factor acts in each model: whether higher values raise or lower susceptibility, and at what value the effect changes sign.

* **RF and CART:** exact path-dependent TreeSHAP from the `shap` package (`shap.TreeExplainer`). Values are contributions to the predicted **probability** of flooding.
* **XGBoost:** XGBoost's built-in exact TreeSHAP (`pred_contribs=True`). Values are contributions to the predicted **log-odds** of flooding. Because the units differ, the models are compared through each factor's **share of total mean |SHAP|**.
* For every model and point, the SHAP values plus the model's base value add up exactly to the model's prediction; this is checked below.
* **Thresholds:** for each continuous factor, the value at which the binned mean SHAP value changes sign. Treat these as approximate, data-driven thresholds, not physical constants.

In [ ]:
import shap
import xgboost as xgb
from scipy.stats import spearmanr
from sklearn.inspection import partial_dependence
from matplotlib.colors import LinearSegmentedColormap

SHAP_MODELS = ["RF", "XGBoost", "CART"]
LC_NAMES = {10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland", 50: "Built-up",
            60: "Bare/sparse", 70: "Snow/ice", 80: "Water", 90: "Wetland", 95: "Mangroves", 100: "Moss/lichen"}
N_DEPENDENCE = 4           # continuous factors shown in the dependence and partial-dependence plots
DIVERGING = LinearSegmentedColormap.from_list("div", ["#2a78d6", "#f0efec", "#e34948"])
SHAP_UNITS = {"RF": "probability", "XGBoost": "log-odds", "CART": "probability"}

def shap_for(name):
    """Return (SHAP matrix n x p, base value) for one fitted model on X."""
    est = fitted[name]
    if name == "XGBoost":
        booster = est.get_booster(); dm = xgb.DMatrix(X)
        contrib = booster.predict(dm, pred_contribs=True)          # last column = bias
        sv, base = contrib[:, :-1], float(contrib[0, -1])
        err = np.abs(contrib.sum(axis=1) - booster.predict(dm, output_margin=True)).max()
    else:
        ex = shap.TreeExplainer(est)
        sv = ex.shap_values(X, check_additivity=False)
        if isinstance(sv, list):                                   # older shap: list per class
            sv = sv[1]
        sv = np.asarray(sv)
        if sv.ndim == 3:                                           # newer shap: (n, p, classes)
            sv = sv[:, :, 1]
        base = float(np.atleast_1d(ex.expected_value)[-1])
        err = np.abs(sv.sum(axis=1) + base - est.predict_proba(X)[:, 1]).max()
    log(f"  {name:<8s} SHAP additivity check: max |sum(SHAP) + base - prediction| = {err:.2e} ({SHAP_UNITS[name]})")
    return pd.DataFrame(sv, columns=feature_names), base

log("SHAP values:")
shap_all, shap_base = {}, {}
for m in SHAP_MODELS:
    shap_all[m], shap_base[m] = shap_for(m)
    shap_all[m].to_csv(os.path.join(OUT_DIR, f"shap_values_{m}.csv"), index=False)

# share of total mean |SHAP| per model (unit-free, comparable across models)
share = pd.DataFrame({m: shap_all[m].abs().mean() / shap_all[m].abs().mean().sum() * 100 for m in SHAP_MODELS})
share.index.name = "factor"
share = share.loc[share.mean(axis=1).sort_values(ascending=False).index]
share.to_csv(os.path.join(OUT_DIR, "shap_importance_share_by_model.csv"))
log("Share of total mean |SHAP| (%):")
log(share.to_string(float_format=lambda v: f"{v:.1f}"))

# ---- (1) SHAP summary (beeswarm) for the three models -------------------------------
fig, axes = plt.subplots(1, 3, figsize=(20, 6.8))
rng = np.random.default_rng(SEED)
for ax, m in zip(axes, SHAP_MODELS):
    sdf = shap_all[m]
    order_m = sdf.abs().mean().sort_values(ascending=True).index.tolist()     # most important on top
    for i, f in enumerate(order_m):
        sv = sdf[f].values; yj = i + rng.uniform(-0.32, 0.32, len(sv))
        if f in CATEGORICAL:
            ax.scatter(sv, yj, s=8, c="#9a9893", lw=0, alpha=0.8)
        else:
            pr = pd.Series(X[f].values).rank(pct=True).values
            ax.scatter(sv, yj, s=8, c=pr, cmap=DIVERGING, vmin=0, vmax=1, lw=0)
    ax.set_yticks(range(len(order_m)), order_m); ax.axvline(0, color="#52514e", lw=0.8)
    ax.set_xlabel(f"SHAP value (contribution to {SHAP_UNITS[m]})")
    auc = met[(met.Model == m) & (met.Dataset == "Testing")]["AUC"].iloc[0]
    ax.set_title(f"({'abc'[SHAP_MODELS.index(m)]}) {m} (test AUC = {auc:.3f})", loc="left")
    ax.spines[["top", "right"]].set_visible(False)
fig.subplots_adjust(left=0.08, right=0.92, wspace=0.42, bottom=0.12, top=0.90)
cax = fig.add_axes([0.935, 0.30, 0.009, 0.40])
sm = plt.cm.ScalarMappable(cmap=DIVERGING); sm.set_array([0, 1])
cb = fig.colorbar(sm, cax=cax, ticks=[0, 1]); cb.ax.set_yticklabels(["Low", "High"])
cb.set_label("Factor value (percentile)\ngrey = land cover (categorical)"); cb.outline.set_visible(False)
fig.suptitle("SHAP summary of the flood conditioning factors", fontsize=13)
fig.savefig(os.path.join(FIG_DIR, "shap_summary_three_models.png"), dpi=300); plt.show()

# ---- (2) comparison of factor importance across the three models -----------------------
fig, ax = plt.subplots(figsize=(9, 6.5))
yy = np.arange(len(share)); h = 0.26
for k, m in enumerate(SHAP_MODELS):
    ax.barh(yy + (k - 1) * h, share[m].values, height=h - 0.03, color=MODEL_COLOURS[m], label=m)
ax.set_yticks(yy, share.index); ax.invert_yaxis()
ax.set_xlabel("Share of total mean |SHAP value| (%)")
ax.set_title("Relative importance of the conditioning factors (SHAP)", loc="left")
ax.legend(frameon=False, loc="lower right"); ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color="#e6e5e1", lw=0.6); ax.set_axisbelow(True)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "shap_importance_comparison.png"), dpi=300); plt.show()

# ---- (3) direction and sign-change thresholds for every model ------------------------------
def sign_changes(x, sv, n_bins=10):
    q = np.unique(np.quantile(x, np.linspace(0, 1, n_bins + 1)))
    if len(q) < 3:
        return [], None, None
    idx = np.clip(np.digitize(x, q[1:-1]), 0, len(q) - 2)
    centres = np.array([np.median(x[idx == k]) for k in range(len(q) - 1) if (idx == k).any()])
    means = np.array([sv[idx == k].mean() for k in range(len(q) - 1) if (idx == k).any()])
    out = []
    for k in range(len(means) - 1):
        if np.sign(means[k]) != np.sign(means[k + 1]) and means[k] != 0:
            x0, x1, y0, y1 = centres[k], centres[k + 1], means[k], means[k + 1]
            out.append(x0 - y0 * (x1 - x0) / (y1 - y0))
    return out, centres, means

thr_rows = []
for m in SHAP_MODELS:
    for f in [f for f in share.index if f not in CATEGORICAL]:
        x = X[f].values; sv = shap_all[m][f].values
        if np.abs(sv).max() < 1e-12:
            thr_rows.append({"model": m, "factor": f, "share of mean |SHAP| (%)": 0.0,
                             "direction": "not used by the model", "Spearman rho": np.nan, "sign change(s)": "none"})
            continue
        rho = spearmanr(x, sv).correlation
        cross, _, _ = sign_changes(x, sv)
        thr_rows.append({"model": m, "factor": f, "share of mean |SHAP| (%)": share.loc[f, m],
                         "direction": "higher values raise susceptibility" if rho > 0 else "higher values lower susceptibility",
                         "Spearman rho": rho,
                         "sign change(s)": "; ".join(f"{c:.3g}" for c in cross) if cross else "none"})
thr = pd.DataFrame(thr_rows)
thr.to_csv(os.path.join(OUT_DIR, "shap_thresholds_three_models.csv"), index=False)
log("SHAP-based direction and thresholds:")
log(thr.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

# ---- (4) dependence plots: leading factors (rows) x models (columns) ---------------------
show = [f for f in share.index if f not in CATEGORICAL][:N_DEPENDENCE]
fig, axes = plt.subplots(len(show), 3, figsize=(16, 3.6 * len(show)), squeeze=False)
for r, f in enumerate(show):
    for c, m in enumerate(SHAP_MODELS):
        ax = axes[r, c]; x = X[f].values; sv = shap_all[m][f].values
        ax.scatter(x, sv, s=9, c=MODEL_COLOURS[m], alpha=0.55, lw=0)
        cross, centres, means = sign_changes(x, sv)
        if centres is not None:
            ax.plot(centres, means, color="#0b0b0b", lw=1.8)
        for j, cr in enumerate(cross):
            ax.axvline(cr, color="#e34948", ls="--", lw=1.1)
            ax.annotate(f"{cr:.3g}", (cr, 1), xycoords=("data", "axes fraction"), xytext=(3, -12 - 11 * j),
                        textcoords="offset points", color="#e34948", fontsize=8)
        ax.axhline(0, color="#52514e", lw=0.7); ax.spines[["top", "right"]].set_visible(False)
        if r == 0:
            ax.set_title(f"{m} (SHAP in {SHAP_UNITS[m]})", loc="left")
        if c == 0:
            ax.set_ylabel(f"{f}\nSHAP value")
        ax.set_xlabel(f, fontsize=9)
fig.suptitle("SHAP dependence of the leading factors (red dashed line = sign change of the binned mean)", fontsize=12)
fig.tight_layout(rect=[0, 0, 1, 0.97]); fig.savefig(os.path.join(FIG_DIR, "shap_dependence_three_models.png"), dpi=300); plt.show()

# land cover: mean SHAP per class for each model
if CATEGORICAL:
    f = CATEGORICAL[0]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
    for ax, m in zip(axes, SHAP_MODELS):
        g = pd.DataFrame({"c": X[f].round().astype(int).values, "s": shap_all[m][f].values}).groupby("c")["s"].agg(["mean", "count"])
        labels = [f"{LC_NAMES.get(cc, cc)}\n(n={n})" for cc, n in zip(g.index, g["count"])]
        ax.bar(labels, g["mean"], color=["#e34948" if v > 0 else "#2a78d6" for v in g["mean"]])
        ax.axhline(0, color="#52514e", lw=0.8); ax.set_title(f"{m}", loc="left")
        ax.set_ylabel(f"Mean SHAP ({SHAP_UNITS[m]})"); ax.tick_params(axis="x", labelsize=7, labelrotation=35)
        ax.spines[["top", "right"]].set_visible(False)
    fig.suptitle("Effect of land-cover class on predicted susceptibility (mean SHAP)", fontsize=12)
    fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "shap_landcover_three_models.png"), dpi=300); plt.show()

# ---- (5) partial dependence for the three models ---------------------------------------------
fig, axes = plt.subplots(1, len(show), figsize=(4.6 * len(show), 4.2), squeeze=False)
pd_rows = []
for ax, f in zip(axes[0], show):
    for m, ls in zip(SHAP_MODELS, ["-", "--", "-."]):
        r_ = partial_dependence(fitted[m], X, [f], grid_resolution=50, kind="average")
        grid = (r_["grid_values"] if "grid_values" in r_ else r_["values"])[0]
        avg = r_["average"][0]
        ax.plot(grid, avg, color=MODEL_COLOURS[m], ls=ls, lw=2, label=m)
        pd_rows += [{"model": m, "factor": f, "value": g_, "mean predicted probability": a_} for g_, a_ in zip(grid, avg)]
    ax.set_xlabel(f); ax.set_ylabel("Mean predicted flood probability")
    ax.spines[["top", "right"]].set_visible(False)
axes[0, 0].legend(frameon=False)
fig.suptitle("Partial dependence of flood probability on the leading factors", fontsize=12)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "partial_dependence_three_models.png"), dpi=300); plt.show()
pd.DataFrame(pd_rows).to_csv(os.path.join(OUT_DIR, "partial_dependence_three_models.csv"), index=False)

## 15. Model agreement and prediction spread
Maps from different algorithms rarely agree everywhere. This section counts, for every pixel, how many models place it in the high or very high class (probability ≥ `AGREE_THRESHOLD`). It also maps the standard deviation of the predicted probabilities across models. Areas flagged by all models are the most robust priorities; areas where the models disagree deserve field verification.

In [ ]:
AGREE_MODELS = [m for m in ["RF", "XGBoost", "CART"] if m in prob_maps]
AGREE_THRESHOLD = 0.6

P = np.stack([prob_maps[m][0] for m in AGREE_MODELS])
fin_all = np.all(np.isfinite(P), axis=0)
agree = np.where(fin_all, (P >= AGREE_THRESHOLD).sum(axis=0), 255).astype(np.uint8)
spread = np.where(fin_all, P.std(axis=0), np.nan).astype(np.float32)
del P
with rasterio.open(os.path.join(MAP_DIR, "FCT_model_agreement_high.tif"), "w", **dict(prof_cls, nodata=255)) as dst:
    dst.write(agree, 1)
with rasterio.open(os.path.join(MAP_DIR, "FCT_model_spread_sd.tif"), "w", **prof) as dst:
    dst.write(np.where(fin_all, spread, OUT_NODATA).astype(np.float32), 1)

pix_area = np.broadcast_to(area_row[:, None], agree.shape)
tot = pix_area[fin_all].sum()
ag_rows = [{"Models classing pixel as high/very high": k, "Area (km2)": pix_area[agree == k].sum(),
            "Area (%)": 100 * pix_area[agree == k].sum() / tot} for k in range(len(AGREE_MODELS) + 1)]
ag = pd.DataFrame(ag_rows)
ag.to_csv(os.path.join(OUT_DIR, "model_agreement_areas.csv"), index=False)
log(f"Model agreement ({', '.join(AGREE_MODELS)}; threshold {AGREE_THRESHOLD}):")
log(ag.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

n_m = len(AGREE_MODELS)
seq = LinearSegmentedColormap.from_list("seq", ["#f0efec", "#f2b8a2", "#e34948", "#9c2626"], N=n_m + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 7))
im1 = a1.imshow(np.ma.masked_equal(agree, 255)[::2, ::2], cmap=seq, vmin=-0.5, vmax=n_m + 0.5,
                extent=extent, interpolation="nearest")
cb1 = fig.colorbar(im1, ax=a1, ticks=range(n_m + 1), shrink=0.8); cb1.set_label("Number of models (high/very high)")
im2 = a2.imshow(np.ma.masked_invalid(spread)[::2, ::2], cmap="Greys", extent=extent, interpolation="nearest")
cb2 = fig.colorbar(im2, ax=a2, shrink=0.8); cb2.set_label("SD of predicted probability")
for ax, ttl in [(a1, "(a) Agreement on high susceptibility"), (a2, "(b) Spread between models")]:
    for g in geoms:
        for poly in (g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]):
            ring = np.asarray(poly[0]); ax.plot(ring[:, 0], ring[:, 1], color="k", lw=0.8)
    ax.set_title(ttl, loc="left"); ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "model_agreement.png"), dpi=300); plt.show()

## 16. Coping capacity and its overlap with flood susceptibility
Four indicators are combined into a **coping-capacity index (CCI)**:

| Indicator | File | Direction used |
|---|---|---|
| Emergency access | `fct_Emergency.tif` | higher value = lower capacity (behaves like distance: low in towns, high in remote areas) |
| Health access | `health.tif` | higher value = lower capacity (low in Abuja and Gwagwalada, high in remote areas) |
| Night-time light | `fctlight.tif` | higher value = higher capacity |
| Road density | `fct_road_density.tif` | higher value = higher capacity |

**Check the directions.** They were inferred by comparing values in the main towns with values in remote areas. The health layer is labelled "healt_acc" in its metadata, but its values are lowest in the towns, so it is treated as a travel-cost or distance-type measure. If your layer means the opposite, set its direction to `+1`.

Each indicator is rescaled to 0–1 between its 2nd and 98th percentiles inside the FCT, reversed where needed so that 1 always means high capacity, and averaged (equal weights unless `CCI_WEIGHTS` is set). The index is split into terciles (low, medium, high capacity) and crossed with three susceptibility classes from `SUSC_MODEL` to give a **3 × 3 bivariate map**. The priority hotspot is **high susceptibility with low coping capacity**. Population is counted on the population raster's own grid, so the totals are not inflated by resampling.

*Optional:* set `COUNCILS_FILE` to a GeoJSON of Area Council boundaries (WGS84) to get the hotspot area and population for each council.

In [ ]:
COPING_INDICATORS = {                 # name: (file, direction: +1 = higher is better, -1 = higher is worse)
    "emergency_access": ("fct_Emergency.tif", -1),
    "health_access":    ("health.tif", -1),
    "night_light":      ("fctlight.tif", +1),
    "road_density":     ("fct_road_density.tif", +1),
}
CCI_WEIGHTS = None                    # e.g. {"emergency_access": 0.3, "health_access": 0.3, "night_light": 0.2, "road_density": 0.2}
MIN_INDICATORS = 3                    # pixels with fewer valid indicators get no index
SUSC_MODEL = "RF"
SUSC_BREAKS = [0.4, 0.6]              # low < 0.4 <= moderate < 0.6 <= high
POP_FILE = "fct_pop.tif"
COUNCILS_FILE = None                  # e.g. "/content/drive/MyDrive/FCT_Area_Councils.geojson"
COUNCIL_NAME_FIELD = "name"

# ---- coping-capacity index ----------------------------------------------------------
norm_layers, wts = {}, {}
for name, (fn, direction) in COPING_INDICATORS.items():
    arr = align_to_reference(os.path.join(FDIR, fn), ref)
    arr[~inside] = np.nan
    v = arr[np.isfinite(arr)]
    lo, hi = np.percentile(v, [2, 98])
    n = np.clip((arr - lo) / (hi - lo), 0, 1)
    norm_layers[name] = (1 - n) if direction < 0 else n
    wts[name] = 1.0 if CCI_WEIGHTS is None else CCI_WEIGHTS[name]
    log(f"  {name:<17s} valid inside FCT: {np.isfinite(arr[inside]).mean()*100:5.1f}%  "
        f"(2nd–98th pct {lo:.4g}–{hi:.4g}, direction {direction:+d})")
num = np.zeros((ref["height"], ref["width"]), np.float32); den = np.zeros_like(num); cnt = np.zeros(num.shape, np.int8)
for name, n in norm_layers.items():
    ok = np.isfinite(n)
    num[ok] += wts[name] * n[ok]; den[ok] += wts[name]; cnt[ok] += 1
cci = np.where((cnt >= MIN_INDICATORS) & inside, num / np.maximum(den, 1e-9), np.nan).astype(np.float32)
del num, den, norm_layers
with rasterio.open(os.path.join(MAP_DIR, "FCT_coping_capacity_index.tif"), "w", **prof) as dst:
    dst.write(np.where(np.isfinite(cci), cci, OUT_NODATA).astype(np.float32), 1)

# ---- classes and bivariate code ------------------------------------------------------
t1, t2 = np.nanquantile(cci, [1 / 3, 2 / 3])
cap_vul = np.zeros(cci.shape, np.uint8)                  # 1 = high capacity ... 3 = low capacity
ok = np.isfinite(cci)
cap_vul[ok] = 3 - np.digitize(cci[ok], [t1, t2])
prob_s = prob_maps[SUSC_MODEL][0]
sus = np.zeros(prob_s.shape, np.uint8)                   # 1 = low ... 3 = high susceptibility
oks = np.isfinite(prob_s)
sus[oks] = np.digitize(prob_s[oks], SUSC_BREAKS) + 1
both = ok & oks
biv = np.zeros(cci.shape, np.uint8)
biv[both] = (cap_vul[both] - 1) * 3 + sus[both]          # 1..9; 9 = high susceptibility + low capacity
with rasterio.open(os.path.join(MAP_DIR, "FCT_bivariate_susceptibility_capacity.tif"), "w", **prof_cls) as dst:
    dst.write(biv, 1)
log(f"CCI terciles: {t1:.3f}, {t2:.3f}; susceptibility from {SUSC_MODEL} with breaks {SUSC_BREAKS}")

# ---- population on its own grid ----------------------------------------------------------
with rasterio.open(os.path.join(FDIR, POP_FILE)) as src:
    pop = src.read(1).astype(np.float64); pt = src.transform; pnd = src.nodata
if pnd is not None:
    pop[np.isclose(pop, pnd)] = 0
pop[~np.isfinite(pop) | (pop < 0)] = 0
pr_, pc_ = np.nonzero(pop)
plon = pt.c + (pc_ + 0.5) * pt.a; plat = pt.f + (pr_ + 0.5) * pt.e
rr, cc = points_to_rowcol(plon, plat, ref)
okp = (rr >= 0) & (rr < ref["height"]) & (cc >= 0) & (cc < ref["width"])
rr, cc, w = rr[okp], cc[okp], pop[pr_[okp], pc_[okp]]
inF = inside[rr, cc]
pop_total = w[inF].sum()
pop_by_biv = np.bincount(biv[rr, cc], weights=w, minlength=10)
log(f"Population inside the FCT (from {POP_FILE}): {pop_total:,.0f}")

SUS_LAB = ["Low", "Moderate", "High"]; CAP_LAB = ["High capacity", "Medium capacity", "Low capacity"]
rows = []
for cv in (1, 2, 3):
    for sv in (1, 2, 3):
        code = (cv - 1) * 3 + sv
        a = pix_area[biv == code].sum()
        rows.append({"Susceptibility": SUS_LAB[sv - 1], "Coping capacity": CAP_LAB[cv - 1], "Code": code,
                     "Area (km2)": a, "Area (%)": 100 * a / pix_area[biv > 0].sum(),
                     "Population": pop_by_biv[code], "Population (%)": 100 * pop_by_biv[code] / pop_total})
bv = pd.DataFrame(rows)
bv.to_csv(os.path.join(OUT_DIR, "bivariate_susceptibility_capacity.csv"), index=False)
log("Susceptibility × coping capacity (area and population):")
log(bv.to_string(index=False, float_format=lambda v: f"{v:,.2f}"))
hot = bv[bv.Code == 9].iloc[0]
log(f"PRIORITY HOTSPOT (high susceptibility, low capacity): {hot['Area (km2)']:,.1f} km² "
    f"({hot['Area (%)']:.1f}% of the FCT), about {hot['Population']:,.0f} people ({hot['Population (%)']:.1f}%).")

# ---- optional Area Council summary ---------------------------------------------------------
if COUNCILS_FILE and os.path.exists(COUNCILS_FILE):
    crow = []
    for ft in read_geojson(COUNCILS_FILE)["features"]:
        if not ft.get("geometry"):
            continue
        m = geometry_mask([ft["geometry"]], out_shape=(ref["height"], ref["width"]),
                          transform=ref["transform"], invert=True)
        in_c = m[rr, cc]
        crow.append({"Area Council": (ft.get("properties") or {}).get(COUNCIL_NAME_FIELD, "?"),
                     "Area (km2)": pix_area[m & (biv > 0)].sum(),
                     "Hotspot area (km2)": pix_area[m & (biv == 9)].sum(),
                     "Population": w[in_c].sum(),
                     "Population in hotspot": w[in_c & (biv[rr, cc] == 9)].sum()})
    cdf = pd.DataFrame(crow)
    cdf["Hotspot area (%)"] = 100 * cdf["Hotspot area (km2)"] / cdf["Area (km2)"]
    cdf["Population in hotspot (%)"] = 100 * cdf["Population in hotspot"] / cdf["Population"].replace(0, np.nan)
    cdf.to_csv(os.path.join(OUT_DIR, "hotspots_by_area_council.csv"), index=False)
    log(cdf.to_string(index=False, float_format=lambda v: f"{v:,.1f}"))
else:
    print("No Area Council file set (COUNCILS_FILE); council summary skipped.")

# ---- maps ---------------------------------------------------------------------------------------
# Optimised 3 x 3 palette: four corner colours blended in the perceptual OKLab space.
#   neutral pale lilac (low susceptibility, high capacity) -> vivid orange as susceptibility rises,
#   -> strong blue as capacity falls, -> deep wine for the priority hotspot (high susceptibility, low capacity).
# Distinctness (min OKLab difference between any two cells): 15.7 normal vision, 13.8 deuteranopia,
# 10.4 protanopia, 14.3 tritanopia.
BIV_COLOURS = [["#F7F2FB", "#FBB79D", "#F6772D"],   # high capacity:   low / moderate / high susceptibility
               ["#9DC2EB", "#AE8595", "#AC403F"],   # medium capacity
               ["#3A90D8", "#685486", "#68003B"]]   # low capacity (B high)
biv_cmap = ListedColormap(["#ffffff"] + [c for row in BIV_COLOURS for c in row])
# Both map panels use identical axes boxes, the same extent and an equal aspect ratio, so they render
# at exactly the same size; the colour bar and the bivariate legend sit outside the map frames.
CCI_CMAP = "Reds"                     # red ramp: pale = low coping capacity, dark red = high capacity
MAP_BOX_W, MAP_BOX_H, MAP_Y = 0.36, 0.82, 0.09
fig = plt.figure(figsize=(18, 7.8))
a1 = fig.add_axes([0.05, MAP_Y, MAP_BOX_W, MAP_BOX_H]); a2 = fig.add_axes([0.52, MAP_Y, MAP_BOX_W, MAP_BOX_H])
for ax in (a1, a2):
    ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3]); ax.set_aspect("equal")
im = a1.imshow(np.ma.masked_invalid(cci)[::2, ::2], cmap=CCI_CMAP, vmin=0, vmax=1, extent=extent, interpolation="nearest")
cax = fig.add_axes([0.05 + MAP_BOX_W + 0.012, 0.30, 0.011, 0.40])
cb = fig.colorbar(im, cax=cax); cb.set_label("Coping-capacity index (0 = low, 1 = high)"); cb.outline.set_visible(False)
a2.imshow(np.ma.masked_equal(biv, 0)[::2, ::2], cmap=biv_cmap, vmin=-0.5, vmax=9.5, extent=extent, interpolation="nearest")
for ax, ttl in [(a1, "(a) Coping-capacity index"), (a2, f"(b) Susceptibility ({SUSC_MODEL}) × coping capacity")]:
    for g in geoms:
        for poly in (g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]):
            ring = np.asarray(poly[0]); ax.plot(ring[:, 0], ring[:, 1], color="k", lw=0.8)
    ax.set_title(ttl, loc="left"); ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
lg = fig.add_axes([0.52 + MAP_BOX_W + 0.025, 0.38, 0.075, 0.22])
for cv in range(3):
    for sv in range(3):
        lg.add_patch(plt.Rectangle((sv, cv), 0.96, 0.96, color=BIV_COLOURS[cv][sv]))
lg.set_xlim(0, 3); lg.set_ylim(0, 3); lg.set_aspect("equal")
lg.set_xticks([0.5, 1.5, 2.5], ["Low", "Mod.", "High"], fontsize=8)
lg.set_yticks([0.5, 1.5, 2.5], ["High", "Med.", "Low"], fontsize=8)
lg.set_xlabel("Susceptibility →", fontsize=8); lg.set_title("Coping capacity\n(rows)", fontsize=8, loc="left")
for sp in lg.spines.values():
    sp.set_visible(False)
lg.tick_params(length=0)
fig.savefig(os.path.join(FIG_DIR, "bivariate_susceptibility_capacity.png"), dpi=300); plt.show()

# ---- the same two maps as separate figures (map frame 6.1 x 6.4 in, identical to the factor maps) ----
MAP_W_IN, MAP_H_IN = 6.12, 6.40
def single_map_axes(fig_w, fig_h, left_in=0.85, bottom_in=0.70):
    return fig.add_axes([left_in / fig_w, bottom_in / fig_h, MAP_W_IN / fig_w, MAP_H_IN / fig_h])

# (1) coping-capacity index
FW, FH = 8.5, 7.8
fig = plt.figure(figsize=(FW, FH)); ax = single_map_axes(FW, FH)
ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3]); ax.set_aspect("equal")
im = ax.imshow(np.ma.masked_invalid(cci)[::2, ::2], cmap=CCI_CMAP, vmin=0, vmax=1, extent=extent, interpolation="nearest")
for g in geoms:
    for poly in (g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]):
        ring = np.asarray(poly[0]); ax.plot(ring[:, 0], ring[:, 1], color="k", lw=0.8)
ax.set_title("Coping-capacity index", loc="left"); ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
cax = fig.add_axes([(0.85 + MAP_W_IN + 0.25) / FW, 0.30, 0.025, 0.40])
cb = fig.colorbar(im, cax=cax); cb.set_label("Coping-capacity index (0 = low, 1 = high)"); cb.outline.set_visible(False)
fig.savefig(os.path.join(FIG_DIR, "map_coping_capacity_index.png"), dpi=300); plt.show()

# (2) susceptibility x coping capacity (bivariate)
FW, FH = 9.6, 7.8
fig = plt.figure(figsize=(FW, FH)); ax = single_map_axes(FW, FH)
ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3]); ax.set_aspect("equal")
ax.imshow(np.ma.masked_equal(biv, 0)[::2, ::2], cmap=biv_cmap, vmin=-0.5, vmax=9.5, extent=extent, interpolation="nearest")
for g in geoms:
    for poly in (g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]):
        ring = np.asarray(poly[0]); ax.plot(ring[:, 0], ring[:, 1], color="k", lw=0.8)
ax.set_title(f"Flood susceptibility ({SUSC_MODEL}) × coping capacity", loc="left")
ax.set_xlabel("Longitude (°E)"); ax.set_ylabel("Latitude (°N)")
lg = fig.add_axes([(0.85 + MAP_W_IN + 0.55) / FW, 0.38, 1.6 / FW, 1.6 / FH])
for cv in range(3):
    for sv in range(3):
        lg.add_patch(plt.Rectangle((sv, cv), 0.96, 0.96, color=BIV_COLOURS[cv][sv]))
lg.set_xlim(0, 3); lg.set_ylim(0, 3); lg.set_aspect("equal")
lg.set_xticks([0.5, 1.5, 2.5], ["Low", "Mod.", "High"], fontsize=8)
lg.set_yticks([0.5, 1.5, 2.5], ["High", "Med.", "Low"], fontsize=8)
lg.set_xlabel("Susceptibility →", fontsize=8); lg.set_title("Coping capacity\n(rows)", fontsize=8, loc="left")
for sp in lg.spines.values():
    sp.set_visible(False)
lg.tick_params(length=0)
fig.savefig(os.path.join(FIG_DIR, "map_susceptibility_x_coping_capacity.png"), dpi=300); plt.show()

with open(os.path.join(OUT_DIR, "run_log.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(LOG))
log(f"Total run time including extensions: {(time.time()-t0)/60:.1f} min")

### 16b. Maps of the individual coping-capacity factors
Each indicator is mapped in its original values, clipped to its 2nd–98th percentile inside the FCT for display. All maps share the same frame and scale. Each factor has its own colour ramp (purple, green, yellow–brown and blue), oriented so that **darker always means better coping capacity**: for emergency and health access, where a lower value is better, the ramp is reversed. Population is added for context, as exposure rather than capacity. The panels are saved together in one figure and also as separate files.

In [ ]:
FACTOR_LABELS = {                     # display title and colour-bar label (units as supplied in the data)
    "emergency_access": ("Emergency access", "Emergency access value (lower = closer)"),
    "health_access":    ("Health access", "Health access value (lower = better access)"),
    "night_light":      ("Night-time light", "Night-time light (log radiance)"),
    "road_density":     ("Road density", "Road density"),
}
FACTOR_CMAPS = {                      # one single-hue ramp per factor; "_r" is added automatically where lower = better
    "emergency_access": "Purples",
    "health_access":    "Greens",
    "night_light":      "YlOrBr",
    "road_density":     "Blues",
}
SHOW_POPULATION = True

def _outline(ax):
    for g in geoms:
        for poly in (g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]):
            ring = np.asarray(poly[0]); ax.plot(ring[:, 0], ring[:, 1], color="k", lw=0.8)

panels = []
for name, (fn, direction) in COPING_INDICATORS.items():
    arr = align_to_reference(os.path.join(FDIR, fn), ref); arr[~inside] = np.nan
    title, cbl = FACTOR_LABELS.get(name, (name, name))
    base = FACTOR_CMAPS.get(name, "Reds")
    panels.append((title, arr, base if direction > 0 else base + "_r", cbl))
if SHOW_POPULATION:
    pop_ref = align_to_reference(os.path.join(FDIR, POP_FILE), ref); pop_ref[~inside] = np.nan
    panels.append(("Population (exposure)", pop_ref, "Greys", "Population per pixel (as supplied)"))

def draw_factor(ax, cax, title, arr, cmap, cbl, letter=None, ylabel=True):
    v = arr[np.isfinite(arr)]
    lo, hi = np.percentile(v, [2, 98])
    ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3]); ax.set_aspect("equal")
    im = ax.imshow(np.ma.masked_invalid(arr)[::2, ::2], cmap=cmap, vmin=lo, vmax=hi, extent=extent, interpolation="nearest")
    _outline(ax)
    ax.set_title(f"({letter}) {title}" if letter else title, loc="left")
    ax.set_xlabel("Longitude (°E)")
    if ylabel:
        ax.set_ylabel("Latitude (°N)")
    cb = fig.colorbar(im, cax=cax); cb.set_label(cbl, fontsize=9); cb.outline.set_visible(False)
    cb.ax.tick_params(labelsize=8)

# combined figure: equal-sized panels in a grid of 3 columns
ncol = 3; nrow = int(np.ceil(len(panels) / ncol))
PITCH_X, W = 0.33, 0.215                  # column pitch and map width (fractions of figure width)
ROW_H = 1.0 / nrow; H = ROW_H * 0.80
fig = plt.figure(figsize=(20, 6.6 * nrow))
for k, (title, arr, cmap, cbl) in enumerate(panels):
    r, c = divmod(k, ncol)
    x0 = 0.045 + c * PITCH_X; y0 = 1 - (r + 1) * ROW_H + ROW_H * 0.11
    ax = fig.add_axes([x0, y0, W, H]); cax = fig.add_axes([x0 + W + 0.006, y0 + H * 0.2, 0.007, H * 0.6])
    draw_factor(ax, cax, title, arr, cmap, cbl, letter="abcdefgh"[k], ylabel=(c == 0))
fig.savefig(os.path.join(FIG_DIR, "coping_capacity_factors.png"), dpi=300); plt.show()

# separate files, one per factor, same frame size
for k, (title, arr, cmap, cbl) in enumerate(panels):
    fig = plt.figure(figsize=(8.5, 7.8))
    ax = fig.add_axes([0.10, 0.09, 0.72, 0.82]); cax = fig.add_axes([0.85, 0.30, 0.025, 0.40])
    draw_factor(ax, cax, title, arr, cmap, cbl)
    fname = "coping_factor_" + title.lower().split(" (")[0].replace(" ", "_").replace("-", "_") + ".png"
    fig.savefig(os.path.join(FIG_DIR, fname), dpi=300); plt.close(fig)
print("Saved:", "coping_capacity_factors.png and", len(panels), "individual factor maps in", FIG_DIR)

# Part B — Ward-level risk, population exposure and the manuscript figures
Part A (sections 1–16) trains the models and writes the susceptibility, agreement, coping-capacity and bivariate rasters to `OUT_DIR/maps`.
Part B starts from those GeoTIFFs, so it can be re-run on its own once Part A has finished (or with the `maps` folder of an earlier run).

It reproduces the remaining analyses and figures of the manuscript:

| Section | Output | Manuscript |
|---|---|---|
| 18 | Confusion matrices of the three models | Figure 7 |
| 19 | Conditioning factors as framed panels with natural-break classes | Figure 4 |
| 20 | Susceptibility maps on the hillshade (relief shading) | Figure 11 |
| 21 | Ward-level composite flood risk, R = S̄ × (1 − C̄), and the fifteen highest-risk wards | Figure 16 |
| 22 | Vector polygons of susceptibility and coping capacity, union with the wards, ward bivariate map | Figure 15 |
| 23 | Hexagon map of exposed population in the fifteen highest-risk wards, with five insets | Figure 17 |
| 24 | Bivariate map of composite risk × ward population | supplementary |
| 25 | Population in the lowest and highest coping-capacity terciles | Figure 18 |

**Extra inputs (not in `FCT_FLOOD.rar`)**
- `Nigeria_-_Ward_Boundaries.zip` — INEC ward polygons for Nigeria (shapefile, EPSG:4326); the 62 FCT wards are selected with `statecode == "FC"`.
- `hillshade.zip` — the study hillshade (`hillshade/hillshade1.tif`), used as the relief basemap in sections 20 and 23–25.

**Implementation notes**
- Shapefiles are read and written with small NumPy functions (section 17) so that the section needs no GDAL, GeoPandas or Shapely; the outputs open normally in ArcGIS/QGIS (`.shp/.shx/.dbf/.prj`).
- Each figure is wrapped in its own function so that its working variables do not overwrite those of Part A.
- Areas use the same per-pixel area convention as Part A (equirectangular, cos(latitude)); ward polygon areas use a local equirectangular projection of each ring.

## 17. Part B configuration, extra inputs and helper functions
Edit the two paths below. Zipped inputs are extracted automatically; a folder or a `.shp`/`.tif` path also works.

In [ ]:
import os, glob, math, json, zipfile, struct, importlib, subprocess, sys
for pkg, pipname in [("skimage", "scikit-image"), ("scipy", "scipy")]:   # normally pre-installed on Colab
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pipname], check=True)
import numpy as np, pandas as pd, rasterio
import matplotlib.pyplot as plt

WARD_INPUT = "/content/drive/MyDrive/Nigeria_-_Ward_Boundaries.zip"   # <-- zip, folder or .shp
HILLSHADE_INPUT = "/content/drive/MyDrive/hillshade.zip"              # <-- zip, folder or .tif
WARD_STATE_CODE = "FC"                                                # FCT wards in the INEC layer
BASEMAP_TIF = None    # optional RGB GeoTIFF (EPSG:4326) to replace the hillshade under the Figure 17 hexagons

MAP_DIR = os.path.join(OUT_DIR, "maps"); FIG_DIR = os.path.join(OUT_DIR, "figures")
WARD_DIR = os.path.join(OUT_DIR, "wards"); VEC_DIR = os.path.join(OUT_DIR, "vectors")
FDIR = os.path.join(DATA_DIR, "flood_vulnerability_mapping-conditioning_factors")
for d_ in (FIG_DIR, WARD_DIR, VEC_DIR):
    os.makedirs(d_, exist_ok=True)
if "log" not in globals():
    def log(msg=""): print(msg)

def find_input(path, pattern, dest):
    """Return the first file matching `pattern` in a zip (extracted to `dest`), a folder, or the path itself."""
    if path and path.lower().endswith(".zip") and os.path.exists(path):
        with zipfile.ZipFile(path) as z:
            z.extractall(dest)
        path = dest
    if path and os.path.isdir(path):
        hits = sorted(glob.glob(os.path.join(path, "**", pattern), recursive=True))
        return hits[0] if hits else None
    return path if path and os.path.exists(path) else None

WARD_SHP = find_input(WARD_INPUT, "*.shp", "/content/wards_input")
HILLSHADE_TIF = find_input(HILLSHADE_INPUT, "*.tif", "/content/hillshade_input")
assert WARD_SHP, "Ward shapefile not found: check WARD_INPUT"
WARD_DBF = WARD_SHP[:-4] + ".dbf"
print("Ward shapefile:", WARD_SHP); print("Hillshade:", HILLSHADE_TIF)

def read_tif(path):
    """Band 1 of a GeoTIFF with its origin and pixel size: (array, (x0, dx, y0, dy), epsg, nodata)."""
    with rasterio.open(path) as src:
        a, t = src.read(1), src.transform
        try:
            epsg = src.crs.to_epsg()
        except Exception:
            epsg = None
        return a, (t.c, t.a, t.f, t.e), epsg, src.nodata

def imread(path):
    return read_tif(path)[0]

# reference grid of the susceptibility maps (identical for all Part A outputs)
_, (OX, SX, OY, _dy), _, _ = read_tif(os.path.join(MAP_DIR, "FCT_flood_susceptibility_RF.tif"))
print(f"Reference grid origin {OX:.6f}, {OY:.6f}; pixel {SX:.10f} deg")

# ---- minimal shapefile reader (polygons) ------------------------------------------------
def read_dbf(path):
    with open(path, "rb") as f:
        h = f.read(32); n = struct.unpack("<I", h[4:8])[0]; hl, rl = struct.unpack("<HH", h[8:12])
        fields = []
        while True:
            d = f.read(32)
            if d[0] == 0x0D:
                break
            fields.append((d[:11].split(b"\0")[0].decode(), chr(d[11]), d[16]))
        f.seek(hl); recs = []
        for _ in range(n):
            r = f.read(rl); pos = 1; rec = {}
            for name, t, ln in fields:
                rec[name] = r[pos:pos + ln].decode("utf8", "replace").strip(); pos += ln
            recs.append(rec)
    return fields, recs

def read_shp(path):
    """List of (bbox, [ring arrays]) per record; None for null shapes."""
    d = open(path, "rb").read(); pos = 100; shapes = []
    while pos < len(d):
        num, clen = struct.unpack(">ii", d[pos:pos + 8]); pos += 8; c = d[pos:pos + clen * 2]; pos += clen * 2
        if struct.unpack("<i", c[:4])[0] == 0:
            shapes.append(None); continue
        bbox = struct.unpack("<4d", c[4:36]); npart, npt = struct.unpack("<ii", c[36:44])
        parts = list(struct.unpack(f"<{npart}i", c[44:44 + 4 * npart]))
        pts = np.frombuffer(c[44 + 4 * npart:44 + 4 * npart + 16 * npt], dtype="<f8").reshape(-1, 2)
        shapes.append((bbox, [pts[parts[i]:(parts[i + 1] if i + 1 < npart else npt)] for i in range(npart)]))
    return shapes

In [ ]:
# ---- raster-to-polygon conversion (exact pixel edges) and shapefile writer --------------------
import struct, numpy as np
from collections import defaultdict
from skimage.measure import label as sklabel

def polygonize(L, ox, oy, px):
    """L: int array, 0 = NoData. Returns list of (value, [outer_ring, hole1, ...]) in map coords.
    Rings are lists of (x, y); outer rings clockwise, holes counter-clockwise (shapefile convention)."""
    H, W = L.shape
    comp = sklabel(L, background=0, connectivity=1)          # 4-connected regions of equal value
    P = np.zeros((H + 2, W + 2), comp.dtype); P[1:-1, 1:-1] = comp
    edges = []                                                # (comp, r0, c0, r1, c1) with comp on the LEFT
    # vertical boundaries between (r, c-1) and (r, c) in padded grid -> vertex coords (row, col) of corners
    a = P[1:-1, :-1]; b = P[1:-1, 1:]
    rr, cc = np.nonzero(a != b)
    la, lb = a[rr, cc], b[rr, cc]
    # edge on line x = cc (corner col), from row rr to rr+1. Region on left when moving down (y increasing in row units)
    # Using image coords (row down, col right): moving down with region to the left means region is at col+ (east). Choose:
    m = lb > 0; edges.append(np.c_[lb[m], rr[m], cc[m], rr[m] + 1, cc[m]])        # east cell on left: go down
    m = la > 0; edges.append(np.c_[la[m], rr[m] + 1, cc[m], rr[m], cc[m]])        # west cell on left: go up
    a = P[:-1, 1:-1]; b = P[1:, 1:-1]
    rr, cc = np.nonzero(a != b)
    la, lb = a[rr, cc], b[rr, cc]
    m = la > 0; edges.append(np.c_[la[m], rr[m], cc[m], rr[m], cc[m] + 1])        # cell above the line: go east
    m = lb > 0; edges.append(np.c_[lb[m], rr[m], cc[m] + 1, rr[m], cc[m]])        # cell below the line: go west
    E = np.vstack(edges)
    nxt = defaultdict(list)
    for k, (cp, r0, c0, r1, c1) in enumerate(E):
        nxt[(cp, r0, c0)].append(k)
    used = np.zeros(len(E), bool)
    rings = defaultdict(list)
    for k0 in range(len(E)):
        if used[k0]:
            continue
        cp = E[k0, 0]; ring = [(E[k0, 1], E[k0, 2])]; k = k0
        while True:
            used[k] = True
            _, r0, c0, r1, c1 = E[k]
            d_in = (r1 - r0, c1 - c0)
            if (r1, c1) == ring[0]:
                break
            cand = [j for j in nxt[(cp, r1, c1)] if not used[j]]
            if not cand:
                break
            if len(cand) > 1:   # saddle: turn so that diagonal pixels stay separate (4-connectivity)
                def turn(j):
                    d = (E[j, 3] - E[j, 1], E[j, 4] - E[j, 2])
                    cross = d_in[0] * d[1] - d_in[1] * d[0]
                    return cross
                cand.sort(key=turn, reverse=True)   # turn towards the region
            k = cand[0]
            ring.append((r1, c1))
        rings[cp].append(ring)
    out = []
    vals = {}
    for cp, rs in rings.items():
        # pick value of this component
        polys = []
        for ring in rs:
            pts = np.array(ring, float)
            # drop collinear vertices
            if len(pts) > 3:
                d1 = np.diff(np.vstack([pts[-1:], pts]), axis=0)
                d2 = np.diff(np.vstack([pts, pts[:1]]), axis=0)
                keep = (d1[:, 0] * d2[:, 1] - d1[:, 1] * d2[:, 0]) != 0
                pts = pts[keep]
            xy = np.c_[ox + pts[:, 1] * px, oy - pts[:, 0] * px]
            xy = np.vstack([xy, xy[:1]])
            area = 0.5 * np.sum(xy[:-1, 0] * xy[1:, 1] - xy[1:, 0] * xy[:-1, 1])
            polys.append((abs(area), area, xy))
        polys.sort(key=lambda t: -t[0])
        rings_xy = []
        for n, (_, area, xy) in enumerate(polys):
            if n == 0:   # outer: clockwise (negative signed area)
                rings_xy.append(xy if area < 0 else xy[::-1])
            else:        # hole: counter-clockwise
                rings_xy.append(xy if area > 0 else xy[::-1])
        out.append((cp, rings_xy))
    value_of = {}
    flat_c, flat_v = comp.ravel(), L.ravel()
    idx = np.unique(flat_c, return_index=True)
    for c_, i_ in zip(*idx):
        value_of[c_] = flat_v[i_]
    return [(int(value_of[cp]), r) for cp, r in out]

# ---------------- shapefile writer (polygon, type 5) ----------------
def write_shapefile(base, features, fields, prj_wkt):
    """features: list of (rings, attrs dict); fields: list of (name, type 'C'/'N', size, decimals)."""
    recs = []
    for rings, _ in features:
        pts = np.vstack(rings); parts = np.cumsum([0] + [len(r) for r in rings[:-1]])
        bbox = (pts[:, 0].min(), pts[:, 1].min(), pts[:, 0].max(), pts[:, 1].max())
        body = struct.pack("<i4d2i", 5, *bbox, len(rings), len(pts)) + struct.pack(f"<{len(parts)}i", *parts) + pts.astype("<f8").tobytes()
        recs.append((body, bbox))
    allb = np.array([b for _, b in recs])
    gb = (allb[:, 0].min(), allb[:, 1].min(), allb[:, 2].max(), allb[:, 3].max())
    def header(length_words):
        return struct.pack(">7i", 9994, 0, 0, 0, 0, 0, length_words) + struct.pack("<2i4d4d", 1000, 5, *gb, 0, 0, 0, 0)
    shp = bytearray(); shx = bytearray(); off = 50
    for n, (body, _) in enumerate(recs, start=1):
        cl = len(body) // 2
        shp += struct.pack(">2i", n, cl) + body
        shx += struct.pack(">2i", off, cl); off += 4 + cl
    open(base + ".shp", "wb").write(header(50 + len(shp) // 2) + shp)
    open(base + ".shx", "wb").write(header(50 + len(shx) // 2) + shx)
    # dbf
    nrec = len(features); hl = 32 + 32 * len(fields) + 1; rl = 1 + sum(f[2] for f in fields)
    d = bytearray(struct.pack("<BBBBIHH20x", 3, 126, 10, 9, nrec, hl, rl))
    for name, t, size, dec in fields:
        d += struct.pack("<11sc4xBB14x", name.encode()[:10].ljust(11, b"\0"), t.encode(), size, dec)
    d += b"\r"
    for _, attrs in features:
        d += b" "
        for name, t, size, dec in fields:
            v = attrs.get(name, "")
            if t == "N":
                s = (f"{v:.{dec}f}" if dec else f"{int(round(v))}").rjust(size)[:size]
            else:
                s = str(v).encode("latin-1", "replace").decode("latin-1").ljust(size)[:size]
            d += s.encode("latin-1")
    d += b"\x1a"
    open(base + ".dbf", "wb").write(bytes(d))
    open(base + ".prj", "w").write(prj_wkt)
    open(base + ".cpg", "w").write("UTF-8")

## 18. Confusion matrices (Figure 7)
Counts of the testing set at a threshold of 0.5, read from `model_performance.csv` (section 9).

In [ ]:
def figure_confusion_matrices():
    met = pd.read_csv(os.path.join(OUT_DIR, "model_performance.csv"))
    t = met[met.Dataset == "Testing"].set_index("Model")
    plt.rcParams["font.family"] = "DejaVu Sans"
    BAR = {"RF": "#2f7ed8", "XGBoost": "#f0703a", "CART": "#1fae7a"}
    fig = plt.figure(figsize=(10, 3.6), dpi=360)
    for k, m in enumerate(["RF", "XGBoost", "CART"]):
        r = t.loc[m]; cm = np.array([[r.TN, r.FP], [r.FN, r.TP]], dtype=float)
        share = cm / cm.sum(axis=1, keepdims=True)
        x0 = 0.09 + k * 0.329
        ax = fig.add_axes([x0, 0.19, 0.24, 0.66])
        ax.imshow(share, cmap="Blues", vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                light = share[i, j] > 0.5
                ax.text(j - 0.40, i - 0.33, [["TN", "FP"], ["FN", "TP"]][i][j], fontsize=7, color="white" if light else "#444")
                ax.text(j, i - 0.05, f"{int(cm[i, j])}", ha="center", va="center", fontsize=15, fontweight="bold", color="white" if light else "#1a1a1a")
                ax.text(j, i + 0.25, f"{100 * share[i, j]:.1f}%", ha="center", va="center", fontsize=8.5, color="white" if light else "#333")
        ax.set_xticks([0, 1], ["Non-flood", "Flood"], fontsize=9); ax.set_yticks([0, 1], ["Non-flood", "Flood"], fontsize=9)
        ax.set_xlabel("Predicted class", fontsize=9.5); ax.set_ylabel("Observed class", fontsize=9.5)
        for xx in (0.5,): ax.axvline(xx, color="white", lw=3); ax.axhline(xx, color="white", lw=3)
        for sp in ax.spines.values(): sp.set_visible(False)
        fig.add_artist(plt.Line2D([x0, x0 + 0.24], [0.88, 0.88], color=BAR[m], lw=4, transform=fig.transFigure))
        fig.text(x0, 0.93, f"({'abc'[k]}) {m}", fontsize=11, fontweight="bold", va="bottom")
        fig.text(x0 + 0.24, 0.93, f"Accuracy {r.Accuracy:.3f}", fontsize=9, ha="right", va="bottom", color="#333")
    n1, n0 = int(t.loc["RF", "TP"] + t.loc["RF", "FN"]), int(t.loc["RF", "TN"] + t.loc["RF", "FP"])
    fig.text(0.5, 0.02, f"Testing dataset (n = {n1 + n0}: {n1} flood, {n0} non-flood); probability threshold 0.5. "
             "Percentages are row-normalised (share of each observed class).", ha="center", fontsize=8, color="#444")
    fig.savefig(os.path.join(FIG_DIR, "confusion_matrices_three_models.png"), facecolor="white")
    plt.show(); plt.close(fig)
    log(t[["TP", "FP", "TN", "FN"]].to_string())

figure_confusion_matrices()

## 19. Conditioning factors as framed panels (Figure 4)
The ten factors are drawn on the same grid as the models (distance to water recomputed from the land-cover water class and rainfall smoothed as in section 5), each classified by Fisher–Jenks natural breaks on a sample of pixels. Uses `smooth_coarse_layer` from section 4.

In [ ]:
import os
def figure_conditioning_factors():
    import sys, math, json, numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap, to_rgba
    from matplotlib.patches import Rectangle, Polygon, FancyArrow
    from scipy import ndimage
    from collections import namedtuple

    ST = 3
    plt.rcParams["font.family"] = "DejaVu Serif"

    # ---------------------------------------------------------------- data
    def rd(name, stride=ST):
        a, (ox, sx, oy, sy), e, nd = read_tif(FDIR + "/" + name)
        a = a.astype(float)
        if nd is not None:
            a[np.isclose(a, nd)] = np.nan
        return a[::stride, ::stride], (ox, sx, oy, sy)

    inside_full = imread(MAP_DIR + "/" + "FCT_flood_susceptibility_RF.tif") >= 0
    inside = inside_full[::ST, ::ST]
    elev, (OX, SX, OY, SY) = rd("fct_elevation.tif")
    H, W = inside.shape
    EXT = [OX, OX + SX * ST * W, OY - SX * ST * H, OY]          # lon/lat extent (stride-adjusted)

    def clip(a):
        a = a[:H, :W].copy(); a[~inside] = np.nan; return a

    layers = {}
    layers["elev"] = clip(elev)
    layers["slope"] = clip(rd("fct_slope.tif")[0])
    layers["twi"] = clip(rd("fct_TWI.tif")[0])
    layers["spi"] = clip(rd("fct_SPI.tif")[0])
    layers["dd"] = clip(rd("fct_drainage_density.tif")[0])
    layers["ndvi"] = clip(rd("fct_NDVI.tif")[0])
    lc_full, _ = rd("fct_landcover.tif", 1)
    lc_full = lc_full[:inside_full.shape[0], :inside_full.shape[1]]
    layers["lulc"] = clip(lc_full[::ST, ::ST])
    layers["built"] = clip(rd("fct_built_up.tif")[0])
    # distance to water: recomputed from WorldCover water (class 80), as in the models
    lat0 = math.radians(9.0)
    dy = SX * 110574.0; dx = SX * 111320.0 * math.cos(lat0)
    dist_full = ndimage.distance_transform_edt(~(lc_full == 80), sampling=(dy, dx))
    layers["dist"] = clip(dist_full[::ST, ::ST])
    del dist_full
    # rainfall: bilinear smoothing of the 0.05-degree CHIRPS cells, as in the models
    rain_full, _ = rd("fct_rainfall.tif", 1)
    rain_full = rain_full[:inside_full.shape[0], :inside_full.shape[1]]
    T_ = namedtuple("T", "a b c d e f")
    ref = {"transform": T_(SX, 0, OX, 0, -SX, OY), "height": rain_full.shape[0], "width": rain_full.shape[1]}
    rain_s = smooth_coarse_layer(rain_full, ref, RAINFALL_CELL_DEG)
    layers["rain"] = clip(rain_s[::ST, ::ST])
    del rain_full, rain_s, lc_full

    # ---------------------------------------------------------------- Jenks (Fisher) natural breaks on a sample
    def jenks(values, k=5, n=4000, seed=0):
        v = values[np.isfinite(values)]
        rng = np.random.default_rng(seed)
        v = np.sort(rng.choice(v, min(n, v.size), replace=False))
        N = v.size
        cs, cs2 = np.concatenate([[0], np.cumsum(v)]), np.concatenate([[0], np.cumsum(v * v)])
        def ssd(a, b):  # inclusive a..b (0-based)
            s = cs[b + 1] - cs[a]; s2 = cs2[b + 1] - cs2[a]; m = b - a + 1
            return s2 - s * s / m
        cost = np.full((k, N), np.inf); back = np.zeros((k, N), int)
        for b in range(N):
            cost[0, b] = ssd(0, b)
        for c in range(1, k):
            for b in range(c, N):
                A = np.arange(c, b + 1)
                s_ = cs[b + 1] - cs[A]; s2 = cs2[b + 1] - cs2[A]; m_ = b - A + 1
                x = cost[c - 1, A - 1] + s2 - s_ * s_ / m_
                k_ = int(np.argmin(x)); cost[c, b], back[c, b] = x[k_], A[k_]
        cuts, b = [], N - 1
        for c in range(k - 1, 0, -1):
            a = back[c, b]; cuts.append(v[a - 1]); b = a - 1
        return sorted(cuts)

    def nice(x, step):
        return round(x / step) * step

    # ---------------------------------------------------------------- class definitions
    def mm(a): v = a[np.isfinite(a)]; return v.min(), v.max()
    breaks = {}
    breaks["elev"] = [208.5, 327.5, 463.5, 609.5]
    breaks["slope"] = [3.915, 8.435, 14.85, 22.95]
    breaks["twi"] = [-1.865, -0.1875, 2.215, 5.945]
    breaks["spi"] = [-6.905, 6.985, 9.165, 12.75]
    breaks["dd"] = [0.3595, 0.9675, 1.585, 2.235]
    breaks["dist"] = [nice(x, 50) for x in jenks(layers["dist"])]
    breaks["rain"] = [nice(x, 5) for x in jenks(layers["rain"])]
    breaks["ndvi"] = [nice(x, 0.01) for x in jenks(layers["ndvi"])]

    def fmt_labels(key, b, lo, hi, dec, unit=""):
        edges = [lo] + list(b) + [hi]
        out = []
        for i in range(5):
            a = edges[i] if i == 0 else edges[i]
            out.append((a, edges[i + 1]))
        return out

    def labels_from(b, lo, hi, f):
        e = [lo] + list(b) + [hi]
        return [f"{f(e[i])} – {f(e[i + 1])}" for i in range(5)]

    f0 = lambda x: f"{x:,.0f}"
    LAB = {
        "elev": ["48 – 208", "209 – 327", "328 – 463", "464 – 609", "610 – 937"],
        "slope": ["0 – 3.91", "3.92 – 8.43", "8.44 – 14.8", "14.9 – 22.9", "23 – 76.8"],
        "twi": ["−6.22 – −1.87", "−1.86 – −0.19", "−0.19 – 2.21", "2.22 – 5.94", "5.95 – 16.4"],
        "spi": ["≤ −6.91", "−6.90 – 6.98", "6.99 – 9.16", "9.17 – 12.7", "12.8 – 22.4"],
        "dd": ["0 – 0.36", "0.36 – 0.97", "0.97 – 1.58", "1.59 – 2.23", "2.24 – 3.98"],
    }
    lo, hi = mm(layers["dist"]); LAB["dist"] = labels_from([b / 1000 for b in breaks["dist"]], 0, hi / 1000, lambda x: f"{x:.1f}")
    lo, hi = mm(layers["rain"]); LAB["rain"] = labels_from(breaks["rain"], lo, hi, f0)
    lo, hi = mm(layers["ndvi"]); LAB["ndvi"] = labels_from(breaks["ndvi"], lo, hi, lambda x: f"{x:.2f}".replace("-", "−"))

    CM = {
        "elev": ["#1a7837", "#7fbf3f", "#e3e88a", "#c39a55", "#6b4423"],
        "slope": ["#ffffd4", "#fee391", "#fec44f", "#d9822b", "#8c3d0b"],
        "twi": ["#eff3ff", "#bdd7e7", "#6baed6", "#2171b5", "#08306b"],
        "spi": ["#f2f0f7", "#cbc9e2", "#9e9ac8", "#756bb1", "#4a1486"],
        "dd": ["#ffffcc", "#a1dab4", "#41b6c4", "#2c7fb8", "#253494"],
        "dist": ["#d7191c", "#fdae61", "#ffffbf", "#a6d96a", "#1a9641"],
        "rain": ["#f1eef6", "#bdc9e1", "#74a9cf", "#2b8cbe", "#045a8d"],
        "ndvi": ["#a50026", "#f46d43", "#fee08b", "#66bd63", "#006837"],
    }
    LULC = [(10, "Tree cover", "#006400"), (20, "Shrubland", "#ffbb22"), (30, "Grassland", "#ffff4c"),
            (40, "Cropland", "#f096ff"), (50, "Built-up", "#fa0000"), (60, "Bare / sparse", "#b4b4b4"),
            (80, "Water", "#0064c8"), (90, "Wetland", "#0096a0")]

    PANELS = [
        ("elev", "Digital Elevation Model (DEM)", "Elevation (m)"),
        ("slope", "Slope", "Slope (°)"),
        ("twi", "Topographic Wetness Index (TWI)", "TWI"),
        ("spi", "Stream Power Index (SPI)", "SPI"),
        ("dd", "Drainage Density", "Density (km/km²)"),
        ("dist", "Distance to Water Bodies", "Distance (km)"),
        ("rain", "Mean Annual Rainfall", "Rainfall (mm)"),
        ("lulc", "Land Use / Land Cover (LULC)", "LULC class"),
        ("built", "Built-up Area", "Built-up"),
        ("ndvi", "Normalized Difference Vegetation Index (NDVI)", "NDVI"),
    ]

    # ---------------------------------------------------------------- drawing helpers
    def classify(a, b):
        out = np.full(a.shape, np.nan); m = np.isfinite(a); out[m] = np.digitize(a[m], b); return out

    def north_arrow(ax, x, y, s):
        ax.add_patch(Polygon([[x, y], [x - 0.35 * s, y - s], [x, y - 0.75 * s]], closed=True, fc="black", ec="black", lw=0.6, transform=ax.transAxes, zorder=9))
        ax.add_patch(Polygon([[x, y], [x + 0.35 * s, y - s], [x, y - 0.75 * s]], closed=True, fc="white", ec="black", lw=0.6, transform=ax.transAxes, zorder=9))
        ax.text(x, y + 0.02, "N", ha="center", va="bottom", fontsize=9, fontweight="bold", transform=ax.transAxes, zorder=9)

    def scale_bar(ax, x0, y0):
        km_per_deg = 111.32 * math.cos(math.radians(9.0))
        segs = [0, 5, 10, 20]
        for k in range(len(segs) - 1):
            a = x0 + segs[k] / km_per_deg; b = x0 + segs[k + 1] / km_per_deg
            ax.add_patch(Rectangle((a, y0), b - a, 0.012, fc="black" if k % 2 == 0 else "white", ec="black", lw=0.5, zorder=9))
        for v in segs:
            ax.text(x0 + v / km_per_deg, y0 - 0.008, f"{v}", ha="center", va="top", fontsize=5.5, zorder=9)
        ax.text(x0 + segs[-1] / km_per_deg / 2, y0 + 0.02, "Kilometres", ha="center", va="bottom", fontsize=5.5, zorder=9)

    def legend(fig, rect, title, items):
        lax = fig.add_axes(rect); lax.axis("off"); lax.set_xlim(0, 1); lax.set_ylim(0, 1)
        lax.text(0, 1, title, fontsize=6.0, fontweight="bold", va="top")
        n = len(items); h = min(0.11, 0.84 / n)
        for i, (c, t) in enumerate(items):
            y = 0.86 - (i + 1) * h
            lax.add_patch(Rectangle((0, y), 0.2, h * 0.75, fc=c, ec="#555", lw=0.4))
            lax.text(0.25, y + h * 0.37, t, fontsize=5.5, va="center")

    fig = plt.figure(figsize=(7.2, 9.6), dpi=400)
    cols, rows = 3, 4
    pw, ph = 0.318, 0.226            # panel size (figure fraction)
    gx, gy = 0.012, 0.022
    x_start = (1 - (cols * pw + (cols - 1) * gx)) / 2
    for n, (key, title, ltitle) in enumerate(PANELS):
        r, c = divmod(n, cols)
        if r == 3:                    # centre the last panel
            c = 1
        x = x_start + c * (pw + gx); y = 1 - (r + 1) * (ph + gy) + 0.004
        frame = fig.add_axes([x, y + 0.016, pw, ph - 0.016]); frame.set_xticks([]); frame.set_yticks([])
        for sp in frame.spines.values(): sp.set_linewidth(0.9)
        frame.text(0.5, 0.975, title, ha="center", va="top", fontsize=7.4 if len(title) < 34 else 5.9, fontweight="bold", transform=frame.transAxes)
        fig.text(x + pw / 2, y + 0.004, f"({'abcdefghij'[n]})", ha="center", va="center", fontsize=9)
        mx = fig.add_axes([x + 0.004, y + 0.022, pw * 0.62, ph - 0.05])
        a = layers[key]
        if key == "lulc":
            present = [t for t in LULC if np.any(a == t[0])]
            rgba = np.zeros(a.shape + (4,))
            for code, _, col in present: rgba[a == code] = to_rgba(col)
            mx.imshow(rgba, extent=EXT, interpolation="nearest")
            items = [(col, name) for _, name, col in present]
        elif key == "built":
            rgba = np.zeros(a.shape + (4,)); rgba[np.isfinite(a)] = to_rgba("#e6e6e6"); rgba[a > 0] = to_rgba("#d7301f")
            mx.imshow(rgba, extent=EXT, interpolation="nearest")
            items = [("#d7301f", "Built-up"), ("#e6e6e6", "Non built-up")]
        else:
            cl = classify(a, breaks[key])
            mx.imshow(cl, extent=EXT, cmap=ListedColormap(CM[key]), vmin=-0.5, vmax=4.5, interpolation="nearest")
            items = list(zip(CM[key], LAB[key]))
        mx.set_aspect(1 / math.cos(math.radians(9.0)))
        mx.axis("off")
        mx.apply_aspect(); pos = mx.get_position()
        # north arrow (figure coords, top of legend column)
        nx, ny, s_ = x + pw * 0.935, y + 0.056, 0.022
        fig.add_artist(Polygon([[nx, ny], [nx - 0.006, ny - s_], [nx, ny - 0.75 * s_]], closed=True, fc="black", ec="black", lw=0.5, transform=fig.transFigure))
        fig.add_artist(Polygon([[nx, ny], [nx + 0.006, ny - s_], [nx, ny - 0.75 * s_]], closed=True, fc="white", ec="black", lw=0.5, transform=fig.transFigure))
        fig.text(nx, ny + 0.0015, "N", ha="center", va="bottom", fontsize=8, fontweight="bold")
        # scale bar (figure coords, true map scale)
        km_per_deg = 111.32 * math.cos(math.radians(9.0)); xl = mx.get_xlim()
        f_per_km = pos.width / ((xl[1] - xl[0]) * km_per_deg)
        sx0, sy0 = x + pw * 0.655, y + 0.032
        for k_, (a_, b_) in enumerate([(0, 10), (10, 20)]):
            fig.add_artist(Rectangle((sx0 + a_ * f_per_km, sy0), (b_ - a_) * f_per_km, 0.0028, fc="black" if k_ == 0 else "white", ec="black", lw=0.4, transform=fig.transFigure))
        for v in (0, 10, 20):
            fig.text(sx0 + v * f_per_km, sy0 - 0.001, str(v), ha="center", va="top", fontsize=5.2)
        fig.text(sx0 + 10 * f_per_km, sy0 + 0.004, "Kilometres", ha="center", va="bottom", fontsize=5.2)
        legend(fig, [x + pw * 0.645, y + 0.078, pw * 0.35, ph * 0.56], ltitle, items)

    fig.savefig(FIG_DIR + "/conditioning_factors_panels.png", facecolor="white")
    plt.show(); plt.close(fig)
    json.dump({"dist": breaks["dist"], "rain": breaks["rain"], "ndvi": breaks["ndvi"], "LAB": LAB,
               "ranges": {k: [float(np.nanmin(layers[k])), float(np.nanmax(layers[k]))] for k in ["dist", "rain", "ndvi", "elev", "slope"]}},
              open(os.path.join(OUT_DIR, "factor_breaks.json"), "w"), indent=1, ensure_ascii=False)
    print("done", json.dumps({k: LAB[k] for k in ["dist", "rain", "ndvi"]}, ensure_ascii=False))

figure_conditioning_factors()

## 20. Susceptibility maps on the hillshade (Figure 11)
The five-class maps of the three models are blended with the hillshade (soft-light blend), which gives an impression of relief while keeping the class colours readable. The class shares in the legends are read from `susceptibility_class_areas.csv`.

In [ ]:
import os, pandas as pd
assert HILLSHADE_TIF, 'Hillshade not found: check HILLSHADE_INPUT'
def figure_hillshade_susceptibility():
    import math, sys, numpy as np, matplotlib
    from matplotlib.colors import to_rgb
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap
    from matplotlib.patches import Rectangle, Polygon

    ST = 1
    plt.rcParams["font.family"] = "DejaVu Serif"
    COL = ["#16a34a", "#8fd45a", "#fff27a", "#ff9a3c", "#ef1c24"]
    NAMES = ["Very low (0–0.2)", "Low (0.2–0.4)", "Moderate (0.4–0.6)", "High (0.6–0.8)", "Very high (0.8–1.0)"]
    _ca = pd.read_csv(os.path.join(OUT_DIR, "susceptibility_class_areas.csv"))
    AREA = {m: [float(_ca[(_ca.Model == m) & (_ca.Class == c)]["Area (%)"].iloc[0]) for c in ["Very low", "Low", "Moderate", "High", "Very high"]]
            for m in ["RF", "XGBoost", "CART"]}

    hs, _, _, _ = read_tif(HILLSHADE_TIF)
    hs = hs.astype(float)[::ST, ::ST]
    fig = plt.figure(figsize=(7.2, 3.15), dpi=600)
    pw, ph, gx = 0.322, 0.86, 0.012
    x0 = (1 - (3 * pw + 2 * gx)) / 2
    for n, m in enumerate(["RF", "XGBoost", "CART"]):
        p = imread(MAP_DIR + "/" + f"FCT_flood_susceptibility_{m}.tif").astype(float)[::ST, ::ST]
        p[p < 0] = np.nan
        cl = np.full(p.shape, np.nan); f = np.isfinite(p); cl[f] = np.digitize(p[f], [0.2, 0.4, 0.6, 0.8])
        H, W = p.shape; ext = [OX, OX + SX * ST * W, OY - SX * ST * H, OY]
        x = x0 + n * (pw + gx); y = 0.09
        fr = fig.add_axes([x, y, pw, ph]); fr.set_xticks([]); fr.set_yticks([])
        for sp in fr.spines.values(): sp.set_linewidth(0.9)
        name = {"RF": "Random Forest (RF)", "XGBoost": "XGBoost", "CART": "CART"}[m]
        fr.text(0.5, 0.975, name, ha="center", va="top", fontsize=8.5, fontweight="bold", transform=fr.transAxes)
        fig.text(x + pw / 2, 0.035, f"({'abc'[n]})", ha="center", va="center", fontsize=9)
        mx = fig.add_axes([x + 0.004, y + 0.03, pw * 0.60, ph - 0.12])
        h = hs[:cl.shape[0], :cl.shape[1]]
        sh = np.clip((h - 1) / 179.0, 0, 1)
        sh = (sh - np.nanpercentile(sh, 2)) / (np.nanpercentile(sh, 99.5) - np.nanpercentile(sh, 2))
        sh = np.clip(sh, 0, 1)
        BRIGHT = ["#16a34a", "#8fd45a", "#fff27a", "#ff9a3c", "#ef1c24"]
        pal = np.array([to_rgb(c) for c in BRIGHT])
        base = np.ones(cl.shape + (3,)); ok = np.isfinite(cl)
        base[ok] = pal[cl[ok].astype(int)]
        # soft-light blend: shadows darken, sunlit slopes brighten, colours keep their saturation
        sl = sh[..., None]
        rgb = np.where(sl < 0.5, base - (1 - 2 * sl) * base * (1 - base) - (0.5 - sl) * 0.55 * base,
                       base + (2 * sl - 1) * (np.sqrt(base) - base))
        rgb = np.clip(rgb, 0, 1); rgb[~ok] = 1
        rgba = np.dstack([rgb, ok.astype(float)])
        mx.imshow(rgba, extent=ext, interpolation="antialiased", interpolation_stage="rgba")
        mx.set_aspect(1 / math.cos(math.radians(9.0))); mx.axis("off")
        mx.apply_aspect(); pos = mx.get_position()
        # legend with class share of the FCT
        lx, ly = x + pw * 0.62, y + ph * 0.80
        fig.text(lx, ly, "Susceptibility", fontsize=6.3, fontweight="bold", va="top")
        fig.text(lx, ly - 0.055, "class (% of FCT area)", fontsize=5.6, va="top", color="#333")
        for k in range(5):
            yy = ly - 0.155 - (4 - k) * 0.072
            fig.add_artist(Rectangle((lx, yy), 0.016, 0.045, fc=COL[k], ec="#555", lw=0.4, transform=fig.transFigure))
            fig.text(lx + 0.021, yy + 0.022, f"{NAMES[k]}\n{AREA[m][k]:.1f}%", fontsize=4.9, va="center", linespacing=1.05)
        # north arrow and scale bar
        nx, ny, s_ = x + pw * 0.93, y + 0.185, 0.07
        fig.add_artist(Polygon([[nx, ny], [nx - 0.006, ny - s_], [nx, ny - 0.75 * s_]], closed=True, fc="black", ec="black", lw=0.5, transform=fig.transFigure))
        fig.add_artist(Polygon([[nx, ny], [nx + 0.006, ny - s_], [nx, ny - 0.75 * s_]], closed=True, fc="white", ec="black", lw=0.5, transform=fig.transFigure))
        fig.text(nx, ny + 0.005, "N", ha="center", va="bottom", fontsize=8, fontweight="bold")
        km_per_deg = 111.32 * math.cos(math.radians(9.0)); xl = mx.get_xlim()
        fpk = pos.width / ((xl[1] - xl[0]) * km_per_deg)
        sx0, sy0 = x + pw * 0.625, y + 0.075
        for k, (a, b) in enumerate([(0, 10), (10, 20)]):
            fig.add_artist(Rectangle((sx0 + a * fpk, sy0), (b - a) * fpk, 0.008, fc="black" if k == 0 else "white", ec="black", lw=0.4, transform=fig.transFigure))
        for v in (0, 10, 20):
            fig.text(sx0 + v * fpk, sy0 - 0.004, str(v), ha="center", va="top", fontsize=5.2)
        fig.text(sx0 + 10 * fpk, sy0 + 0.012, "Kilometres", ha="center", va="bottom", fontsize=5.2)
    fig.savefig(FIG_DIR + "/susceptibility_maps_hillshade.png", facecolor="white")
    plt.show(); plt.close(fig)
    print("ok")

figure_hillshade_susceptibility()

## 21. Ward-level composite flood risk (Figure 16)
Each 30 m cell is assigned to the ward that contains its centre. For every ward the area-weighted mean RF susceptibility (S̄) and mean coping-capacity index (C̄) give the composite risk index **R = S̄ × (1 − C̄)**, classified into five classes by Fisher–Jenks natural breaks (exact on the 62 values). Population (native ~92 m grid) is assigned to wards by cell centre and counted in total, on high-susceptibility land (RF ≥ 0.6) and in the bivariate hotspot (high susceptibility, low capacity).

Outputs in `OUT_DIR/wards`: `fct_ward_flood_risk.csv`, `ward_id_raster.npy`, `summary.json` and `fct_ward_composite_flood_risk.png`.

In [ ]:
def ward_composite_risk():
    import sys, json, math, numpy as np, pandas as pd
    from matplotlib.path import Path

    OUT = WARD_DIR + "/"

    rf = imread(MAP_DIR + "/" + "FCT_flood_susceptibility_RF.tif").astype(np.float64); rf[rf < 0] = np.nan
    cci = imread(MAP_DIR + "/" + "FCT_coping_capacity_index.tif").astype(np.float64); cci[cci < -1] = np.nan
    biv = imread(MAP_DIR + "/" + "FCT_bivariate_susceptibility_capacity.tif")
    H, W = rf.shape
    lats = OY - (np.arange(H) + 0.5) * SX
    area_row = SX * 111320.0 * np.cos(np.radians(lats)) * SX * 110574.0 / 1e6     # km2 per pixel, as in the models

    fields, recs = read_dbf(WARD_DBF)
    shapes = read_shp(WARD_SHP)
    idx = [i for i, r in enumerate(recs) if r["statecode"] == WARD_STATE_CODE]

    wid = np.zeros((H, W), np.int16)            # ward id raster (1..62)
    ward_area_poly = {}
    for k, i in enumerate(idx, start=1):
        (x0, y0, x1, y1), rings = shapes[i]
        c0 = max(int(math.floor((x0 - OX) / SX)), 0); c1 = min(int(math.ceil((x1 - OX) / SX)), W - 1)
        r0 = max(int(math.floor((OY - y1) / SX)), 0); r1 = min(int(math.ceil((OY - y0) / SX)), H - 1)
        if c1 < c0 or r1 < r0:
            continue
        cc, rr = np.meshgrid(np.arange(c0, c1 + 1), np.arange(r0, r1 + 1))
        pts = np.column_stack([OX + (cc.ravel() + 0.5) * SX, OY - (rr.ravel() + 0.5) * SX])
        inside = np.zeros(len(pts), bool)
        for ring in rings:                       # even-odd rule handles holes and multipart wards
            inside ^= Path(ring).contains_points(pts)
        sub = wid[r0:r1 + 1, c0:c1 + 1].ravel()
        sub[inside & (sub == 0)] = k
        wid[r0:r1 + 1, c0:c1 + 1] = sub.reshape(cc.shape)
        # polygon area (km2) by shoelace on a local equirectangular projection, signed rings summed
        a = 0.0
        for ring in rings:
            lat0 = np.radians(ring[:, 1].mean())
            x = np.radians(ring[:, 0]) * 6371.0088 * np.cos(lat0); y = np.radians(ring[:, 1]) * 6371.0088
            a += 0.5 * (np.dot(x[:-1], y[1:]) - np.dot(x[1:], y[:-1]))
        ward_area_poly[k] = abs(a)

    # population on its native grid, assigned to wards by cell centre
    pop, (pox, psx, poy, psy), _, pnd = read_tif(FDIR + "/" + "fct_pop.tif")
    pop = pop.astype(np.float64)
    if pnd is not None: pop[np.isclose(pop, pnd)] = 0
    pop[~np.isfinite(pop) | (pop < 0)] = 0
    pr, pc = np.nonzero(pop)
    plon = pox + (pc + 0.5) * psx; plat = poy + (pr + 0.5) * psy
    rr = np.floor((OY - plat) / SX).astype(int); cc = np.floor((plon - OX) / SX).astype(int)
    ok = (rr >= 0) & (rr < H) & (cc >= 0) & (cc < W)
    pw = pop[pr[ok], pc[ok]]; pid = wid[rr[ok], cc[ok]]; pbiv = biv[rr[ok], cc[ok]]
    prf = rf[rr[ok], cc[ok]]

    AREA = np.broadcast_to(area_row[:, None], (H, W))
    rows = []
    for k, i in enumerate(idx, start=1):
        m = wid == k
        v = m & np.isfinite(rf) & np.isfinite(cci)
        a_all = AREA[m].sum(); a_v = AREA[v].sum()
        if a_v == 0:
            continue
        w = AREA[v]
        S_mean = np.average(rf[v], weights=w)
        C_mean = np.average(cci[v], weights=w)
        high = v & (rf >= 0.6)
        hot = m & (biv == 9)
        pm = pid == k
        rows.append({
            "ward_id": k, "wardname": recs[i]["wardname"], "wardcode": recs[i]["wardcode"],
            "area_council": recs[i]["lganame"].replace("Municipal Area Council", "AMAC"),
            "area_km2": round(ward_area_poly[k], 1), "covered_km2": round(a_v, 1),
            "coverage_pct": round(100 * a_v / ward_area_poly[k], 1),
            "mean_susceptibility": S_mean, "high_susc_pct": 100 * AREA[high].sum() / a_v,
            "mean_cci": C_mean,
            "population": pw[pm].sum(),
            "pop_high_susc": pw[pm & (prf >= 0.6)].sum(),
            "hotspot_km2": AREA[hot].sum(), "pop_hotspot": pw[pm & (pbiv == 9)].sum(),
        })
    df = pd.DataFrame(rows)
    # Composite risk index: hazard x lack of coping capacity, both on 0-1 scales
    df["risk_index"] = df["mean_susceptibility"] * (1 - df["mean_cci"])

    # five classes by natural breaks (Fisher-Jenks, exact on 62 values)
    def jenks(v, k=5):
        v = np.sort(np.asarray(v)); n = len(v)
        cs, cs2 = np.r_[0, np.cumsum(v)], np.r_[0, np.cumsum(v * v)]
        def ssd(a, b):
            s = cs[b + 1] - cs[a]; return cs2[b + 1] - cs2[a] - s * s / (b - a + 1)
        cost = np.full((k, n), np.inf); back = np.zeros((k, n), int)
        for b in range(n): cost[0, b] = ssd(0, b)
        for c in range(1, k):
            for b in range(c, n):
                best = [(cost[c - 1, a - 1] + ssd(a, b), a) for a in range(c, b + 1)]
                cost[c, b], back[c, b] = min(best)
        cuts, b = [], n - 1
        for c in range(k - 1, 0, -1):
            a = back[c, b]; cuts.append((v[a - 1] + v[a]) / 2); b = a - 1
        return sorted(cuts)
    br = jenks(df["risk_index"].values)
    names = ["Very low", "Low", "Moderate", "High", "Very high"]
    df["risk_class"] = [names[int(np.digitize(x, br))] for x in df["risk_index"]]
    df["rank"] = df["risk_index"].rank(ascending=False).astype(int)
    df = df.sort_values("rank")
    df.to_csv(OUT + "fct_ward_flood_risk.csv", index=False)
    np.save(OUT + "ward_id_raster.npy", wid)
    json.dump({"breaks": br, "fct_wards": len(idx), "n_with_data": len(df),
               "pop_total_assigned": float(pw[pid > 0].sum()), "pop_total": float(pw.sum()),
               "unassigned_valid_km2": float(AREA[(wid == 0) & np.isfinite(rf)].sum())},
              open(OUT + "summary.json", "w"), indent=1)
    log(json.load(open(OUT + "summary.json")))
    pd.set_option("display.width", 250); pd.set_option("display.max_rows", 80)
    log(df[["rank", "wardname", "area_council", "area_km2", "coverage_pct", "mean_susceptibility", "high_susc_pct", "mean_cci",
              "population", "hotspot_km2", "pop_hotspot", "risk_index", "risk_class"]].round(3).to_string(index=False))

ward_composite_risk()

In [ ]:
def figure_ward_composite_risk():
    import sys, math, json, numpy as np, pandas as pd, matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.patches import PathPatch, Rectangle, Polygon
    from matplotlib.path import Path
    from matplotlib.colors import ListedColormap

    OUT = WARD_DIR + "/"
    df = pd.read_csv(OUT + "fct_ward_flood_risk.csv")
    br = json.load(open(OUT + "summary.json"))["breaks"]
    fields, recs = read_dbf(WARD_DBF)
    shapes = read_shp(WARD_SHP)
    idx = [i for i, r in enumerate(recs) if r["statecode"] == WARD_STATE_CODE]
    code2id = {recs[i]["wardcode"]: k for k, i in enumerate(idx, start=1)}
    if "ward_id" not in df: df["ward_id"] = df.wardcode.map(code2id)
    wid = np.load(OUT + "ward_id_raster.npy")

    NAMES = ["Very low", "Low", "Moderate", "High", "Very high"]
    COL = ["#fef0d9", "#fdcc8a", "#fc8d59", "#e34a33", "#b30000"]
    cmap = dict(zip(NAMES, COL))
    plt.rcParams["font.family"] = "DejaVu Serif"

    fig = plt.figure(figsize=(7.2, 5.0), dpi=400)
    # ---------------- frame + map
    fr = fig.add_axes([0.01, 0.06, 0.585, 0.92]); fr.set_xticks([]); fr.set_yticks([])
    for sp in fr.spines.values(): sp.set_linewidth(0.9)
    fr.text(0.5, 0.985, "(a) Composite flood risk by ward", ha="center", va="top", fontsize=9, fontweight="bold", transform=fr.transAxes)
    mx = fig.add_axes([0.02, 0.09, 0.40, 0.84])
    for k, i in enumerate(idx, start=1):
        row = df[df.ward_id == k]
        if row.empty: continue
        rings = shapes[i][1]
        verts, codes = [], []
        for r in rings:
            verts += r.tolist(); codes += [Path.MOVETO] + [Path.LINETO] * (len(r) - 2) + [Path.CLOSEPOLY]
        mx.add_patch(PathPatch(Path(verts, codes), fc=cmap[row.risk_class.iloc[0]], ec="#6e6e6e", lw=0.25))
    # Area Council boundaries: contour of each council mask (closed to bridge slivers between ward polygons)
    from scipy import ndimage
    st = 2
    council_of = {k: recs[i]["lganame"] for k, i in enumerate(idx, start=1)}
    cn = sorted(set(council_of.values())); cid = {c: n + 1 for n, c in enumerate(cn)}
    lut = np.zeros(len(idx) + 1, np.int8)
    for k, c in council_of.items(): lut[k] = cid[c]
    cr = lut[wid[::st, ::st]]
    H, W = cr.shape
    xs = OX + (np.arange(W) + 0.5) * SX * st; ys = OY - (np.arange(H) + 0.5) * SX * st
    for c in cn:
        msk = ndimage.binary_closing(cr == cid[c], iterations=4)
        msk = ndimage.binary_fill_holes(msk)
        mx.contour(xs, ys, msk.astype(float), levels=[0.5], colors="#1e1e1e", linewidths=0.9, zorder=6)
    # outer boundary outline (all wards)
    for i in idx:
        for r in shapes[i][1]:
            pass
    # council labels
    lab = {"Municipal Area Council": "AMAC", "Abaji": "ABAJI", "Bwari": "BWARI", "Gwagwalada": "GWAGWALADA", "Kuje": "KUJE", "Kwali": "KWALI"}
    off = {"AMAC": (0.06, -0.04), "BWARI": (0.0, 0.05), "GWAGWALADA": (-0.03, 0.02), "KUJE": (0.03, -0.02), "KWALI": (0.0, -0.02), "ABAJI": (0.0, -0.03)}
    for c in cn:
        rr, ccc = np.nonzero(cr == cid[c])
        x = OX + (ccc.mean() + 0.5) * SX * st; y = OY - (rr.mean() + 0.5) * SX * st
        L = lab[c]; dx, dy = off[L]
        mx.text(x + dx, y + dy, L, fontsize=5.6, ha="center", va="center", color="#1a1a1a", fontweight="bold", zorder=7,
                bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.65))
    # rank numbers for the 10 highest-risk wards (callouts for small wards)
    top = df.nsmallest(10, "rank")
    small = []
    for _, r in top.iterrows():
        rr, ccc = np.nonzero(wid == r.ward_id)
        cy, cx = rr.mean(), ccc.mean(); j = np.argmin((rr - cy) ** 2 + (ccc - cx) ** 2)
        x = OX + (ccc[j] + 0.5) * SX; y = OY - (rr[j] + 0.5) * SX
        if r.area_km2 < 15:
            small.append((int(r["rank"]), x, y)); continue
        mx.text(x, y, str(int(r["rank"])), fontsize=5.2, ha="center", va="center", color="white", fontweight="bold", zorder=8,
                bbox=dict(boxstyle="circle,pad=0.15", fc="#222222", ec="white", lw=0.4))
    small.sort(key=lambda t: t[1])
    for n_, (rk, x, y) in enumerate(small):
        tx, ty = 7.00 + n_ * 0.06, 8.365
        mx.plot([x, tx], [y, ty + 0.012], color="#222222", lw=0.4, zorder=8)
        mx.plot([x], [y], "o", ms=1.2, color="#222222", zorder=8)
        mx.text(tx, ty, str(rk), fontsize=5.2, ha="center", va="center", color="white", fontweight="bold", zorder=9,
                bbox=dict(boxstyle="circle,pad=0.15", fc="#222222", ec="white", lw=0.4))
    mx.set_xlim(6.76, 7.74); mx.set_ylim(8.34, 9.43)
    mx.set_aspect(1 / math.cos(math.radians(9.0))); mx.axis("off")
    mx.apply_aspect(); pos = mx.get_position()
    # legend
    lx, ly = 0.425, 0.86
    fig.text(lx, ly, "Risk class", fontsize=6.6, fontweight="bold", va="top")
    fig.text(lx, ly - 0.03, "(index, no. of wards)", fontsize=5.4, va="top", color="#333")
    edges = [df.risk_index.min()] + br + [df.risk_index.max()]
    for k in range(5):
        yy = ly - 0.085 - (4 - k) * 0.052
        n = int((df.risk_class == NAMES[k]).sum())
        fig.add_artist(Rectangle((lx, yy), 0.024, 0.032, fc=COL[k], ec="#555", lw=0.4, transform=fig.transFigure))
        fig.text(lx + 0.03, yy + 0.016, f"{NAMES[k]}\n{edges[k]:.2f}–{edges[k + 1]:.2f} ({n})", fontsize=5.2, va="center", linespacing=1.1)
    yy = ly - 0.085 - 5 * 0.052 + 0.01
    fig.add_artist(plt.Line2D([lx, lx + 0.024], [yy, yy], color="#1e1e1e", lw=1.4, transform=fig.transFigure))
    fig.text(lx + 0.03, yy, "Area Council", fontsize=5.2, va="center")
    fig.text(lx, yy - 0.04, "", fontsize=5)
    fig.text(lx + 0.012, yy - 0.04, "1", fontsize=5, ha="center", va="center", color="white", fontweight="bold",
             bbox=dict(boxstyle="circle,pad=0.15", fc="#222222", ec="white", lw=0.4))
    fig.text(lx + 0.03, yy - 0.04, "Risk rank (top 10)", fontsize=5.2, va="center")
    # north arrow + scale bar
    nx, ny, s_ = 0.555, 0.26, 0.06
    fig.add_artist(Polygon([[nx, ny], [nx - 0.007, ny - s_], [nx, ny - 0.75 * s_]], closed=True, fc="black", ec="black", lw=0.5, transform=fig.transFigure))
    fig.add_artist(Polygon([[nx, ny], [nx + 0.007, ny - s_], [nx, ny - 0.75 * s_]], closed=True, fc="white", ec="black", lw=0.5, transform=fig.transFigure))
    fig.text(nx, ny + 0.004, "N", ha="center", va="bottom", fontsize=8, fontweight="bold")
    km_per_deg = 111.32 * math.cos(math.radians(9.0)); xl = mx.get_xlim()
    fpk = pos.width / ((xl[1] - xl[0]) * km_per_deg)
    sx0, sy0 = 0.43, 0.105
    for k, (a, b) in enumerate([(0, 10), (10, 20)]):
        fig.add_artist(Rectangle((sx0 + a * fpk, sy0), (b - a) * fpk, 0.007, fc="black" if k == 0 else "white", ec="black", lw=0.4, transform=fig.transFigure))
    for v in (0, 10, 20):
        fig.text(sx0 + v * fpk, sy0 - 0.004, str(v), ha="center", va="top", fontsize=5.2)
    fig.text(sx0 + 10 * fpk, sy0 + 0.011, "Kilometres", ha="center", va="bottom", fontsize=5.2)

    # ---------------- ranked bars (top 15)
    fr2 = fig.add_axes([0.605, 0.06, 0.385, 0.92]); fr2.set_xticks([]); fr2.set_yticks([])
    for sp in fr2.spines.values(): sp.set_linewidth(0.9)
    fr2.text(0.5, 0.985, "(b) Fifteen highest-risk wards", ha="center", va="top", fontsize=9, fontweight="bold", transform=fr2.transAxes)
    ax = fig.add_axes([0.745, 0.225, 0.112, 0.675])
    t = df.nsmallest(15, "rank").iloc[::-1]
    ax.barh(range(len(t)), t.risk_index, color=[cmap[c] for c in t.risk_class], ec="#555", lw=0.35, height=0.72)
    ax.set_yticks(range(len(t)))
    ax.set_yticklabels([f"{int(r['rank'])}. {r.wardname} ({r.area_council})" for _, r in t.iterrows()], fontsize=5.4)
    for y_, (_, r) in enumerate(t.iterrows()):
        ax.text(r.risk_index + 0.012, y_, f"{r.risk_index:.2f}", fontsize=4.8, va="center")
    ax.set_xlim(0, 0.85); ax.set_xticks([0, 0.3, 0.6]); ax.tick_params(axis="x", labelsize=5.2); ax.tick_params(axis="y", length=0)
    ax.set_xlabel("Risk index\nS \u00d7 (1 \u2212 CCI)", fontsize=5.6)
    ax.set_title("Risk index", fontsize=6.2, loc="left", fontweight="bold")
    for sp in ["top", "right"]: ax.spines[sp].set_visible(False)
    # population panel: total vs living on high-susceptibility land (probability >= 0.6)
    ax2 = fig.add_axes([0.872, 0.225, 0.098, 0.675], sharey=ax)
    ax2.barh(range(len(t)), t.population / 1000, color="#deebf7", ec="#555", lw=0.35, height=0.72, label="Total population")
    ax2.barh(range(len(t)), t.pop_high_susc / 1000, color="#2171b5", ec="#555", lw=0.35, height=0.72, label="On high-susceptibility land")
    for y_, (_, r) in enumerate(t.iterrows()):
        ax2.text(r.population / 1000 + 0.8, y_, f"{r.pop_high_susc/1000:.1f}k", fontsize=4.6, va="center")
    ax2.set_xlim(0, 52); ax2.set_xticks([0, 20, 40]); ax2.tick_params(axis="x", labelsize=5.2); ax2.tick_params(axis="y", length=0, labelleft=False)
    ax2.set_xlabel("Population\n(thousands)", fontsize=5.6)
    ax2.set_title("People exposed", fontsize=6.2, loc="left", fontweight="bold")
    for sp in ["top", "right"]: ax2.spines[sp].set_visible(False)
    h_, l_ = ax2.get_legend_handles_labels()
    fig.legend(h_, l_, fontsize=4.8, frameon=False, loc="center", bbox_to_anchor=(0.7975, 0.122), ncol=2, handlelength=1.2, columnspacing=1.0)
    tot_h = df.nsmallest(15, "rank").pop_high_susc.sum()
    fig.text(0.7975, 0.100, f"S = mean RF susceptibility; CCI = coping capacity index; high = p \u2265 0.6.\nThese 15 wards: about {tot_h/1000:.0f},000 people on high-susceptibility land.",
             fontsize=4.6, ha="center", color="#444", style="italic", va="top")
    fig.savefig(OUT + "fct_ward_composite_flood_risk.png", facecolor="white")
    plt.show(); plt.close(fig)
    print("ok")

figure_ward_composite_risk()

## 22. Vector layers and the ward-level bivariate map (Figure 15)
The RF susceptibility (five classes and three bivariate classes) and the coping-capacity terciles are generalised to ~90 m (3 × 3 block majority) and patches smaller than five cells are merged into their neighbours, then converted to polygons that follow the cell edges exactly. Their union with the wards gives one polygon per ward × susceptibility × capacity combination, and each ward is assigned the bivariate class of its area-weighted mean susceptibility and CCI.

Outputs in `OUT_DIR/vectors`: `FCT_RF_susceptibility_polygons`, `FCT_coping_capacity_polygons`, `FCT_ward_susceptibility_capacity_union`, `FCT_ward_bivariate_RF_CCI` (shapefiles), a CSV of the ward classes and `FCT_bivariate_RF_CCI_wards.png`.

In [ ]:
def vectorise_and_union():
    import sys, json, math, os, numpy as np, pandas as pd
    from scipy import ndimage
    from skimage.measure import label as sklabel

    OUT = VEC_DIR + "/"
    B = 3                       # 3 x 3 blocks -> ~90 m cells for vectorisation
    MIN_CELLS = 5               # patches smaller than 5 cells (~4 ha) merged into their neighbours
    PRJ = 'GEOGCS["GCS_WGS_1984",DATUM["D_WGS_1984",SPHEROID["WGS_1984",6378137.0,298.257223563]],PRIMEM["Greenwich",0.0],UNIT["Degree",0.0174532925199433]]'

    p = imread(MAP_DIR + "/FCT_flood_susceptibility_RF.tif").astype(float); p[p < 0] = np.nan
    c = imread(MAP_DIR + "/FCT_coping_capacity_index.tif").astype(float); c[c < -1] = np.nan
    wid = np.load(WARD_DIR + "/ward_id_raster.npy")
    t1, t2 = np.nanquantile(c, [1 / 3, 2 / 3])
    print("CCI terciles", round(t1, 3), round(t2, 3))
    H, W = p.shape; h, w = H // B, W // B
    def bmean(a):
        a = a[:h * B, :w * B].reshape(h, B, w, B)
        with np.errstate(invalid="ignore"):
            return np.nanmean(a, axis=(1, 3))
    pb, cb = bmean(p), bmean(c)
    wb = wid[1:h * B:B, 1:w * B:B][:h, :w]
    px = SX * B
    lats = OY - (np.arange(h) + 0.5) * px
    cell_km2 = (px * 111320.0 * np.cos(np.radians(lats)) * px * 110574.0 / 1e6)[:, None] * np.ones((1, w))

    def sieve(cls, nod=0, minc=MIN_CELLS):
        """Merge patches smaller than minc cells into the nearest larger patch (like ArcGIS Majority/Nibble)."""
        cls = cls.copy()
        lab = sklabel(cls, background=nod, connectivity=1)
        sizes = np.bincount(lab.ravel())
        small = (sizes[lab] < minc) & (cls != nod)
        if small.any():
            good = ~small & (cls != nod)
            idx = ndimage.distance_transform_edt(~good, return_distances=False, return_indices=True)
            cls[small] = cls[tuple(idx)][small]
        return cls

    def bmajority(cls30, k):
        """3x3 block majority of a 30 m class grid (0 = NoData); NoData if most of the block is NoData."""
        a = cls30[:h * B, :w * B].reshape(h, B, w, B).transpose(0, 2, 1, 3).reshape(h, w, B * B)
        cnt = np.stack([(a == v).sum(-1) for v in range(k + 1)], -1)
        return cnt.argmax(-1).astype(np.int32)
    v30 = np.isfinite(p) & np.isfinite(c)
    s5_30 = np.zeros(p.shape, np.int32); f = np.isfinite(p); s5_30[f] = np.digitize(p[f], [0.2, 0.4, 0.6, 0.8]) + 1
    s3_30 = np.zeros(p.shape, np.int32); s3_30[v30] = np.digitize(p[v30], [0.4, 0.6]) + 1               # 1 low, 2 moderate, 3 high
    c3_30 = np.zeros(p.shape, np.int32); c3_30[v30] = 3 - np.digitize(c[v30], [t1, t2])                # 1 high, 2 medium, 3 low capacity
    s5 = sieve(bmajority(s5_30, 5)); s3 = sieve(bmajority(s3_30, 3)); c3 = sieve(bmajority(c3_30, 3))
    valid = (s3 > 0) & (c3 > 0)
    S5N = {1: "Very low", 2: "Low", 3: "Moderate", 4: "High", 5: "Very high"}
    S3N = {1: "Low", 2: "Moderate", 3: "High"}
    C3N = {1: "High", 2: "Medium", 3: "Low"}
    fields_area = [("AREA_KM2", "N", 14, 4)]

    def export(name, L, attr_fn, fields):
        polys = polygonize(L, OX, OY, px)
        feats = []
        for v, rings in polys:
            # area of polygon from cells (exact, since polygons follow cell edges)
            feats.append((rings, attr_fn(v)))
        # area via ring shoelace on local projection
        for rings, a in feats:
            tot = 0.0
            for r in rings:
                lat0 = math.radians(r[:, 1].mean())
                x = r[:, 0] * 111.320 * math.cos(lat0); y = r[:, 1] * 110.574
                tot += -0.5 * np.sum(x[:-1] * y[1:] - x[1:] * y[:-1])
            a["AREA_KM2"] = tot
        write_shapefile(OUT + name, feats, fields + fields_area, PRJ)
        print(name, len(feats), "polygons")
        return feats

    f_s5 = export("FCT_RF_susceptibility_polygons", s5,
                  lambda v: {"CLASS_ID": v, "SUSC_CLASS": S5N[v], "PROB_RANGE": f"{0.2*(v-1):.1f}-{0.2*v:.1f}"},
                  [("CLASS_ID", "N", 2, 0), ("SUSC_CLASS", "C", 12, 0), ("PROB_RANGE", "C", 8, 0)])
    f_c3 = export("FCT_coping_capacity_polygons", c3,
                  lambda v: {"CAP_ID": v, "CAP_CLASS": C3N[v], "CCI_RANGE": {1: f">={t2:.3f}", 2: f"{t1:.3f}-{t2:.3f}", 3: f"<{t1:.3f}"}[v]},
                  [("CAP_ID", "N", 2, 0), ("CAP_CLASS", "C", 8, 0), ("CCI_RANGE", "C", 14, 0)])

    # 3) union of susceptibility, capacity and wards (= ArcGIS Union/Intersect of the three layers)
    fields_w, recs = read_dbf(WARD_DBF)
    idx = [i for i, r in enumerate(recs) if r["statecode"] == WARD_STATE_CODE]
    wname = {k: recs[i]["wardname"] for k, i in enumerate(idx, start=1)}
    wcode = {k: recs[i]["wardcode"] for k, i in enumerate(idx, start=1)}
    wlga = {k: recs[i]["lganame"].replace("Municipal Area Council", "AMAC") for k, i in enumerate(idx, start=1)}
    U = np.where(valid & (wb > 0) & (s3 > 0) & (c3 > 0), wb * 100 + s3 * 10 + c3, 0).astype(np.int32)
    def uattr(v):
        k, s_, c_ = v // 100, (v // 10) % 10, v % 10
        code = (c_ - 1) * 3 + s_
        return {"WARD": wname[k], "WARDCODE": wcode[k], "COUNCIL": wlga[k], "SUSC": S3N[s_], "CAPACITY": C3N[c_],
                "BIV_CODE": code, "BIV_CLASS": f"{S3N[s_]} susc. / {C3N[c_].lower()} cap.", "HOTSPOT": "Yes" if code == 9 else "No"}
    f_u = export("FCT_ward_susceptibility_capacity_union", U, uattr,
                 [("WARD", "C", 40, 0), ("WARDCODE", "C", 10, 0), ("COUNCIL", "C", 12, 0), ("SUSC", "C", 9, 0), ("CAPACITY", "C", 8, 0),
                  ("BIV_CODE", "N", 2, 0), ("BIV_CLASS", "C", 34, 0), ("HOTSPOT", "C", 3, 0)])

    # 4) ward-level bivariate class from the union table (area-weighted)
    ut = pd.DataFrame([a for _, a in f_u])
    ward_rows = []
    lat30 = OY - (np.arange(H) + 0.5) * SX
    AREA30 = np.broadcast_to((SX * 111320.0 * np.cos(np.radians(lat30)) * SX * 110574.0 / 1e6)[:, None], (H, W))
    pw = pd.read_csv(WARD_DIR + "/fct_ward_flood_risk.csv")
    for k in sorted(wname):
        m = (wid == k) & v30
        if not m.any(): continue
        wt = AREA30[m]
        S_mean = np.average(p[m], weights=wt); C_mean = np.average(c[m], weights=wt)
        s_cls = int(np.digitize(S_mean, [0.4, 0.6]) + 1); c_cls = int(3 - np.digitize(C_mean, [t1, t2]))
        sub = ut[ut.WARDCODE == wcode[k]]
        tot = sub.AREA_KM2.sum()
        hs = sub[sub.HOTSPOT == "Yes"].AREA_KM2.sum()
        hi = sub[sub.SUSC == "High"].AREA_KM2.sum()
        prow = pw[pw.wardcode == wcode[k]]
        ward_rows.append({"ward_id": k, "WARD": wname[k], "WARDCODE": wcode[k], "COUNCIL": wlga[k],
                          "AREA_KM2": tot, "RF_MEAN": S_mean, "CCI_MEAN": C_mean,
                          "SUSC": S3N[s_cls], "CAPACITY": C3N[c_cls], "BIV_CODE": (c_cls - 1) * 3 + s_cls,
                          "BIV_CLASS": f"{S3N[s_cls]} susc. / {C3N[c_cls].lower()} cap.",
                          "HIGHSUSC_PCT": 100 * hi / tot if tot else 0, "HOTSPOT_KM2": hs, "HOTSPOT_PCT": 100 * hs / tot if tot else 0,
                          "POP": float(prow.population.iloc[0]) if len(prow) else np.nan})
    wdf = pd.DataFrame(ward_rows)
    shapes = read_shp(WARD_SHP)
    feats = []
    for _, r in wdf.iterrows():
        rings = [np.asarray(x) for x in shapes[idx[r.ward_id - 1]][1]]
        feats.append((rings, r.to_dict()))
    write_shapefile(OUT + "FCT_ward_bivariate_RF_CCI", feats,
                    [("WARD", "C", 40, 0), ("WARDCODE", "C", 10, 0), ("COUNCIL", "C", 12, 0), ("AREA_KM2", "N", 12, 2),
                     ("RF_MEAN", "N", 8, 3), ("CCI_MEAN", "N", 8, 3), ("SUSC", "C", 9, 0), ("CAPACITY", "C", 8, 0),
                     ("BIV_CODE", "N", 2, 0), ("BIV_CLASS", "C", 34, 0), ("HIGHSUSC_PCT", "N", 7, 1), ("HOTSPOT_KM2", "N", 10, 2),
                     ("HOTSPOT_PCT", "N", 7, 1), ("POP", "N", 10, 0)], PRJ)
    wdf.drop(columns="ward_id").to_csv(OUT + "FCT_ward_bivariate_RF_CCI.csv", index=False)
    np.save(OUT + "union_grid.npy", U); np.save(OUT + "s5_grid.npy", s5); np.save(OUT + "c3_grid.npy", c3); np.save(OUT + "s3_grid.npy", s3)
    json.dump({"t1": t1, "t2": t2, "block": B, "min_cells": MIN_CELLS}, open(OUT + "params.json", "w"))
    pd.set_option("display.width", 200)
    print(wdf.BIV_CLASS.value_counts())
    print(wdf.groupby("BIV_CODE").agg(n=("WARD", "size"), pop=("POP", "sum")))
    print("union area total", round(ut.AREA_KM2.sum(), 1), "hotspot km2", round(ut[ut.HOTSPOT == "Yes"].AREA_KM2.sum(), 1))

vectorise_and_union()

In [ ]:
def figure_ward_bivariate():
    import sys, math, numpy as np, pandas as pd, matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.patches import PathPatch, Rectangle, Polygon
    from matplotlib.path import Path
    from matplotlib.collections import PatchCollection

    O = VEC_DIR + "/"
    BIV = [["#F7F2FB", "#FBB79D", "#F6772D"], ["#9DC2EB", "#AE8595", "#AC403F"], ["#3A90D8", "#685486", "#68003B"]]
    def col(code):
        code = int(code); cap = (code - 1) // 3; s = (code - 1) % 3
        return BIV[cap][s]
    plt.rcParams["font.family"] = "DejaVu Serif"

    def patches_of(shp_path, dbf_path, colour_field, lw=0.0, ec="none"):
        f, recs = read_dbf(dbf_path); shapes = read_shp(shp_path)
        pats, cols = [], []
        for r, (bb, rings) in zip(recs, shapes):
            v, c = [], []
            for ring in rings:
                v += ring.tolist(); c += [Path.MOVETO] + [Path.LINETO] * (len(ring) - 2) + [Path.CLOSEPOLY]
            pats.append(PathPatch(Path(v, c))); cols.append(col(r[colour_field]))
        return pats, cols, recs, shapes

    wards_f, wards_r = read_dbf(WARD_DBF)
    wards_s = read_shp(WARD_SHP)
    fct = [i for i, r in enumerate(wards_r) if r["statecode"] == WARD_STATE_CODE]
    def ward_outline(ax, lw=0.35, color="#3c3c3c"):
        for i in fct:
            for ring in wards_s[i][1]:
                ax.plot(ring[:, 0], ring[:, 1], color=color, lw=lw, zorder=5)

    fig = plt.figure(figsize=(7.2, 4.6), dpi=400)
    panels = [(0.008, "(a) Susceptibility × coping capacity polygons"), (0.504, "(b) Ward-level bivariate class")]
    wdf = pd.read_csv(O + "FCT_ward_bivariate_RF_CCI.csv")
    hot = wdf[wdf.BIV_CODE == 9].sort_values("RF_MEAN", ascending=False).reset_index(drop=True)
    for n, (x0, title) in enumerate(panels):
        fr = fig.add_axes([x0, 0.02, 0.488, 0.96]); fr.set_xticks([]); fr.set_yticks([])
        for sp in fr.spines.values(): sp.set_linewidth(0.9)
        fr.text(0.5, 0.982, title, ha="center", va="top", fontsize=8.6, fontweight="bold", transform=fr.transAxes)
        ax = fig.add_axes([x0 + 0.006, 0.04, 0.476, 0.88])
        if n == 0:
            pats, cols, _, _ = patches_of(O + "FCT_ward_susceptibility_capacity_union.shp", O + "FCT_ward_susceptibility_capacity_union.dbf", "BIV_CODE")
            ax.add_collection(PatchCollection(pats, facecolors=cols, edgecolors="none", lw=0, zorder=2))
            ward_outline(ax, lw=0.3, color="#2b2b2b")
        else:
            pats, cols, recs, shapes = patches_of(O + "FCT_ward_bivariate_RF_CCI.shp", O + "FCT_ward_bivariate_RF_CCI.dbf", "BIV_CODE")
            ax.add_collection(PatchCollection(pats, facecolors=cols, edgecolors="#4a4a4a", lw=0.3, zorder=2))
            # number the hotspot wards
            name_to_rings = {r["WARD"]: rings for r, (bb, rings) in zip(recs, shapes)}
            offs = {"Pandagi": (0, 0), "Yaba": (0, 0), "Ebagi": (0, 0), "Ashara": (0, 0), "North East": (0, 0),
                    "Gurdi": (0, 0), "Alu Mamagi": (0, 0), "Dafa": (0, 0), "Wako": (0, 0)}
            for k, r in hot.iterrows():
                ring = max(name_to_rings[r.WARD], key=len)
                pth = Path(ring); cx, cy = ring[:, 0].mean(), ring[:, 1].mean()
                if not pth.contains_point((cx, cy)):   # fall back to an interior grid point
                    xs = np.linspace(ring[:, 0].min(), ring[:, 0].max(), 25); ys = np.linspace(ring[:, 1].min(), ring[:, 1].max(), 25)
                    gx, gy = np.meshgrid(xs, ys); pts = np.c_[gx.ravel(), gy.ravel()]; ins = pts[pth.contains_points(pts)]
                    d = (ins[:, 0] - cx) ** 2 + (ins[:, 1] - cy) ** 2; cx, cy = ins[d.argmin()]
                ax.text(cx, cy, str(k + 1), fontsize=5.0, ha="center", va="center", color="white", fontweight="bold", zorder=8,
                        bbox=dict(boxstyle="circle,pad=0.14", fc="#222222", ec="white", lw=0.35))
            # key of hotspot wards (upper-left, empty corner)
            ax.text(6.775, 9.52, "Hotspot wards (high susceptibility, low capacity)", fontsize=5.0, fontweight="bold", va="top", zorder=9)
            for k, r in hot.iterrows():
                cx_, cy_ = (6.775, 9.488 - k * 0.026) if k < 5 else (7.06, 9.488 - (k - 5) * 0.026)
                ax.text(cx_, cy_, f"{k + 1}. {r.WARD} ({r.COUNCIL})", fontsize=4.7, va="top", zorder=9)
        ax.set_xlim(6.76, 7.75); ax.set_ylim(8.40, 9.53)
        ax.set_aspect(1 / math.cos(math.radians(9.0))); ax.axis("off")
        ax.apply_aspect(); pos = ax.get_position()
        # 3 x 3 legend (lower right): x = susceptibility, y = coping capacity (low at top)
        lx, ly, cs = x0 + 0.37, 0.115, 0.024
        fig.add_artist(Rectangle((lx - 0.03, ly - 0.042), 3 * cs + 0.04, 3 * cs * 1.55 + 0.075, fc="white", ec="none", alpha=0.92, transform=fig.transFigure))
        for cap in range(3):
            for s in range(3):
                fig.add_artist(Rectangle((lx + s * cs, ly + cap * cs * 1.55), cs, cs * 1.55, fc=BIV[cap][s], ec="white", lw=0.5, transform=fig.transFigure))
        fig.add_artist(plt.Line2D([lx, lx + 3 * cs], [ly - 0.006, ly - 0.006], color="#333", lw=0.6, transform=fig.transFigure))
        fig.text(lx, ly - 0.009, "Low", fontsize=4.4, ha="left", va="top", color="#333")
        fig.text(lx + 3 * cs, ly - 0.009, "High", fontsize=4.4, ha="right", va="top", color="#333")
        fig.text(lx + 1.5 * cs, ly - 0.026, "Susceptibility", fontsize=4.9, ha="center", va="top")
        fig.add_artist(plt.Line2D([lx - 0.005, lx - 0.005], [ly, ly + 3 * cs * 1.55], color="#333", lw=0.6, transform=fig.transFigure))
        fig.text(lx - 0.008, ly, "High", fontsize=4.4, ha="right", va="bottom", rotation=90, color="#333")
        fig.text(lx - 0.008, ly + 3 * cs * 1.55, "Low", fontsize=4.4, ha="right", va="top", rotation=90, color="#333")
        fig.text(lx - 0.021, ly + 1.5 * cs * 1.55, "Coping capacity", fontsize=4.9, ha="center", va="center", rotation=90)
        fig.text(lx + 1.5 * cs, ly + 3 * cs * 1.55 + 0.012, "Susceptibility \u00d7 CCI", fontsize=5.2, ha="center", fontweight="bold")
        # north arrow (N above tip) and scale bar
        nx, ny, s_ = x0 + 0.455, 0.85, 0.055
        fig.add_artist(Polygon([[nx, ny], [nx - 0.006, ny - s_], [nx, ny - 0.75 * s_]], closed=True, fc="black", ec="black", lw=0.5, transform=fig.transFigure))
        fig.add_artist(Polygon([[nx, ny], [nx + 0.006, ny - s_], [nx, ny - 0.75 * s_]], closed=True, fc="white", ec="black", lw=0.5, transform=fig.transFigure))
        fig.text(nx, ny + 0.004, "N", ha="center", va="bottom", fontsize=7.5, fontweight="bold")
        km = 111.32 * math.cos(math.radians(9.0)); xl = ax.get_xlim(); fpk = pos.width / ((xl[1] - xl[0]) * km)
        sx0, sy0 = x0 + 0.03, 0.06
        for k, (a, b) in enumerate([(0, 10), (10, 20)]):
            fig.add_artist(Rectangle((sx0 + a * fpk, sy0), (b - a) * fpk, 0.007, fc="black" if k == 0 else "white", ec="black", lw=0.4, transform=fig.transFigure))
        for v in (0, 10, 20):
            fig.text(sx0 + v * fpk, sy0 - 0.004, str(v), ha="center", va="top", fontsize=5)
        fig.text(sx0 + 10 * fpk, sy0 + 0.011, "Kilometres", ha="center", va="bottom", fontsize=5)
    fig.savefig(O + "FCT_bivariate_RF_CCI_wards.png", facecolor="white")
    plt.show(); plt.close(fig)
    print("ok")

figure_ward_bivariate()

## 23. Exposed population in the fifteen highest-risk wards (Figure 17)
Exposed population = residents on cells that combine high susceptibility (RF ≥ 0.6) with low coping capacity (lowest CCI tercile), i.e. the bivariate hotspot class. Residents are summed in hexagons (circumradius 0.45 km, ≈0.53 km²) on a local equirectangular projection. The main map shows the whole FCT with the fifteen wards outlined; the five insets zoom on the wards with the most exposed residents. Set `BASEMAP_TIF` (section 17) to place the hexagons on satellite imagery instead of the hillshade.

In [ ]:
def figure_exposed_population_hexagons():
    import sys, os, math, numpy as np, pandas as pd, matplotlib
    import matplotlib.pyplot as plt, matplotlib.patheffects
    from matplotlib.patches import Rectangle, Polygon, PathPatch
    from matplotlib.path import Path
    from matplotlib.colors import ListedColormap, BoundaryNorm
    from matplotlib.collections import PolyCollection, PatchCollection

    OUT = WARD_DIR + "/"
    BASEMAP = BASEMAP_TIF
    LAT0, LON0 = 9.0, 6.75
    KX, KY = 111.320 * math.cos(math.radians(LAT0)), 110.574
    def km(lon, lat): return (np.asarray(lon) - LON0) * KX, (np.asarray(lat) - LAT0) * KY

    d = pd.read_csv(OUT + "fct_ward_flood_risk.csv").sort_values("rank")
    top = d.head(15).reset_index(drop=True)
    biv = imread(MAP_DIR + "/FCT_bivariate_susceptibility_capacity.tif")
    wid = np.load(OUT + "ward_id_raster.npy"); H, W = biv.shape
    fields, recs = read_dbf(WARD_DBF)
    shapes = read_shp(WARD_SHP)
    idx = [i for i, r in enumerate(recs) if r["statecode"] == WARD_STATE_CODE]
    code2k = {recs[i]["wardcode"]: k for k, i in enumerate(idx, start=1)}
    top_k = {code2k[c] for c in top.wardcode}

    # ---- exposed population = residents on hotspot cells (high susceptibility x low coping capacity, bivariate class 9)
    pop, (pox, psx, poy, psy), _, _ = read_tif(FDIR + "/" + "fct_pop.tif")
    pop = pop.astype(float); pop[~np.isfinite(pop) | (pop < 0)] = 0
    pr, pc = np.nonzero(pop)
    plon = pox + (pc + 0.5) * psx; plat = poy + (pr + 0.5) * psy
    rr = np.floor((OY - plat) / SX).astype(int); cc = np.floor((plon - OX) / SX).astype(int)
    ok = (rr >= 0) & (rr < H) & (cc >= 0) & (cc < W)
    pv, plon, plat, rr, cc = pop[pr[ok], pc[ok]], plon[ok], plat[ok], rr[ok], cc[ok]
    k_ = wid[rr, cc]; b_ = biv[rr, cc]
    m = np.isin(k_, list(top_k))
    exp = np.where(b_[m] == 9, pv[m], 0.0)
    x, y = km(plon[m], plat[m])
    by_ward = pd.Series(exp).groupby(k_[m]).sum()
    top["exposed"] = [by_ward.get(code2k[c], 0.0) for c in top.wardcode]
    print("check vs CSV pop_hotspot:", (top.exposed.round() - top.pop_hotspot).abs().max())
    five = top.sort_values("exposed", ascending=False).head(5).reset_index(drop=True)
    print(five[["rank", "wardname", "area_council", "exposed", "population", "risk_index"]])
    print("total exposed in 15 wards", round(exp.sum()), "of", round(pv[m].sum()))

    # ---- hexagon binning (pointy-top, circumradius A km)
    A = 0.45
    def hex_index(x, y):
        q = (math.sqrt(3) / 3 * x - y / 3) / A; r = (2 / 3 * y) / A
        X, Z = q, r; Y = -X - Z
        rx, ry, rz = np.round(X), np.round(Y), np.round(Z)
        ax_, ay_, az_ = abs(rx - X), abs(ry - Y), abs(rz - Z)
        fx = (ax_ > ay_) & (ax_ > az_); fz = ~fx & (ay_ <= az_)
        rx = np.where(fx, -ry - rz, rx); rz = np.where(fz, -rx - ry, rz)
        return rx.astype(int), rz.astype(int)
    hq, hr = hex_index(x, y)
    hx = pd.DataFrame({"q": hq, "r": hr, "e": exp, "p": pv[m]}).groupby(["q", "r"]).agg(e=("e", "sum"), p=("p", "sum")).reset_index()
    hx = hx[hx.p >= 1].copy()
    hx["x_km"] = A * math.sqrt(3) * (hx.q + hx.r / 2); hx["y_km"] = A * 1.5 * hx.r
    print("hexagons", len(hx), "with exposure", int((hx.e > 0).sum()), "max", round(hx.e.max()),
          "pct", np.percentile(hx.e[hx.e > 0], [25, 50, 75, 90, 99]).round())
    ang = np.radians(np.arange(6) * 60 + 30)
    verts = np.stack([np.c_[a + A * np.cos(ang), b + A * np.sin(ang)] for a, b in zip(hx.x_km, hx.y_km)])

    top_e = int(math.ceil(hx.e.max() / 100) * 100)
    BINS = [0, 0.5, 10, 25, 50, 100, 250, 500, 1000, max(top_e, 1001)]
    LAB = ["0", "(0; 10]", "(10; 25]", "(25; 50]", "(50; 100]", "(100; 250]", "(250; 500]", "(500; 1,000]", f"(1,000; {BINS[-1]:,}]"]
    COLS = ["#fff5eb", "#fdd0a2", "#fdae6b", "#f4794f", "#e04a3f", "#b8243f", "#86124d", "#510b45", "#14061c"]
    cmap = ListedColormap(COLS); norm = BoundaryNorm(BINS, len(COLS))
    order = np.argsort(hx.e.values); fc = cmap(norm(hx.e.values))
    fc[hx.e.values == 0, 3] = 0.45                # unexposed residents: faint, so imagery shows through

    # ---- basemap
    def load_basemap():
        if BASEMAP and os.path.exists(BASEMAP):
            img, (bx, bsx, by, bsy), _, _ = read_tif(BASEMAP)
            img = np.asarray(img)
            if img.ndim == 3 and img.shape[0] in (3, 4): img = np.moveaxis(img, 0, -1)
            h, w = img.shape[:2]; x0, y0 = km(bx, by + h * bsy); x1, y1 = km(bx + w * bsx, by)
            return img[..., :3], [float(x0), float(x1), float(y0), float(y1)], None, "Imagery: Google Satellite"
        hs, (bx, bsx, by, bsy), _, _ = read_tif(HILLSHADE_TIF)
        hs = hs.astype(float)[::2, ::2]; hs[hs == 0] = np.nan
        h, w = hs.shape; x0, y0 = km(bx, by + h * 2 * bsy); x1, y1 = km(bx + w * 2 * bsx, by)
        return hs, [float(x0), float(x1), float(y0), float(y1)], "gray", "Basemap: hillshade (placeholder for satellite imagery)"
    BM, BEXT, BCMAP, BNOTE = load_basemap()

    ward_path = {}
    for k, i in enumerate(idx, start=1):
        v, c = [], []
        for ring in shapes[i][1]:
            X, Y = km(ring[:, 0], ring[:, 1]); v += np.c_[X, Y].tolist()
            c += [Path.MOVETO] + [Path.LINETO] * (len(ring) - 2) + [Path.CLOSEPOLY]
        ward_path[k] = Path(v, c)
    def ward_label_xy(code):
        ring = max(shapes[idx[code2k[code] - 1]][1], key=len); X, Y = km(ring[:, 0], ring[:, 1])
        pth = Path(np.c_[X, Y]); c0 = np.array([X.mean(), Y.mean()])
        if not pth.contains_point(c0):
            gx, gy = np.meshgrid(np.linspace(X.min(), X.max(), 30), np.linspace(Y.min(), Y.max(), 30))
            pts = np.c_[gx.ravel(), gy.ravel()]; ins = pts[pth.contains_points(pts)]
            c0 = ins[((ins - c0) ** 2).sum(1).argmin()]
        return c0

    def draw(ax, xl, yl, lw_all=0.3, lw_top=0.6, focus=None):
        ax.set_facecolor("#ffffff")
        ax.imshow(BM, extent=BEXT, cmap=BCMAP, vmin=0 if BCMAP else None, vmax=255 if BCMAP else None, interpolation="bilinear", alpha=1.0 if BCMAP is None else 0.55, zorder=0)
        # veil over everything outside the 15 wards so the focus stands out while the whole FCT stays visible
        others = [PathPatch(ward_path[k]) for k in ward_path if k not in top_k]
        ax.add_collection(PatchCollection(others, facecolors=(1, 1, 1, 0.55), edgecolors=(0.25, 0.25, 0.25, 0.8), lw=lw_all, zorder=1))
        ax.add_collection(PolyCollection(verts[order], facecolors=fc[order], edgecolors="none", zorder=3))
        ax.add_collection(PatchCollection([PathPatch(ward_path[k]) for k in sorted(top_k)], facecolors="none",
                                          edgecolors="#111111", lw=lw_top, zorder=4))
        if focus is not None:
            ax.add_patch(PathPatch(ward_path[focus], fc="none", ec="#00e5ff", lw=1.4, zorder=5))
        ax.set_xlim(*xl); ax.set_ylim(*yl); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])

    def north(fig, nx, ny, s_, fs=7):
        fig.add_artist(Polygon([[nx, ny], [nx - 0.006, ny - s_], [nx, ny - 0.75 * s_]], closed=True, fc="black", ec="black", lw=0.5, transform=fig.transFigure))
        fig.add_artist(Polygon([[nx, ny], [nx + 0.006, ny - s_], [nx, ny - 0.75 * s_]], closed=True, fc="white", ec="black", lw=0.5, transform=fig.transFigure))
        fig.text(nx, ny + 0.003, "N", ha="center", va="bottom", fontsize=fs, fontweight="bold")
    def scalebar(ax, x0, y0, L, fs=4.6):
        h = (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.012
        ax.add_patch(Rectangle((x0 - L * 0.08, y0 - 4 * h), L * 1.22, 9.5 * h, fc="white", ec="none", alpha=0.85, zorder=9))
        ax.add_patch(Rectangle((x0, y0), L / 2, h, fc="black", ec="black", lw=0.4, zorder=10))
        ax.add_patch(Rectangle((x0 + L / 2, y0), L / 2, h, fc="white", ec="black", lw=0.4, zorder=10))
        for v in (0, L / 2, L):
            ax.text(x0 + v, y0 - h * 0.6, f"{v:g}", ha="center", va="top", fontsize=fs, zorder=10)
        ax.text(x0 + L / 2, y0 + h * 1.6, "km", ha="center", va="bottom", fontsize=fs, zorder=10)

    plt.rcParams["font.family"] = "DejaVu Serif"
    FW, FH = 7.2, 5.3
    fig = plt.figure(figsize=(FW, FH), dpi=400)
    fr = fig.add_axes([0.004, 0.004, 0.992, 0.992]); fr.set_xticks([]); fr.set_yticks([])
    for sp in fr.spines.values(): sp.set_linewidth(0.9)
    fig.text(0.5, 0.962, "Population exposed to flooding in the 15 highest-risk wards of the FCT, Abuja",
             ha="center", fontsize=8.6, fontweight="bold")

    # ---------- main map: the entire FCT
    XL, YL = (2.0, 109.0), (-67.5, 48.5)
    ax = fig.add_axes([0.012, 0.03, 0.585, 0.915]); draw(ax, XL, YL)
    for sp in ax.spines.values(): sp.set_linewidth(0.6)
    for _, r in top.iterrows():
        X, Y = ward_label_xy(r.wardcode)
        if r.area_km2 < 15: continue      # tiny Abaji-town wards: shown in the key, not labelled on the map
        ax.text(X, Y, str(r["rank"]), fontsize=4.2, ha="center", va="center", fontweight="bold", zorder=7,
                bbox=dict(boxstyle="circle,pad=0.15", fc="white", ec="#222", lw=0.3, alpha=0.9))
    BOX = "#00a8c6"
    INS = []
    for n, r in five.iterrows():
        ring = max(shapes[idx[code2k[r.wardcode] - 1]][1], key=len); X, Y = km(ring[:, 0], ring[:, 1])
        xc, yc = (X.min() + X.max()) / 2, (Y.min() + Y.max()) / 2
        half = max(X.max() - X.min(), Y.max() - Y.min()) / 2 * 1.12 + 0.8
        INS.append((n + 1, r, xc, yc, half))
        ax.add_patch(Rectangle((xc - half, yc - half), 2 * half, 2 * half, fill=False, ec=BOX, lw=0.9, zorder=8))
        ax.text(xc + half + 0.6, yc + half - 0.3, str(n + 1), color=BOX, fontsize=6.4, ha="left", va="top", fontweight="bold", zorder=8,
                path_effects=[matplotlib.patheffects.withStroke(linewidth=1.4, foreground="white")])
    # key of the 15 wards (upper right of main map, over the dimmed AMAC/Bwari area)
    kx, ky = 0.395, 0.915
    fig.add_artist(Rectangle((kx - 0.006, ky - 0.255), 0.205, 0.268, fc="white", ec="#666", lw=0.4, alpha=0.92, transform=fig.transFigure, zorder=20))
    fig.text(kx, ky, "15 highest-risk wards (rank)", fontsize=5.2, fontweight="bold", va="top", zorder=21)
    for j, r in top.iterrows():
        fig.text(kx, ky - 0.018 - j * 0.0155, f"{r['rank']:>2}. {r.wardname} ({r.area_council})", fontsize=4.5, va="top", zorder=21)
    north(fig, 0.04, 0.905, 0.045)
    scalebar(ax, 80, -48, 20)
    ax.text(XL[0] + 1, YL[0] + 1, BNOTE, fontsize=4.0, style="italic", color="#333", va="bottom", zorder=10,
            bbox=dict(fc="white", ec="none", alpha=0.8, pad=1))

    # ---------- five insets (2 x 3 grid on the right; sixth slot = legend)
    gx0, gy0, iw, ih, gap = 0.607, 0.03, 0.188, 0.302, 0.006
    slots = [(0, 2), (1, 2), (0, 1), (1, 1), (0, 0), (1, 0)]
    for (n, r, xc, yc, half), (cx_, cy_) in zip(INS, slots):
        left = gx0 + cx_ * (iw + gap); bottom = gy0 + cy_ * (ih + gap)
        ia = fig.add_axes([left, bottom, iw, ih * 0.79])
        draw(ia, (xc - half, xc + half), (yc - half, yc + half), lw_all=0.4, lw_top=0.7, focus=code2k[r.wardcode])
        for sp in ia.spines.values(): sp.set_edgecolor(BOX); sp.set_linewidth(1.0)
        fig.text(left + 0.003, bottom + ih * 0.79 + 0.034, f"{n}. {r.wardname} ({r.area_council})", fontsize=5.4, fontweight="bold", va="bottom", color="#111")
        fig.text(left + 0.003, bottom + ih * 0.79 + 0.006, f"Exposed: {r.exposed:,.0f} of {r.population:,.0f} residents\nRisk rank {r['rank']}, R = {r.risk_index:.2f}",
                 fontsize=4.3, va="bottom", color="#333", linespacing=1.15)
        L = 5 if half < 7 else 10
        scalebar(ia, xc + half - L * 1.25, yc - half + half * 0.08, L, fs=4.0)
    # legend in the sixth slot
    lx = gx0 + (iw + gap) + 0.012; ly = gy0 + 0.262
    fig.text(lx, ly, "Exposed population", fontsize=5.8, fontweight="bold", va="top")
    fig.text(lx, ly - 0.019, "(people per hexagon, ≈0.53 km²)", fontsize=4.2, va="top", color="#444")
    for j, (c, t) in enumerate(zip(COLS, LAB)):
        yy = ly - 0.05 - j * 0.0205
        fig.add_artist(Rectangle((lx, yy), 0.016, 0.016, fc=c, ec="#888", lw=0.3, transform=fig.transFigure))
        fig.text(lx + 0.022, yy + 0.008, t, fontsize=4.6, va="center")
    yy = ly - 0.05 - 9 * 0.0205
    fig.add_artist(Rectangle((lx, yy), 0.016, 0.016, fc="none", ec="#00e5ff", lw=1.2, transform=fig.transFigure))
    fig.text(lx + 0.022, yy + 0.008, "Inset ward", fontsize=4.6, va="center")
    ax.text(XL[0] + 1, YL[0] + 4.2, "Exposed = residents on cells of high susceptibility (RF ≥ 0.6) and low coping capacity (lowest CCI tercile)",
            fontsize=4.0, style="italic", color="#333", va="bottom", zorder=10, bbox=dict(fc="white", ec="none", alpha=0.8, pad=1))
    tag = "_satellite" if (BASEMAP and os.path.exists(BASEMAP)) else ""
    fig.savefig(OUT + f"fct_top15_exposed_population_hex_5insets{tag}.png", facecolor="white")
    plt.show(); plt.close(fig)
    hx.to_csv(OUT + "fct_top15_exposed_population_hex_5insets.csv", index=False)
    top[["rank", "wardname", "area_council", "risk_index", "population", "exposed"]].to_csv(OUT + "fct_top15_exposed_by_ward.csv", index=False)

figure_exposed_population_hexagons()

## 24. Composite flood risk × ward population (bivariate, supplementary)
Wards are split into terciles of the composite risk index and of total population and shown with a 3 × 3 colour scheme; the scatter plot shows the same classes with the tercile breaks.

In [ ]:
def figure_risk_population_bivariate():
    import sys, math, numpy as np, pandas as pd, matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.patches import PathPatch, Rectangle, Polygon
    from matplotlib.path import Path
    from matplotlib.collections import PatchCollection

    OUT = WARD_DIR + "/"
    d = pd.read_csv(OUT + "fct_ward_flood_risk.csv")
    r1, r2 = np.quantile(d.risk_index, [1 / 3, 2 / 3]); p1, p2 = np.quantile(d.population, [1 / 3, 2 / 3])
    d["risk_c"] = np.digitize(d.risk_index, [r1, r2]) + 1          # 1 low, 2 medium, 3 high risk
    d["pop_c"] = np.digitize(d.population, [p1, p2]) + 1            # 1 small, 2 medium, 3 large population
    # rows = population (low -> high), columns = risk (low -> high); Stevens-style pink-blue scheme
    PAL = [["#e8e8e8", "#e4acac", "#c85a5a"],
           ["#b0d5df", "#ad9ea5", "#985356"],
           ["#64acbe", "#627f8c", "#574249"]]
    d["colour"] = [PAL[p - 1][r - 1] for p, r in zip(d.pop_c, d.risk_c)]
    names_r = {1: "Low", 2: "Medium", 3: "High"}; names_p = {1: "Small", 2: "Medium", 3: "Large"}
    d["biv_class"] = [f"{names_r[r]} risk / {names_p[p].lower()} population" for p, r in zip(d.pop_c, d.risk_c)]
    d.to_csv(OUT + "fct_ward_risk_population_bivariate.csv", index=False)

    fields, recs = read_dbf(WARD_DBF)
    shapes = read_shp(WARD_SHP)
    idx = {r["wardcode"]: i for i, r in enumerate(recs) if r["statecode"] == WARD_STATE_CODE}
    plt.rcParams["font.family"] = "DejaVu Serif"
    fig = plt.figure(figsize=(7.2, 4.6), dpi=400)

    # ---------------- (a) map
    fr = fig.add_axes([0.008, 0.02, 0.53, 0.96]); fr.set_xticks([]); fr.set_yticks([])
    for sp in fr.spines.values(): sp.set_linewidth(0.9)
    fr.text(0.5, 0.982, "(a) Composite flood risk × population by ward", ha="center", va="top", fontsize=8.4, fontweight="bold", transform=fr.transAxes)
    ax = fig.add_axes([0.012, 0.04, 0.52, 0.88])
    pats, cols = [], []
    for _, r in d.iterrows():
        rings = shapes[idx[r.wardcode]][1]; v, c = [], []
        for ring in rings:
            v += ring.tolist(); c += [Path.MOVETO] + [Path.LINETO] * (len(ring) - 2) + [Path.CLOSEPOLY]
        pats.append(PathPatch(Path(v, c))); cols.append(r.colour)
    ax.add_collection(PatchCollection(pats, facecolors=cols, edgecolors="#4a4a4a", lw=0.3, zorder=2))
    # number the high-risk / medium-population wards (most people at high risk)
    focus = d[(d.risk_c == 3) & (d.pop_c == 2)].sort_values("population", ascending=False).reset_index(drop=True)
    for k, r in focus.iterrows():
        ring = max(shapes[idx[r.wardcode]][1], key=len); pth = Path(ring)
        cx, cy = ring[:, 0].mean(), ring[:, 1].mean()
        if not pth.contains_point((cx, cy)):
            xs = np.linspace(ring[:, 0].min(), ring[:, 0].max(), 25); ys = np.linspace(ring[:, 1].min(), ring[:, 1].max(), 25)
            gx, gy = np.meshgrid(xs, ys); pts = np.c_[gx.ravel(), gy.ravel()]; ins = pts[pth.contains_points(pts)]
            dd = (ins[:, 0] - cx) ** 2 + (ins[:, 1] - cy) ** 2; cx, cy = ins[dd.argmin()]
        tx, ty = cx, cy
        if r.area_km2 < 15:          # small ward: callout below the boundary
            tx, ty = cx + 0.09, 8.43
            ax.plot([cx, tx], [cy, ty + 0.01], color="#222222", lw=0.4, zorder=8); ax.plot([cx], [cy], "o", ms=1.0, color="#222222", zorder=8)
        ax.text(tx, ty, str(k + 1), fontsize=4.8, ha="center", va="center", color="white", fontweight="bold", zorder=8,
                bbox=dict(boxstyle="circle,pad=0.14", fc="#222222", ec="white", lw=0.35))
    ax.text(6.775, 9.52, "High risk, medium population", fontsize=4.9, fontweight="bold", va="top", zorder=9)
    for k, r in focus.iterrows():
        cx_, cy_ = (6.775, 9.49 - k * 0.025) if k < 5 else (7.05, 9.49 - (k - 5) * 0.025)
        ax.text(cx_, cy_, f"{k + 1}. {r.wardname} ({r.area_council}), {r.population/1000:.1f}k", fontsize=4.4, va="top", zorder=9)
    ax.set_xlim(6.76, 7.75); ax.set_ylim(8.40, 9.53)
    ax.set_aspect(1 / math.cos(math.radians(9.0))); ax.axis("off")
    ax.apply_aspect(); pos = ax.get_position()
    # legend 3x3 with number of wards per cell
    lx, ly, cs = 0.40, 0.115, 0.026
    fig.add_artist(Rectangle((lx - 0.034, ly - 0.045), 3 * cs + 0.045, 3 * cs * 1.45 + 0.08, fc="white", ec="none", alpha=0.92, transform=fig.transFigure))
    for p in range(3):
        for r in range(3):
            fig.add_artist(Rectangle((lx + r * cs, ly + p * cs * 1.45), cs, cs * 1.45, fc=PAL[p][r], ec="white", lw=0.5, transform=fig.transFigure))
            n = int(((d.pop_c == p + 1) & (d.risk_c == r + 1)).sum())
            fig.text(lx + (r + 0.5) * cs, ly + (p + 0.5) * cs * 1.45, str(n), fontsize=4.6, ha="center", va="center",
                     color="white" if (p + r) >= 3 else "#222")
    fig.add_artist(plt.Line2D([lx, lx + 3 * cs], [ly - 0.006, ly - 0.006], color="#333", lw=0.6, transform=fig.transFigure))
    fig.text(lx, ly - 0.009, "Low", fontsize=4.3, ha="left", va="top", color="#333")
    fig.text(lx + 3 * cs, ly - 0.009, "High", fontsize=4.3, ha="right", va="top", color="#333")
    fig.text(lx + 1.5 * cs, ly - 0.026, "Composite flood risk", fontsize=4.8, ha="center", va="top")
    fig.add_artist(plt.Line2D([lx - 0.005, lx - 0.005], [ly, ly + 3 * cs * 1.45], color="#333", lw=0.6, transform=fig.transFigure))
    fig.text(lx - 0.008, ly, "Small", fontsize=4.3, ha="right", va="bottom", rotation=90, color="#333")
    fig.text(lx - 0.008, ly + 3 * cs * 1.45, "Large", fontsize=4.3, ha="right", va="top", rotation=90, color="#333")
    fig.text(lx - 0.022, ly + 1.5 * cs * 1.45, "Population", fontsize=4.8, ha="center", va="center", rotation=90)
    fig.text(lx + 1.5 * cs, ly + 3 * cs * 1.45 + 0.026, "Risk × population", fontsize=5.1, ha="center", fontweight="bold")
    fig.text(lx + 1.5 * cs, ly + 3 * cs * 1.45 + 0.010, "(number of wards)", fontsize=4.3, ha="center", color="#444")
    # north arrow and scale bar
    nx, ny, s_ = 0.51, 0.85, 0.055
    fig.add_artist(Polygon([[nx, ny], [nx - 0.006, ny - s_], [nx, ny - 0.75 * s_]], closed=True, fc="black", ec="black", lw=0.5, transform=fig.transFigure))
    fig.add_artist(Polygon([[nx, ny], [nx + 0.006, ny - s_], [nx, ny - 0.75 * s_]], closed=True, fc="white", ec="black", lw=0.5, transform=fig.transFigure))
    fig.text(nx, ny + 0.004, "N", ha="center", va="bottom", fontsize=7.5, fontweight="bold")
    km = 111.32 * math.cos(math.radians(9.0)); xl = ax.get_xlim(); fpk = pos.width / ((xl[1] - xl[0]) * km)
    sx0, sy0 = 0.035, 0.06
    for k, (a, b) in enumerate([(0, 10), (10, 20)]):
        fig.add_artist(Rectangle((sx0 + a * fpk, sy0), (b - a) * fpk, 0.007, fc="black" if k == 0 else "white", ec="black", lw=0.4, transform=fig.transFigure))
    for v in (0, 10, 20):
        fig.text(sx0 + v * fpk, sy0 - 0.004, str(v), ha="center", va="top", fontsize=5)
    fig.text(sx0 + 10 * fpk, sy0 + 0.011, "Kilometres", ha="center", va="bottom", fontsize=5)

    # ---------------- (b) scatter
    fr2 = fig.add_axes([0.546, 0.02, 0.446, 0.96]); fr2.set_xticks([]); fr2.set_yticks([])
    for sp in fr2.spines.values(): sp.set_linewidth(0.9)
    fr2.text(0.5, 0.982, "(b) Risk index and population of the 62 wards", ha="center", va="top", fontsize=8.4, fontweight="bold", transform=fr2.transAxes)
    sx = fig.add_axes([0.62, 0.13, 0.35, 0.76])
    for xv in (r1, r2): sx.axvline(xv, color="#999", lw=0.6, ls="--", zorder=1)
    for yv in (p1, p2): sx.axhline(yv, color="#999", lw=0.6, ls="--", zorder=1)
    sx.scatter(d.risk_index, d.population, s=22, c=d.colour, edgecolors="#333", linewidths=0.4, zorder=3)
    for _, r in d.iterrows():
        if r.risk_c == 3 and (r.pop_c == 2 or r["rank"] <= 3):
            off = {"Pai": (-3, 4, "right"), "Paiko": (4, 3, "left"), "Dobi": (4, -6, "left"), "Ibwa": (-3, 2, "right"), "Gawu": (4, -5, "left")}.get(r.wardname, (3, 2, "left"))
            sx.annotate(r.wardname, (r.risk_index, r.population), xytext=off[:2], textcoords="offset points", fontsize=4.3, ha=off[2], zorder=4)
        elif r.population > 3e5:
            sx.annotate(r.wardname, (r.risk_index, r.population), xytext=(3, 2), textcoords="offset points", fontsize=4.3, color="#444", zorder=4)
    sx.set_yscale("log"); sx.set_xlim(0, 0.7); sx.set_ylim(800, 1.2e6)
    sx.set_xlabel("Composite flood risk index, R = S × (1 − CCI)", fontsize=5.8)
    sx.set_ylabel("Ward population (log scale)", fontsize=5.8)
    sx.tick_params(labelsize=5)
    for sp in ["top", "right"]: sx.spines[sp].set_visible(False)
    sx.text(r1, 1.05e6, f"{r1:.2f}", fontsize=4.3, ha="center", color="#666"); sx.text(r2, 1.05e6, f"{r2:.2f}", fontsize=4.3, ha="center", color="#666")
    sx.text(0.695, p1, f"{p1/1000:.0f}k", fontsize=4.3, ha="right", va="bottom", color="#666"); sx.text(0.695, p2, f"{p2/1000:.0f}k", fontsize=4.3, ha="right", va="bottom", color="#666")
    sx.text(0.6, 4e5, "No ward is both\nhigh risk and\nhighly populated", fontsize=4.6, ha="center", style="italic", color="#555")
    fig.text(0.77, 0.04, "Dashed lines: tercile breaks used for the bivariate classes", fontsize=4.4, ha="center", color="#555", style="italic")
    fig.savefig(OUT + "fct_ward_risk_population_bivariate.png", facecolor="white")
    plt.show(); plt.close(fig)
    log(f"Risk terciles {r1:.3f}, {r2:.3f}; population terciles {p1:,.0f}, {p2:,.0f}")

figure_risk_population_bivariate()

## 25. Population by coping capacity (Figure 18)
Residents are assigned to the CCI tercile of the cell that contains them (terciles by area, as in section 16). Panel (a) maps the lowest tercile and panel (b) the highest, in hexagons of ≈0.79 km² with identical class breaks. A table of every ward's population in each tercile is written to `fct_ward_population_by_coping_capacity.csv`.

In [ ]:
def population_by_cci_tercile():
    import sys, math, json, numpy as np, pandas as pd
    OUT = WARD_DIR + "/"
    c = imread(MAP_DIR + "/FCT_coping_capacity_index.tif").astype(float); c[c < -1] = np.nan
    t1, t2 = np.nanquantile(c, [1/3, 2/3]); log(f"CCI terciles {t1:.3f}, {t2:.3f}")
    wid = np.load(OUT + "ward_id_raster.npy"); H, W = c.shape
    pop, (pox, psx, poy, psy), _, pnd = read_tif(FDIR + "/" + "fct_pop.tif"); pop = pop.astype(float)
    if pnd is not None: pop[np.isclose(pop, pnd)] = 0
    pop[~np.isfinite(pop) | (pop < 0)] = 0
    pr, pc = np.nonzero(pop); plon = pox + (pc + .5) * psx; plat = poy + (pr + .5) * psy
    rr = np.floor((OY - plat) / SX).astype(int); cc = np.floor((plon - OX) / SX).astype(int)
    ok = (rr >= 0) & (rr < H) & (cc >= 0) & (cc < W)
    pv, lon, lat, rr, cc = pop[pr[ok], pc[ok]], plon[ok], plat[ok], rr[ok], cc[ok]
    cv = c[rr, cc]; k = wid[rr, cc]; m = np.isfinite(cv) & (k > 0)
    cls = np.digitize(cv[m], [t1, t2]) + 1     # 1 lowest, 3 highest capacity
    np.savez(OUT + "cci_pop_points.npz", lon=lon[m], lat=lat[m], pop=pv[m], cls=cls, wid=k[m])
    tot = pv[m].sum()
    for j in (1, 2, 3):
        pj = pv[m][cls == j].sum(); log(f"Tercile {j}: {pj:,.0f} residents ({100 * pj / tot:.1f}%)")
    log(f"Population on the CCI grid: {tot:,.0f}")
    json.dump({"t1": t1, "t2": t2}, open(OUT + "cci_terciles.json", "w"))

population_by_cci_tercile()

In [ ]:
def figure_population_by_coping_capacity():
    import sys, math, json, numpy as np, pandas as pd, matplotlib
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle, Polygon, PathPatch
    from matplotlib.path import Path
    from matplotlib.colors import ListedColormap, BoundaryNorm
    from matplotlib.collections import PolyCollection, PatchCollection
    OUT = WARD_DIR + "/"
    LAT0, LON0 = 9.0, 6.75; KX, KY = 111.320 * math.cos(math.radians(LAT0)), 110.574
    def km(lon, lat): return (np.asarray(lon) - LON0) * KX, (np.asarray(lat) - LAT0) * KY
    T = json.load(open(OUT + "cci_terciles.json"))
    z = np.load(OUT + "cci_pop_points.npz"); x, y = km(z["lon"], z["lat"]); P, CL, WK = z["pop"], z["cls"], z["wid"]
    fields, recs = read_dbf(WARD_DBF); shapes = read_shp(WARD_SHP)
    idx = [i for i, r in enumerate(recs) if r["statecode"] == WARD_STATE_CODE]
    short = {"Municipal Area Council": "AMAC"}
    wname = {k: recs[i]["wardname"] for k, i in enumerate(idx, start=1)}
    wlga = {k: short.get(recs[i]["lganame"], recs[i]["lganame"]) for k, i in enumerate(idx, start=1)}

    # ---- per-ward table
    tw = pd.DataFrame({"k": WK, "c": CL, "p": P}).pivot_table(index="k", columns="c", values="p", aggfunc="sum", fill_value=0)
    tw.columns = ["pop_lowest_cci", "pop_middle_cci", "pop_highest_cci"]
    tw["population"] = tw.sum(axis=1); tw["pct_lowest"] = 100 * tw.pop_lowest_cci / tw.population; tw["pct_highest"] = 100 * tw.pop_highest_cci / tw.population
    tw.insert(0, "area_council", [wlga[k] for k in tw.index]); tw.insert(0, "wardname", [wname[k] for k in tw.index])
    tw.round(1).to_csv(OUT + "fct_ward_population_by_coping_capacity.csv", index=False)

    # ---- hexagons (pointy-top, circumradius A km, area 2.598 A^2)
    A = 0.55
    def hex_index(x, y):
        q = (math.sqrt(3) / 3 * x - y / 3) / A; r = (2 / 3 * y) / A; X, Z = q, r; Y = -X - Z
        rx, ry, rz = np.round(X), np.round(Y), np.round(Z); dx, dy, dz = abs(rx - X), abs(ry - Y), abs(rz - Z)
        fx = (dx > dy) & (dx > dz); fz = ~fx & (dy <= dz)
        rx = np.where(fx, -ry - rz, rx); rz = np.where(fz, -rx - ry, rz); return rx.astype(int), rz.astype(int)
    hq, hr = hex_index(x, y)
    ang = np.radians(np.arange(6) * 60 + 30)
    def hexes(cls):
        m = CL == cls
        h = pd.DataFrame({"q": hq[m], "r": hr[m], "p": P[m]}).groupby(["q", "r"]).p.sum().reset_index(); h = h[h.p >= 0.5]
        h["x"] = A * math.sqrt(3) * (h.q + h.r / 2); h["y"] = A * 1.5 * h.r
        return h, np.stack([np.c_[a + A * np.cos(ang), b + A * np.sin(ang)] for a, b in zip(h.x, h.y)])
    BINS = [0.5, 10, 50, 100, 250, 500, 1000, 2500, 5000, 10000, 15000]
    LAB = ["(0; 10]", "(10; 50]", "(50; 100]", "(100; 250]", "(250; 500]", "(500; 1,000]", "(1,000; 2,500]", "(2,500; 5,000]", "(5,000; 10,000]", "(10,000; 15,000]"]
    RAMP = {1: ["#fff0e6", "#fdd5bf", "#fcb595", "#f8916d", "#ef6548", "#d7301f", "#b3141f", "#860a21", "#55061c", "#2a0310"],
            3: ["#eef5fb", "#d3e5f3", "#b0d1ea", "#86b9de", "#5b9dcf", "#3a7fbf", "#2362a6", "#164a86", "#0d3363", "#061c3b"]}
    norm = BoundaryNorm(BINS, 10)

    # ---- basemap and wards
    hs, (bx, bsx, by, bsy), _, _ = read_tif(HILLSHADE_TIF)
    hs = hs.astype(float)[::2, ::2]; hs[hs == 0] = np.nan; hh, ww = hs.shape
    x0, y0 = km(bx, by + hh * 2 * bsy); x1, y1 = km(bx + ww * 2 * bsx, by); BEXT = [float(x0), float(x1), float(y0), float(y1)]
    WP = {}
    for k, i in enumerate(idx, start=1):
        v, c = [], []
        for ring in shapes[i][1]:
            X, Y = km(ring[:, 0], ring[:, 1]); v += np.c_[X, Y].tolist(); c += [Path.MOVETO] + [Path.LINETO] * (len(ring) - 2) + [Path.CLOSEPOLY]
        WP[k] = Path(v, c)
    ALL = np.vstack([p.vertices for p in WP.values()])
    XL, YL = (ALL[:, 0].min() - 1.5, ALL[:, 0].max() + 1.5), (ALL[:, 1].min() - 1.5, ALL[:, 1].max() + 1.5)
    lg_xy = {}
    rows_, cols_ = np.nonzero(wid := np.load(OUT + "ward_id_raster.npy"))
    lgv = np.array([wlga[v] for v in wid[rows_, cols_]])
    for nm in sorted(set(lgv)):
        mm = lgv == nm; lg_xy[nm] = km(OX + (np.median(cols_[mm]) + .5) * SX, OY - (np.median(rows_[mm]) + .5) * SX)

    plt.rcParams["font.family"] = "DejaVu Serif"
    fig = plt.figure(figsize=(7.2, 4.75), dpi=400)
    fr = fig.add_axes([0.004, 0.004, 0.992, 0.992]); fr.set_xticks([]); fr.set_yticks([])
    for sp in fr.spines.values(): sp.set_linewidth(0.9)
    fig.text(0.5, 0.955, "Population by coping capacity, FCT Abuja", ha="center", fontsize=9.5, fontweight="bold")
    tot = P.sum()
    PANELS = [(1, 0.012, "(a) Lowest coping capacity", f"CCI < {T['t1']:.2f} (lowest tercile)"),
              (3, 0.506, "(b) Highest coping capacity", f"CCI ≥ {T['t2']:.2f} (highest tercile)")]
    for cls, px0, title, sub in PANELS:
        h, verts = hexes(cls); cols = np.array(ListedColormap(RAMP[cls])(norm(h.p.values)))
        ax = fig.add_axes([px0, 0.15, 0.482, 0.76])
        ax.set_facecolor("white")
        ax.imshow(hs, extent=BEXT, cmap="gray", vmin=0, vmax=255, interpolation="bilinear", alpha=0.45, zorder=0)
        o = np.argsort(h.p.values)
        ax.add_collection(PolyCollection(verts[o], facecolors=cols[o], edgecolors="none", zorder=3))
        ax.add_collection(PatchCollection([PathPatch(p) for p in WP.values()], facecolors="none", edgecolors="#555", lw=0.22, zorder=4))
        ax.set_xlim(*XL); ax.set_ylim(*YL); ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values(): sp.set_linewidth(0.7)
        for nm, (lx_, ly_) in lg_xy.items():
            ax.text(lx_, ly_, nm.upper(), fontsize=4.8, ha="center", va="center", fontweight="bold", color="#222", alpha=0.8, zorder=6,
                    bbox=dict(boxstyle="round,pad=0.12", fc="white", ec="none", alpha=0.6))
        s = P[CL == cls].sum()
        ax.text(0.5, 1.012, title, transform=ax.transAxes, ha="center", va="bottom", fontsize=7.6, fontweight="bold")
        TA = ax.transAxes; bb = dict(boxstyle="round,pad=0.35", fc="white", ec="#bbb", lw=0.3, alpha=0.88)
        # NW corner: summary and wards with the most residents
        col = "pop_lowest_cci" if cls == 1 else "pop_highest_cci"
        tt = tw.sort_values(col, ascending=False).head(5)
        lines = [f"{j + 1}. {r.wardname} ({r.area_council})" for j, (_, r) in enumerate(tt.iterrows())]
        nums = [f"{r[col]:,.0f}" for _, r in tt.iterrows()]
        ax.text(0.02, 0.985, f"{sub}: {s:,.0f} residents ({100 * s / tot:.1f}% of the FCT)", transform=TA, fontsize=4.9, va="top", fontweight="bold", zorder=9)
        ax.text(0.02, 0.952, "Wards with the most residents in this class", transform=TA, fontsize=4.6, va="top", style="italic", zorder=9)
        for j, (l_, n_) in enumerate(zip(lines, nums)):
            ax.text(0.02, 0.925 - j * 0.025, l_, transform=TA, fontsize=4.6, va="top", zorder=9)
            ax.text(0.40, 0.925 - j * 0.025, n_, transform=TA, fontsize=4.6, va="top", ha="right", zorder=9)
        ax.add_patch(Rectangle((0.008, 0.79), 0.40, 0.205, transform=TA, fc="white", ec="#bbb", lw=0.3, alpha=0.85, zorder=8))
        # horizontal legend under the map
        ax.apply_aspect(); pz = ax.get_position()
        lw_ = pz.width * 0.86 / 10; lx0 = pz.x0 + pz.width * 0.07; ly0 = pz.y0 - 0.045
        fig.text(pz.x0 + pz.width / 2, ly0 + 0.022, "People per hexagon (\u2248 0.79 km\u00b2)", ha="center", fontsize=5.2, fontweight="bold")
        for j, c_ in enumerate(RAMP[cls]):
            fig.add_artist(Rectangle((lx0 + j * lw_, ly0), lw_, 0.016, fc=c_, ec="#888", lw=0.25, transform=fig.transFigure))
        for j, t_ in enumerate(["0", "10", "50", "100", "250", "500", "1,000", "2,500", "5,000", "10,000", "15,000"]):
            fig.text(lx0 + j * lw_, ly0 - 0.004, t_, ha="center", va="top", fontsize=4.4)
        if cls == 1:
            ax.apply_aspect(); pos = ax.get_position(); fpk = pos.width / (XL[1] - XL[0])
            sx0, sy0 = pos.x0 + pos.width * 0.72, pos.y0 + 0.03
            for j, (a, b) in enumerate([(0, 10), (10, 20)]):
                fig.add_artist(Rectangle((sx0 + a * fpk, sy0), (b - a) * fpk, 0.006, fc="black" if j == 0 else "white", ec="black", lw=0.4, transform=fig.transFigure))
            for v in (0, 10, 20): fig.text(sx0 + v * fpk, sy0 - 0.003, str(v), ha="center", va="top", fontsize=4.8)
            fig.text(sx0 + 10 * fpk, sy0 + 0.009, "Kilometres", ha="center", va="bottom", fontsize=4.8)
            nx, ny = pos.x0 + pos.width * 0.50, pos.y0 + pos.height * 0.83
            fig.add_artist(Polygon([[nx, ny], [nx - 0.007, ny - 0.055], [nx, ny - 0.041]], fc="black", ec="black", lw=0.5, transform=fig.transFigure))
            fig.add_artist(Polygon([[nx, ny], [nx + 0.007, ny - 0.055], [nx, ny - 0.041]], fc="white", ec="black", lw=0.5, transform=fig.transFigure))
            fig.text(nx, ny + 0.004, "N", ha="center", va="bottom", fontsize=7.5, fontweight="bold")
    fig.text(0.5, 0.032, f"Coping capacity classes are terciles of the CCI raster (by area). Both panels share the same class breaks; the middle tercile ({T['t1']:.2f}–{T['t2']:.2f}) is not shown.\n"
             "Population: native ~92 m grid summed into hexagons on a local equirectangular projection. Basemap: hillshade.",
             ha="center", va="center", fontsize=4.6, style="italic", color="#444", linespacing=1.4)
    fig.savefig(OUT + "fct_population_lowest_highest_coping_capacity.png", facecolor="white")
    plt.show(); plt.close(fig)
    print(tw.sort_values("pop_lowest_cci", ascending=False).head(5)[["wardname","area_council","pop_lowest_cci","pct_lowest"]].round(1))
    print(tw.sort_values("pop_highest_cci", ascending=False).head(5)[["wardname","area_council","pop_highest_cci","pct_highest"]].round(1))
    print("wards with >50% lowest", (tw.pct_lowest > 50).sum(), tw[tw.pct_lowest > 50].wardname.tolist())

figure_population_by_coping_capacity()

## 26. Save the results
The whole results folder (Part A and Part B) is zipped and copied to Google Drive (if mounted); it is also offered as a browser download.

In [ ]:
with open(os.path.join(OUT_DIR, "run_log.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(LOG) if "LOG" in globals() else "")
zip_path = shutil.make_archive("/content/fct_flood_ml_results", "zip", OUT_DIR)
if os.path.isdir("/content/drive/MyDrive"):
    shutil.copy(zip_path, "/content/drive/MyDrive/")
    print("Copied to Google Drive:", "/content/drive/MyDrive/" + os.path.basename(zip_path))
from google.colab import files
files.download(zip_path)